In [ ]:
# Cell 1 — Objective: Environment setup (install/verify only required dependencies for this notebook)

%pip install --quiet --upgrade pip
%pip install --quiet oci ipywidgets ipyfilechooser paramiko tqdm numpy pandas locust requests "httpx[http2]"

import importlib.metadata as md

import oci
import ipywidgets as widgets
from ipyfilechooser import FileChooser
import paramiko
import tqdm
import numpy as np
import pandas as pd
import requests
import httpx

print("Cell 1 complete: Environment ready for FLB session-concurrency testing (HTTP/2 preferred + HTTP/1.1 support).")
print(
    "Package versions:",
    {
        "oci": md.version("oci"),
        "ipywidgets": md.version("ipywidgets"),
        "ipyfilechooser": md.version("ipyfilechooser"),
        "paramiko": md.version("paramiko"),
        "tqdm": md.version("tqdm"),
        "numpy": md.version("numpy"),
        "pandas": md.version("pandas"),
        "locust": md.version("locust"),
        "requests": md.version("requests"),
        "httpx": md.version("httpx"),
    },
)
print("NEXT: Run Cell 2 for central variables and credentials (single source of truth).")


In [ ]:
# Cell 2 — Objective: Central variables and credentials (single source of truth; no network calls)

import os
import json
import ipaddress
from pathlib import Path
from datetime import datetime, timezone

import oci

# ----------------------------
# Helpers (env parsing/validation)
# ----------------------------
def _env_str(name: str, default: str = "") -> str:
    return str(os.environ.get(name, default)).strip()


def _env_int(name: str, default: int, minimum: int | None = None, maximum: int | None = None) -> int:
    raw = os.environ.get(name, None)
    value = int(default if raw is None or str(raw).strip() == "" else str(raw).replace("_", ""))
    if minimum is not None and value < minimum:
        raise ValueError(f"{name} must be >= {minimum}. Current: {value}")
    if maximum is not None and value > maximum:
        raise ValueError(f"{name} must be <= {maximum}. Current: {value}")
    return value


def _env_float(name: str, default: float, minimum: float | None = None, maximum: float | None = None) -> float:
    raw = os.environ.get(name, None)
    value = float(default if raw is None or str(raw).strip() == "" else raw)
    if minimum is not None and value < minimum:
        raise ValueError(f"{name} must be >= {minimum}. Current: {value}")
    if maximum is not None and value > maximum:
        raise ValueError(f"{name} must be <= {maximum}. Current: {value}")
    return value


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.environ.get(name, None)
    if raw is None or str(raw).strip() == "":
        return bool(default)
    return str(raw).strip().lower() in ("1", "true", "yes", "y", "on")


def _env_optional_int(name: str, default: int | None = None, minimum: int | None = None, maximum: int | None = None) -> int | None:
    raw = os.environ.get(name, None)
    if raw is None or str(raw).strip() == "":
        return default
    value = int(str(raw).replace("_", ""))
    if minimum is not None and value < minimum:
        raise ValueError(f"{name} must be >= {minimum}. Current: {value}")
    if maximum is not None and value > maximum:
        raise ValueError(f"{name} must be <= {maximum}. Current: {value}")
    return value


def _env_csv_int_list(name: str, default: list[int], minimum: int = 1) -> list[int]:
    raw = os.environ.get(name, None)
    if raw is None or str(raw).strip() == "":
        vals = list(default)
    else:
        vals = []
        for p in str(raw).split(","):
            t = p.strip()
            if not t:
                continue
            n = int(t.replace("_", ""))
            if n < minimum:
                raise ValueError(f"{name} contains value < {minimum}: {n}")
            vals.append(n)
    vals = sorted(set(vals))
    if not vals:
        raise ValueError(f"{name} resolved to an empty list.")
    return vals


def _require_in(name: str, value: str, allowed: set[str]) -> str:
    if value not in allowed:
        raise ValueError(f"{name} must be one of {sorted(allowed)}. Current: {value}")
    return value


def _expand(path_text: str) -> str:
    return os.path.expanduser(path_text) if path_text else path_text


def _exists(path_text: str) -> bool:
    return bool(path_text) and os.path.exists(_expand(path_text))


def _ensure_path_exists(path: str, label: str, optional: bool = False) -> None:
    if not path:
        if optional:
            return
        raise FileNotFoundError(f"{label} path not set.")
    p = _expand(path)
    if not os.path.exists(p):
        if optional:
            return
        raise FileNotFoundError(f"{label} not found: {p}")


def _validate_cidr(name: str, cidr_text: str, version: int) -> str:
    try:
        net = ipaddress.ip_network(cidr_text, strict=False)
    except Exception as e:
        raise ValueError(f"{name} is not a valid CIDR: {cidr_text!r} ({e})") from e
    if net.version != version:
        raise ValueError(f"{name} must be IPv{version} CIDR. Current: {cidr_text!r}")
    return str(net)


def _validate_cidr_or_auto(name: str, value: str, version: int) -> str:
    text = (value or "").strip()
    if text.lower() == "auto":
        return "auto"
    return _validate_cidr(name, text, version)


def _calc_run_math(users: int, avg_session_sec: int, heartbeat_interval_sec: int) -> dict:
    ramp = users / avg_session_sec
    rps = users / heartbeat_interval_sec
    return {
        "users": int(users),
        "ramp_users_per_sec": round(ramp, 3),
        "expected_steady_heartbeat_rps": round(rps, 3),
        "ramp_derivation": f"{users} / {avg_session_sec} = {round(ramp, 3)} users/s",
        "rps_derivation": f"{users} / {heartbeat_interval_sec} = {round(rps, 3)} rps",
    }


# ----------------------------
# OCI credentials/config (local file + env overrides only)
# ----------------------------
OCI_CONFIG_FILE = _expand(_env_str("OCI_CONFIG_FILE", "~/.oci/config"))
OCI_PROFILE = _env_str("OCI_PROFILE", "DEFAULT")

if not os.path.exists(OCI_CONFIG_FILE):
    raise FileNotFoundError(f"OCI_CONFIG_FILE not found: {OCI_CONFIG_FILE}")

_cfg = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)
oci.config.validate_config(_cfg)

TENANCY_OCID = _env_str("TENANCY_OCID", _cfg.get("tenancy", ""))
USER_OCID = _env_str("USER_OCID", _cfg.get("user", ""))
FINGERPRINT = _env_str("FINGERPRINT", _cfg.get("fingerprint", ""))

OCI_PRIVATE_KEY_PATH = _expand(_env_str("OCI_PRIVATE_KEY_PATH", _cfg.get("key_file", "")))
PRIVATE_KEY_PASSPHRASE = _env_str(
    "OCI_PASSPHRASE",
    _env_str("OCI_PRIVATE_KEY_PASSPHRASE", _cfg.get("pass_phrase", "")),
)
REGION = _env_str("REGION", _cfg.get("region", ""))

_ensure_path_exists(OCI_PRIVATE_KEY_PATH, "OCI private key", optional=False)

if not TENANCY_OCID or not USER_OCID or not FINGERPRINT or not REGION:
    raise ValueError(
        "Missing required OCI identity fields after config/env load. "
        "Required: TENANCY_OCID, USER_OCID, FINGERPRINT, REGION."
    )


# ----------------------------
# Test mode and session-concurrency model
# ----------------------------
TEST_MODE = _env_str("TEST_MODE", "session_concurrency").lower()
TEST_MODE = _require_in("TEST_MODE", TEST_MODE, {"session_concurrency"})

CONCURRENCY_MODEL = _env_str("CONCURRENCY_MODEL", "one_session_one_connection").lower()
CONCURRENCY_MODEL = _require_in("CONCURRENCY_MODEL", CONCURRENCY_MODEL, {"one_session_one_connection"})

SESSION_CLOSE_MODE = _env_str("SESSION_CLOSE_MODE", "lifecycle_ttl").lower()
SESSION_CLOSE_MODE = _require_in("SESSION_CLOSE_MODE", SESSION_CLOSE_MODE, {"lifecycle_ttl"})

TARGET_CONCURRENCY = _env_int("TARGET_CONCURRENCY", 5_000_000, minimum=1)
AVG_SESSION_SEC = _env_int("AVG_SESSION_SEC", 1200, minimum=1)
HEARTBEAT_INTERVAL_SEC = _env_int("HEARTBEAT_INTERVAL_SEC", 50, minimum=1)
UPLOAD_BYTES = _env_int("UPLOAD_BYTES", 5120, minimum=1)

SESSION_TTL_JITTER_ENABLED = _env_bool("SESSION_TTL_JITTER_ENABLED", True)
SESSION_TTL_JITTER_PCT = _env_float("SESSION_TTL_JITTER_PCT", 10.0, minimum=0.0, maximum=95.0)

H2_CLIENT_PERCENT = _env_float("H2_CLIENT_PERCENT", 100.0, minimum=0.0, maximum=100.0)
PROTOCOL_PROFILE = _env_str("PROTOCOL_PROFILE", "http2_preferred").lower()
PROTOCOL_PROFILE = _require_in(
    "PROTOCOL_PROFILE",
    PROTOCOL_PROFILE,
    {"http2_preferred", "https_h1_compatible", "tcp_ppv2"},
)

STEADY_STATE_TOLERANCE_PCT = _env_float("STEADY_STATE_TOLERANCE_PCT", 10.0, minimum=0.0, maximum=100.0)
STEADY_STATE_EVAL_WINDOW_SEC = _env_int("STEADY_STATE_EVAL_WINDOW_SEC", 600, minimum=30)

RAMP_UP_SEC = _env_int("RAMP_UP_SEC", 1200, minimum=1)
RUN_HOLD_SEC = _env_int("RUN_HOLD_SEC", 3600, minimum=1)

TARGET_OPENS_PER_SEC = TARGET_CONCURRENCY / AVG_SESSION_SEC
TARGET_CLOSES_PER_SEC = TARGET_OPENS_PER_SEC
TARGET_OPENS_PER_MIN = TARGET_OPENS_PER_SEC * 60.0
TARGET_CLOSES_PER_MIN = TARGET_CLOSES_PER_SEC * 60.0

DEFAULT_RUN_SCENARIO_USERS = [10_000, 25_000, 50_000, 1_000_000, 2_000_000, 3_000_000, 4_000_000, 5_000_000]
RUN_TARGET_USERS = _env_int("RUN_TARGET_USERS", 50_000, minimum=1)
RUN_SCENARIO_USERS = _env_csv_int_list("RUN_SCENARIO_USERS", DEFAULT_RUN_SCENARIO_USERS, minimum=1)
if RUN_TARGET_USERS not in RUN_SCENARIO_USERS:
    RUN_SCENARIO_USERS = sorted(set(RUN_SCENARIO_USERS + [RUN_TARGET_USERS]))

RUN_SPAWN_RATE = RUN_TARGET_USERS / AVG_SESSION_SEC
RUN_TARGET_EXPECTED_HEARTBEAT_RPS = RUN_TARGET_USERS / HEARTBEAT_INTERVAL_SEC
RUN_MATH_TABLE = [_calc_run_math(u, AVG_SESSION_SEC, HEARTBEAT_INTERVAL_SEC) for u in RUN_SCENARIO_USERS]


# ----------------------------
# Topology, capacity, and compute defaults
# ----------------------------
LB_TOPOLOGY = _env_str("LB_TOPOLOGY", "single").lower()
LB_TOPOLOGY = _require_in("LB_TOPOLOGY", LB_TOPOLOGY, {"single", "multi"})

LB_COUNT = _env_int("LB_COUNT", 1, minimum=1)
if LB_TOPOLOGY == "single":
    LB_COUNT = 1

BACKEND_COUNT = _env_int("BACKEND_COUNT", 120, minimum=1)
GENERATOR_COUNT = _env_int("GENERATOR_COUNT", 1, minimum=1)

BACKEND_SHAPE = _env_str("BACKEND_SHAPE", "VM.Standard.E5.Flex")
BACKEND_OCPUS = _env_float("BACKEND_OCPUS", 16.0, minimum=1.0)
BACKEND_MEMORY_GB = _env_float("BACKEND_MEMORY_GB", 64.0, minimum=1.0)

GENERATOR_SHAPE = _env_str("GENERATOR_SHAPE", "VM.Standard.E5.Flex")
GENERATOR_OCPUS = _env_float("GENERATOR_OCPUS", 16.0, minimum=1.0)
GENERATOR_MEMORY_GB = _env_float("GENERATOR_MEMORY_GB", 64.0, minimum=1.0)

LB_MIN_MBPS = _env_int("LB_MIN_MBPS", 8000, minimum=10, maximum=32000)
LB_MAX_MBPS = _env_int("LB_MAX_MBPS", 8000, minimum=10, maximum=32000)
if LB_MAX_MBPS < LB_MIN_MBPS:
    raise ValueError(f"LB_MAX_MBPS ({LB_MAX_MBPS}) must be >= LB_MIN_MBPS ({LB_MIN_MBPS}).")


# ----------------------------
# Networking and protocol behavior
# ----------------------------
ENABLE_IPV6_FRONTEND = _env_bool("ENABLE_IPV6_FRONTEND", True)
USE_SSH_IPV6 = _env_bool("USE_SSH_IPV6", False)

SSH_ALLOWED_CIDR = _env_str("SSH_ALLOWED_CIDR", "0.0.0.0/0")
SSH_ALLOWED_V6_CIDR = _env_str("SSH_ALLOWED_V6_CIDR", "::/0")

MASTER_IPV6_OVERRIDE = _env_str("MASTER_IPV6_OVERRIDE", "")
MASTER_PRIVATE_IP_OVERRIDE = _env_str("MASTER_PRIVATE_IP_OVERRIDE", "")

HEALTH_ENDPOINT_PATH = _env_str("HEALTH_ENDPOINT_PATH", "/healthz")
UPLOAD_ENDPOINT_PATH = _env_str("UPLOAD_ENDPOINT_PATH", "/upload_5k")

LOCUST_VERIFY_TLS = _env_bool("LOCUST_VERIFY_TLS", False)
LOCUST_CONNECT_TIMEOUT_MS = _env_int("LOCUST_CONNECT_TIMEOUT_MS", 8000, minimum=1000)
LOCUST_READ_TIMEOUT_MS = _env_int("LOCUST_READ_TIMEOUT_MS", 15000, minimum=1000)

LOCUST_CONNECT_TIMEOUT = LOCUST_CONNECT_TIMEOUT_MS
LOCUST_READ_TIMEOUT = LOCUST_READ_TIMEOUT_MS
LOCUST_WORKDIR = _env_str("LOCUST_WORKDIR", "/home/opc/locustwork")

LB_VISIBILITY = _env_str("LB_VISIBILITY", "public").lower()
LB_VISIBILITY = _require_in("LB_VISIBILITY", LB_VISIBILITY, {"public", "private"})

LB_ALLOWED_CIDR_V4 = _validate_cidr("LB_ALLOWED_CIDR_V4", _env_str("LB_ALLOWED_CIDR_V4", "0.0.0.0/0"), 4)
LB_ALLOWED_CIDR_V6 = _validate_cidr("LB_ALLOWED_CIDR_V6", _env_str("LB_ALLOWED_CIDR_V6", "::/0"), 6)

LB_PRIVATE_ALLOWED_CIDR_V4 = _validate_cidr_or_auto(
    "LB_PRIVATE_ALLOWED_CIDR_V4",
    _env_str("LB_PRIVATE_ALLOWED_CIDR_V4", "auto"),
    4,
)
LB_PRIVATE_ALLOWED_CIDR_V6 = _validate_cidr_or_auto(
    "LB_PRIVATE_ALLOWED_CIDR_V6",
    _env_str("LB_PRIVATE_ALLOWED_CIDR_V6", "auto"),
    6,
)

if LB_VISIBILITY == "public":
    LB_EFFECTIVE_ALLOWED_CIDR_V4 = LB_ALLOWED_CIDR_V4
    LB_EFFECTIVE_ALLOWED_CIDR_V6 = LB_ALLOWED_CIDR_V6 if ENABLE_IPV6_FRONTEND else ""
else:
    LB_EFFECTIVE_ALLOWED_CIDR_V4 = LB_PRIVATE_ALLOWED_CIDR_V4
    LB_EFFECTIVE_ALLOWED_CIDR_V6 = LB_PRIVATE_ALLOWED_CIDR_V6 if ENABLE_IPV6_FRONTEND else ""

UPLOAD_ACK_ENABLED = _env_bool("UPLOAD_ACK_ENABLED", True)
UPLOAD_ACK_STATUS = _env_int("UPLOAD_ACK_STATUS", 200, minimum=200, maximum=299)
UPLOAD_ACK_BODY = _env_str("UPLOAD_ACK_BODY", "ack")


# ----------------------------
# UI + run controls
# ----------------------------
UI_ENABLE = _env_bool("UI_ENABLE", True)
UI_EXPOSE_MODE = _env_str("UI_EXPOSE_MODE", "tunnel").lower()
UI_EXPOSE_MODE = _require_in("UI_EXPOSE_MODE", UI_EXPOSE_MODE, {"tunnel", "nsg"})
UI_WEB_HOST = _env_str("UI_WEB_HOST", "0.0.0.0")
UI_WEB_PORT = _env_int("UI_WEB_PORT", 8089, minimum=1024, maximum=65535)
UI_ALLOWED_CIDR = _env_str("UI_ALLOWED_CIDR", "0.0.0.0/0")

HEADLESS_ENABLE = _env_bool("HEADLESS_ENABLE", False)
EXPECT_WORKERS_STRICT = _env_bool("EXPECT_WORKERS_STRICT", False)
EXPECTED_WORKERS_OVERRIDE = _env_optional_int("EXPECTED_WORKERS_OVERRIDE", default=None, minimum=1)
GRACE_SEC = _env_int("GRACE_SEC", 60, minimum=1)

WORKERS_PER_HOST = _env_str("WORKERS_PER_HOST", "auto").lower()
WORKERS_PER_HOST = _require_in("WORKERS_PER_HOST", WORKERS_PER_HOST, {"auto", "fixed"})
CPU_RESERVE = _env_int("CPU_RESERVE", 1, minimum=0, maximum=32)
MIN_WORKERS_PER_HOST = _env_int("MIN_WORKERS_PER_HOST", 1, minimum=1)
MAX_WORKERS_PER_HOST = _env_int("MAX_WORKERS_PER_HOST", 64, minimum=1)
if MAX_WORKERS_PER_HOST < MIN_WORKERS_PER_HOST:
    raise ValueError(
        f"MAX_WORKERS_PER_HOST ({MAX_WORKERS_PER_HOST}) must be >= MIN_WORKERS_PER_HOST ({MIN_WORKERS_PER_HOST})."
    )
FIXED_WORKERS_PER_HOST = _env_int("FIXED_WORKERS_PER_HOST", MIN_WORKERS_PER_HOST, minimum=1)


# ----------------------------
# TLS certificate inputs (for listener offload modes)
# ----------------------------
LB_CERT_MODE = _env_str("LB_CERT_MODE", "browse").lower()
LB_CERT_MODE = _require_in("LB_CERT_MODE", LB_CERT_MODE, {"browse", "generate"})

LB_CERT_PEM_PATH = _expand(_env_str("LB_CERT_PEM_PATH", ""))
LB_KEY_PEM_PATH = _expand(_env_str("LB_KEY_PEM_PATH", ""))
LB_CA_PEM_PATH = _expand(_env_str("LB_CA_PEM_PATH", ""))

LB_CERT_KIND = _env_str("LB_CERT_KIND", "ecdsa").lower()
LB_CERT_KIND = _require_in("LB_CERT_KIND", LB_CERT_KIND, {"ecdsa", "rsa"})
LB_ECDSA_CURVE = _env_str("LB_ECDSA_CURVE", "secp256r1")
LB_RSA_BITS = _env_int("LB_RSA_BITS", 2048, minimum=2048)
LB_CERT_CN = _env_str("LB_CERT_CN", "lb.local")
LB_CERT_DAYS = _env_int("LB_CERT_DAYS", 3650, minimum=1)
LB_PEM_OUTPUT_DIR = _expand(_env_str("LB_PEM_OUTPUT_DIR", "./local-lb-pems"))
LB_PEM_BASENAME = _env_str("LB_PEM_BASENAME", "lb_current")


# ----------------------------
# Lazy PEM generator helper (used by Cell 3 only when LB_CERT_MODE=generate)
# ----------------------------
def generate_self_signed_lb_pems(
    kind: str,
    curve_name: str,
    rsa_bits: int,
    cn: str,
    days: int,
    out_dir: str,
):
    try:
        from cryptography import x509
        from cryptography.x509.oid import NameOID
        from cryptography.hazmat.primitives import hashes, serialization
        from cryptography.hazmat.primitives.asymmetric import ec, rsa
        from cryptography.hazmat.backends import default_backend
        from datetime import datetime as _dt2, timedelta
    except ImportError:
        import sys
        import subprocess

        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "cryptography"])
        from cryptography import x509
        from cryptography.x509.oid import NameOID
        from cryptography.hazmat.primitives import hashes, serialization
        from cryptography.hazmat.primitives.asymmetric import ec, rsa
        from cryptography.hazmat.backends import default_backend
        from datetime import datetime as _dt2, timedelta

    kind_l = (kind or "ecdsa").strip().lower()
    out_dir_abs = _expand(out_dir or "./local-lb-pems")
    os.makedirs(out_dir_abs, exist_ok=True)

    if kind_l == "ecdsa":
        curve_l = (curve_name or "secp256r1").strip().lower()
        curve_map = {
            "secp256r1": ec.SECP256R1(),
            "prime256v1": ec.SECP256R1(),
            "p-256": ec.SECP256R1(),
        }
        curve = curve_map.get(curve_l, ec.SECP256R1())
        key = ec.generate_private_key(curve, backend=default_backend())
    else:
        key = rsa.generate_private_key(
            public_exponent=65537,
            key_size=int(rsa_bits or 2048),
            backend=default_backend(),
        )

    cn_value = (cn or "lb.local").strip()
    subject = issuer = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, cn_value)])

    builder = (
        x509.CertificateBuilder()
        .subject_name(subject)
        .issuer_name(issuer)
        .public_key(key.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(_dt2.utcnow())
        .not_valid_after(_dt2.utcnow() + timedelta(days=int(days or 3650)))
        .add_extension(x509.BasicConstraints(ca=False, path_length=None), critical=True)
    )

    try:
        builder = builder.add_extension(
            x509.SubjectAlternativeName([x509.DNSName(cn_value)]),
            critical=False,
        )
    except Exception:
        pass

    cert = builder.sign(private_key=key, algorithm=hashes.SHA256(), backend=default_backend())

    key_pem = key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption(),
    )
    cert_pem = cert.public_bytes(serialization.Encoding.PEM)

    base = (LB_PEM_BASENAME or "lb_current").strip()
    key_path = os.path.join(out_dir_abs, f"{base}.key.pem")
    cert_path = os.path.join(out_dir_abs, f"{base}.cert.pem")
    ca_path = os.path.join(out_dir_abs, f"{base}.ca.pem")

    with open(key_path, "wb") as f:
        f.write(key_pem)
    with open(cert_path, "wb") as f:
        f.write(cert_pem)
    with open(ca_path, "wb") as f:
        f.write(b"")

    return cert_path, key_path, ca_path


# ----------------------------
# Local SSH defaults
# ----------------------------
_default_ssh_pub = _expand(_env_str("SSH_PUBLIC_KEY_PATH", "~/.ssh/id_rsa.pub"))
_default_ssh_priv = _expand(_env_str("SSH_PRIVATE_KEY_PATH", "~/.ssh/id_rsa"))

SSH_PUBLIC_KEY_PATH = _default_ssh_pub if _exists(_default_ssh_pub) else ""
SSH_PRIVATE_KEY_PATH = _default_ssh_priv if _exists(_default_ssh_priv) else ""


# ----------------------------
# Output and run identity
# ----------------------------
OUTPUT_DIR = os.path.abspath(_env_str("OUTPUT_DIR", "./results"))
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

TS_UTC = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID = _env_str("RUN_ID", f"flb_concurrency_{TS_UTC}")


# ----------------------------
# Placeholders populated by later cells
# ----------------------------
SELECTED_REGION = None
COMPARTMENT_ID = ""
AD_A = ""
AD_B = ""
IMAGE_ID = ""
SSH_PUBLIC_KEY_CONTENT = ""

LB_ID = ""
LB_IDS = []
LB_IPS_SINGLE = []
LB_IPS_MULTI = []
LB_IPS_FLAT = []
TARGET_HOST = ""
TARGET_URL = ""

GEN_IPS_V4 = []
GEN_IPS_V6 = []
BACKEND_IPS = []


# ----------------------------
# Docs metadata (kept for downstream/UI visibility)
# ----------------------------
REQUIRED_ENV_DOC = {
    "OCI_CONFIG_FILE": "Path to OCI config file (default: ~/.oci/config).",
    "OCI_PROFILE": "OCI profile in config file (default: DEFAULT).",
}

OPTIONAL_ENV_DOC = {
    "TARGET_CONCURRENCY": "Desired concurrent sessions (default: 5000000).",
    "AVG_SESSION_SEC": "Average session lifetime in seconds (default: 1200).",
    "HEARTBEAT_INTERVAL_SEC": "Upload interval in seconds (default: 50).",
    "UPLOAD_BYTES": "Per-heartbeat upload bytes (default: 5120).",
    "H2_CLIENT_PERCENT": "Percent of H2-capable clients (default: 100).",
    "PROTOCOL_PROFILE": "http2_preferred | https_h1_compatible | tcp_ppv2.",
    "RUN_TARGET_USERS": "Primary run scenario users for Cell 12 math guidance (default: 50000).",
    "RUN_SCENARIO_USERS": "Comma-separated scenario users for guidance table.",
    "LB_VISIBILITY": "public | private (default: public).",
    "LB_ALLOWED_CIDR_V4": "Public LB only: allowed IPv4 ingress CIDR (default: 0.0.0.0/0).",
    "LB_ALLOWED_CIDR_V6": "Public LB only: allowed IPv6 ingress CIDR (default: ::/0).",
    "LB_PRIVATE_ALLOWED_CIDR_V4": "Private LB only: IPv4 ingress CIDR or 'auto'.",
    "LB_PRIVATE_ALLOWED_CIDR_V6": "Private LB only: IPv6 ingress CIDR or 'auto'.",
    "BACKEND_COUNT": "Backend instance count (default: 120).",
    "GENERATOR_COUNT": "Generator instance count (default: 1).",
}

print(f"Cell 2 complete: config loaded from {OCI_CONFIG_FILE} [{OCI_PROFILE}]")
print(
    json.dumps(
        {
            "mode": TEST_MODE,
            "protocol_profile": PROTOCOL_PROFILE,
            "target_concurrency": TARGET_CONCURRENCY,
            "avg_session_sec": AVG_SESSION_SEC,
            "heartbeat_interval_sec": HEARTBEAT_INTERVAL_SEC,
            "target_opens_per_sec": round(TARGET_OPENS_PER_SEC, 3),
            "run_target_users": RUN_TARGET_USERS,
            "run_spawn_rate_users_per_sec": round(RUN_SPAWN_RATE, 3),
            "run_expected_steady_rps": round(RUN_TARGET_EXPECTED_HEARTBEAT_RPS, 3),
            "backend_count": BACKEND_COUNT,
            "generator_count": GENERATOR_COUNT,
            "lb_visibility": LB_VISIBILITY,
            "output_dir": OUTPUT_DIR,
            "run_id": RUN_ID,
        },
        indent=2,
    )
)
print("NEXT: Run Cell 3. Ensure To Use This Image (Oracle-Linux-9.7-2026.03.31-0 | Oracle Linux 9 | 2026-04-23 |)")


In [ ]:
# Cell 3 — Enhanced Configuration UI (region/compartment/image discovery first, then cert mode)
import os
import ipaddress
import json
from pathlib import Path
import oci
import ipywidgets as widgets
from IPython.display import display, HTML
from ipyfilechooser import FileChooser
# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def _g(name, default=None):
    return globals().get(name, default)
def _d(name, fallback=""):
    v = _g(name, None)
    if v is None or str(v).strip() == "":
        v = os.environ.get(name, fallback)
    return v
def _setenv(name, value):
    os.environ[name] = "" if value is None else str(value)
def _to_int(v, d=0):
    try:
        return int(v)
    except Exception:
        return d
def _to_float(v, d=0.0):
    try:
        return float(v)
    except Exception:
        return d
def _to_bool(v, d=False):
    if isinstance(v, bool):
        return v
    if v is None:
        return d
    s = str(v).strip().lower()
    if s in {"1", "true", "yes", "y", "on"}:
        return True
    if s in {"0", "false", "no", "n", "off"}:
        return False
    return d
def _expand(p):
    return os.path.expanduser(p) if p else ""
def _html_escape(s):
    return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
def _validate_cidr(text, version):
    try:
        n = ipaddress.ip_network((text or "").strip(), strict=False)
    except Exception as e:
        raise ValueError(f"Invalid CIDR '{text}': {e}") from e
    if n.version != version:
        raise ValueError(f"CIDR '{text}' must be IPv{version}.")
    return str(n)
def _validate_cidr_or_auto(text, version):
    t = (text or "").strip()
    if t.lower() == "auto":
        return "auto"
    return _validate_cidr(t, version)
def _first_nonempty(*vals):
    for v in vals:
        if v is None:
            continue
        s = str(v).strip()
        if s:
            return s
    return ""
def _first_existing(*paths):
    for p in paths:
        if not p:
            continue
        ep = _expand(str(p).strip())
        if ep and os.path.exists(ep):
            return ep
    return ""
def _list_all(fn, *args, **kwargs):
    return oci.pagination.list_call_get_all_results(fn, *args, **kwargs).data
def _safe_set_options(dd, options, preferred=None):
    if not options:
        options = [("No options available", "")]
    values = [v for _, v in options]
    current = dd.value if dd.value in values else None
    if preferred in values:
        selected = preferred
    elif current in values:
        selected = current
    else:
        selected = values[0]
    dd.options = options
    dd.value = selected
def _is_flex(shape_name):
    return bool(shape_name and str(shape_name).endswith(".Flex"))
def _shape_bounds(shape_obj):
    oc = getattr(shape_obj, "ocpu_options", None)
    mo = getattr(shape_obj, "memory_options", None)
    min_o = _to_float(getattr(oc, "min", getattr(oc, "min_in_ocpus", 1.0)), 1.0)
    max_o = _to_float(getattr(oc, "max", getattr(oc, "max_in_ocpus", 64.0)), 64.0)
    min_m = _to_float(getattr(mo, "min_in_g_bs", getattr(mo, "min_in_gbs", 1.0)), 1.0)
    max_m = _to_float(getattr(mo, "max_in_g_bs", getattr(mo, "max_in_gbs", 1024.0)), 1024.0)
    dpo = _to_float(getattr(mo, "default_per_ocpu_in_g_bs", getattr(mo, "default_per_ocpu_in_gbs", 16.0)), 16.0)
    return min_o, max_o, min_m, max_m, dpo
def _fc_selected(fc):
    try:
        return _expand((fc.selected or "").strip())
    except Exception:
        return ""
def _row(label_text, *controls, label_width="360px"):
    lbl = widgets.HTML(
        f"<span class='nb3-lbl'>{label_text}</span>",
        layout=widgets.Layout(width=label_width, min_width=label_width),
    )
    for c in controls:
        if hasattr(c, "layout"):
            c.layout.flex = "1 1 auto"
            c.layout.width = "auto"
    return widgets.HBox([lbl] + list(controls), layout=widgets.Layout(width="100%", align_items="center", gap="12px"))
def _section(title, *children):
    return widgets.VBox(
        [widgets.HTML(f"<div class='nb3-title'>{title}</div>"), *children],
        layout=widgets.Layout(width="100%", align_items="stretch"),
        _dom_classes=["nb3-section"],
    )
# -----------------------------------------------------------------------------
# Required baseline
# -----------------------------------------------------------------------------
OCI_CONFIG_FILE = _first_nonempty(_d("OCI_CONFIG_FILE", "~/.oci/config"), "~/.oci/config")
OCI_PROFILE = _first_nonempty(_d("OCI_PROFILE", "DEFAULT"), "DEFAULT")
try:
    cfg_home = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)
except Exception as e:
    raise RuntimeError(f"Unable to load OCI config from {OCI_CONFIG_FILE} [{OCI_PROFILE}]: {e}")
_cfg = _g("_cfg", cfg_home)
TENANCY_OCID = _first_nonempty(_g("TENANCY_OCID", ""), _cfg.get("tenancy", ""))
if not TENANCY_OCID:
    raise RuntimeError("TENANCY_OCID is missing. Run Cell 2 first.")
if _g("generate_self_signed_lb_pems", None) is None:
    print("Warning: generate_self_signed_lb_pems not found. Cert generate mode will fail until Cell 2 is re-run.")
# -----------------------------------------------------------------------------
# Styles
# -----------------------------------------------------------------------------
display(
    HTML(
        """
<style>
  .nb3-container { max-width: 1500px; margin: 0 auto; }
  .nb3-section { border: 1px solid #e0e0e0; background: #f8f9fb; padding: 12px; margin: 8px 0; border-left: 4px solid #d2d7e5; }
  .nb3-title { font-weight: 600; margin: 0 0 6px 0; text-align: left; }
  .nb3-lbl, .widget-label {
    white-space: normal !important; overflow: visible !important; text-overflow: clip !important; line-height: 1.2;
  }
  .widget-label { min-width: 340px !important; }
  .issues-strip { background: #fdecea; color: #ba1a1a; padding: 6px 10px; border: 1px solid #f5c2c7; border-radius: 6px; font-size: 13px; margin: 0 0 6px 0; }
  .ok-strip { background: #e8f4fd; color: #0b5394; padding: 6px 10px; border: 1px solid #a4c2f4; border-radius: 6px; font-size: 13px; margin: 0 0 6px 0; }
  .hint-strip { background: #fff8e1; color: #6b4f00; padding: 6px 10px; border: 1px solid #f0d98c; border-radius: 6px; font-size: 13px; margin: 0 0 6px 0; }
</style>
"""
    )
)
# -----------------------------------------------------------------------------
# Discovery bootstrap (OKE-style region/compartment preload)
# -----------------------------------------------------------------------------
idc_home = oci.identity.IdentityClient(cfg_home)
subs = _list_all(idc_home.list_region_subscriptions, TENANCY_OCID)
region_names = sorted({s.region_name for s in subs if getattr(s, "status", "READY") in ("READY", "ACTIVE", None)})
if not region_names:
    raise RuntimeError("No subscribed regions returned for tenancy.")
comp_data = _list_all(
    idc_home.list_compartments,
    TENANCY_OCID,
    compartment_id_in_subtree=True,
    access_level="ACCESSIBLE",
)
comp_active = [c for c in comp_data if getattr(c, "lifecycle_state", "") == "ACTIVE"]
comp_options = [("TENANCY_ROOT", TENANCY_OCID)] + sorted(
    [(f"{c.name} ({c.id[-8:]})", c.id) for c in comp_active], key=lambda x: x[0].lower()
)
REGION_DEFAULT = _first_nonempty(_d("REGION", cfg_home.get("region", "")), region_names[0])
if REGION_DEFAULT not in region_names:
    REGION_DEFAULT = region_names[0]
COMP_DEFAULT = _first_nonempty(_d("COMPARTMENT_ID", TENANCY_OCID), TENANCY_OCID)
if COMP_DEFAULT not in [v for _, v in comp_options]:
    COMP_DEFAULT = TENANCY_OCID
# -----------------------------------------------------------------------------
# Widgets
# -----------------------------------------------------------------------------
status_html = widgets.HTML("<div class='ok-strip'><b>Cell 3</b>: loading discovery...</div>")
issues_html = widgets.HTML("")
preview_out = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))
apply_out = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))
_state = {"shape_map": {}}
# Discovery
region_dd = widgets.Dropdown(options=[(r, r) for r in region_names], value=REGION_DEFAULT)
comp_dd = widgets.Dropdown(options=comp_options, value=COMP_DEFAULT)
ad_dd = widgets.Dropdown(options=[("Loading ADs...", "")])
shape_filter_in = widgets.Text(value="", placeholder="Example: E5.Flex")
backend_shape_dd = widgets.Dropdown(options=[("Loading shapes...", "")])
generator_shape_dd = widgets.Dropdown(options=[("Loading shapes...", "")])
backend_ocpus_in = widgets.BoundedFloatText(value=_to_float(_d("BACKEND_OCPUS", 4), 4.0), min=1, max=256, step=1)
backend_mem_in = widgets.BoundedFloatText(value=_to_float(_d("BACKEND_MEMORY_GB", 64), 64.0), min=1, max=4096, step=1)
generator_ocpus_in = widgets.BoundedFloatText(value=_to_float(_d("GENERATOR_OCPUS", 8), 8.0), min=1, max=256, step=1)
generator_mem_in = widgets.BoundedFloatText(value=_to_float(_d("GENERATOR_MEMORY_GB", 128), 128.0), min=1, max=4096, step=1)
image_dd = widgets.Dropdown(options=[("Loading images...", "")])
reload_discovery_btn = widgets.Button(description="Reload Discovery", icon="refresh", button_style="info")
refresh_images_btn = widgets.Button(description="Refresh Images", icon="refresh")
# Cert
cert_mode_dd = widgets.Dropdown(
    options=[("Browse existing PEM files", "browse"), ("Generate self-signed PEM files", "generate")],
    value=(_d("LB_CERT_MODE", "browse") if _d("LB_CERT_MODE", "browse") in {"browse", "generate"} else "browse"),
)
fc_home = _expand("~")
fc_cert = FileChooser(path=fc_home, title="Select LB certificate PEM/CRT/CER", show_hidden=True, use_dir_icons=True, show_only_dirs=False, filter_pattern="*")
fc_key = FileChooser(path=fc_home, title="Select LB private key PEM/KEY", show_hidden=True, use_dir_icons=True, show_only_dirs=False, filter_pattern="*")
fc_ca = FileChooser(path=fc_home, title="Optional: select LB CA chain PEM", show_hidden=True, use_dir_icons=True, show_only_dirs=False, filter_pattern="*")
show_hidden_cb = widgets.Checkbox(value=True, description="Show hidden files")
jump_path_in = widgets.Text(value=fc_home)
jump_btn = widgets.Button(description="Go", icon="sign-in")
lb_cert_path_in = widgets.Text(value=_d("LB_CERT_PEM_PATH", ""))
lb_key_path_in = widgets.Text(value=_d("LB_KEY_PEM_PATH", ""))
lb_ca_path_in = widgets.Text(value=_d("LB_CA_PEM_PATH", ""))
cert_kind_dd = widgets.Dropdown(options=[("ECDSA P-256", "ecdsa"), ("RSA", "rsa")], value=(_d("LB_CERT_KIND", "ecdsa") if _d("LB_CERT_KIND", "ecdsa") in {"ecdsa", "rsa"} else "ecdsa"))
ecdsa_curve_dd = widgets.Dropdown(options=[("secp256r1 (prime256v1)", "secp256r1")], value="secp256r1")
rsa_bits_dd = widgets.Dropdown(options=[("2048", 2048), ("3072", 3072), ("4096", 4096)], value=(_to_int(_d("LB_RSA_BITS", 2048), 2048) if _to_int(_d("LB_RSA_BITS", 2048), 2048) in {2048, 3072, 4096} else 2048))
cert_cn_in = widgets.Text(value=_d("LB_CERT_CN", "lb.local"))
cert_days_in = widgets.BoundedIntText(value=_to_int(_d("LB_CERT_DAYS", 365), 365), min=1, max=36500)
pem_output_dir_in = widgets.Text(value=_d("LB_PEM_OUTPUT_DIR", "./local-lb-pems"))
pem_basename_in = widgets.Text(value=_d("LB_PEM_BASENAME", "lb_current"))
overwrite_generate_cb = widgets.Checkbox(value=False, description="Overwrite existing generated PEM files")
# SSH
fc_ssh_pub = FileChooser(path=fc_home, title="Select SSH public key file (*.pub)", show_hidden=True, use_dir_icons=True, show_only_dirs=False, filter_pattern="*")
fc_ssh_priv = FileChooser(path=fc_home, title="Select SSH private key file", show_hidden=True, use_dir_icons=True, show_only_dirs=False, filter_pattern="*")
use_ssh_override_cb = widgets.Checkbox(value=False, description="Use manual SSH path overrides (advanced)")
ssh_pub_override_in = widgets.Text(value=_d("SSH_PUBLIC_KEY_PATH", ""))
ssh_priv_override_in = widgets.Text(value=_d("SSH_PRIVATE_KEY_PATH", ""))
# Topology/runtime
lb_topology_dd = widgets.Dropdown(options=[("Single LB", "single"), ("Multi LB", "multi")], value=(_d("LB_TOPOLOGY", "single") if _d("LB_TOPOLOGY", "single") in {"single", "multi"} else "single"))
lb_count_in = widgets.IntText(value=_to_int(_d("LB_COUNT", 1), 1))
backend_count_in = widgets.IntText(value=_to_int(_d("BACKEND_COUNT", 120), 120))
generator_count_in = widgets.IntText(value=_to_int(_d("GENERATOR_COUNT", 1), 1))
lb_min_mbps_in = widgets.IntText(value=_to_int(_d("LB_MIN_MBPS", 100), 100))
lb_max_mbps_in = widgets.IntText(value=_to_int(_d("LB_MAX_MBPS", 8000), 8000))
enable_ipv6_frontend_cb = widgets.Checkbox(value=_to_bool(_d("ENABLE_IPV6_FRONTEND", True), True), description="Enable IPv6 frontend VIP")
use_ssh_ipv6_cb = widgets.Checkbox(value=_to_bool(_d("USE_SSH_IPV6", False), False), description="Use IPv6 for SSH")
ssh_allowed_cidr_in = widgets.Text(value=_d("SSH_ALLOWED_CIDR", "0.0.0.0/0"))
ssh_allowed_v6_cidr_in = widgets.Text(value=_d("SSH_ALLOWED_V6_CIDR", "::/0"))
lb_visibility_dd = widgets.Dropdown(
    options=[("Public LB", "public"), ("Private LB", "private")],
    value=(_d("LB_VISIBILITY", "public") if _d("LB_VISIBILITY", "public") in {"public", "private"} else "public"),
)
lb_allowed_cidr_v4_in = widgets.Text(value=_d("LB_ALLOWED_CIDR_V4", "0.0.0.0/0"))
lb_allowed_cidr_v6_in = widgets.Text(value=_d("LB_ALLOWED_CIDR_V6", "::/0"))
lb_private_allowed_cidr_v4_in = widgets.Text(value=_d("LB_PRIVATE_ALLOWED_CIDR_V4", "auto"))
lb_private_allowed_cidr_v6_in = widgets.Text(value=_d("LB_PRIVATE_ALLOWED_CIDR_V6", "auto"))
lb_private_cidr_hint = widgets.HTML("<div class='hint-strip'><b>Private LB mode:</b> Use <code>auto</code> to derive CIDRs from generator subnet(s) in Cell 5.</div>")
show_overrides_cb = widgets.Checkbox(value=bool(_first_nonempty(_d("MASTER_IPV6_OVERRIDE", ""), _d("MASTER_PRIVATE_IP_OVERRIDE", ""))), description="Show control-plane overrides")
master_ipv6_override_in = widgets.Text(value=_d("MASTER_IPV6_OVERRIDE", ""))
master_private_ip_override_in = widgets.Text(value=_d("MASTER_PRIVATE_IP_OVERRIDE", ""))
protocol_profile_dd = widgets.Dropdown(
    options=[
        ("HTTP/2 preferred (with HTTP/1.1 support)", "http2_preferred"),
        ("HTTPS HTTP/1.1 compatible", "https_h1_compatible"),
        ("TCP with PROXY protocol v2", "tcp_ppv2"),
    ],
    value=(_d("PROTOCOL_PROFILE", "http2_preferred") if _d("PROTOCOL_PROFILE", "http2_preferred") in {"http2_preferred", "https_h1_compatible", "tcp_ppv2"} else "http2_preferred"),
)
h2_client_percent_in = widgets.BoundedFloatText(value=_to_float(_d("H2_CLIENT_PERCENT", 100), 100.0), min=0.0, max=100.0, step=1.0)
target_concurrency_in = widgets.IntText(value=_to_int(_d("TARGET_CONCURRENCY", 5000000), 5000000))
avg_session_sec_in = widgets.IntText(value=_to_int(_d("AVG_SESSION_SEC", 1200), 1200))
heartbeat_interval_sec_in = widgets.IntText(value=_to_int(_d("HEARTBEAT_INTERVAL_SEC", 50), 50))
upload_bytes_in = widgets.IntText(value=_to_int(_d("UPLOAD_BYTES", 5120), 5120))
session_ttl_jitter_enabled_cb = widgets.Checkbox(value=_to_bool(_d("SESSION_TTL_JITTER_ENABLED", False), False), description="Enable TTL jitter")
session_ttl_jitter_pct_in = widgets.BoundedFloatText(value=_to_float(_d("SESSION_TTL_JITTER_PCT", 10), 10.0), min=0.0, max=95.0, step=0.5)
steady_state_tolerance_pct_in = widgets.BoundedFloatText(value=_to_float(_d("STEADY_STATE_TOLERANCE_PCT", 10), 10.0), min=0.0, max=100.0, step=0.5)
steady_state_eval_window_sec_in = widgets.IntText(value=_to_int(_d("STEADY_STATE_EVAL_WINDOW_SEC", 600), 600))
ramp_up_sec_in = widgets.IntText(value=_to_int(_d("RAMP_UP_SEC", 1200), 1200))
run_hold_sec_in = widgets.IntText(value=_to_int(_d("RUN_HOLD_SEC", 3600), 3600))
health_endpoint_in = widgets.Text(value=_d("HEALTH_ENDPOINT_PATH", "/healthz"))
upload_endpoint_in = widgets.Text(value=_d("UPLOAD_ENDPOINT_PATH", "/upload_5k"))
upload_ack_enabled_cb = widgets.Checkbox(value=_to_bool(_d("UPLOAD_ACK_ENABLED", True), True), description="Enable upload acknowledgment response")
upload_ack_status_in = widgets.BoundedIntText(value=_to_int(_d("UPLOAD_ACK_STATUS", 200), 200), min=200, max=299)
upload_ack_body_in = widgets.Text(value=_d("UPLOAD_ACK_BODY", "ack"))
ui_enable_cb = widgets.Checkbox(value=_to_bool(_d("UI_ENABLE", True), True), description="Enable Locust UI")
ui_expose_mode_dd = widgets.Dropdown(options=[("tunnel", "tunnel"), ("nsg", "nsg")], value=(_d("UI_EXPOSE_MODE", "tunnel") if _d("UI_EXPOSE_MODE", "tunnel") in {"tunnel", "nsg"} else "tunnel"))
ui_web_port_in = widgets.BoundedIntText(value=_to_int(_d("UI_WEB_PORT", 8089), 8089), min=1024, max=65535)
ui_allowed_cidr_in = widgets.Text(value=_d("UI_ALLOWED_CIDR", "0.0.0.0/0"))
headless_enable_cb = widgets.Checkbox(value=_to_bool(_d("HEADLESS_ENABLE", True), True), description="Enable headless mode")
expect_workers_strict_cb = widgets.Checkbox(value=_to_bool(_d("EXPECT_WORKERS_STRICT", False), False), description="Strict worker-count check")
grace_sec_in = widgets.IntText(value=_to_int(_d("GRACE_SEC", 300), 300))
workers_per_host_dd = widgets.Dropdown(options=[("auto", "auto"), ("fixed", "fixed")], value=(_d("WORKERS_PER_HOST", "auto") if _d("WORKERS_PER_HOST", "auto") in {"auto", "fixed"} else "auto"))
fixed_workers_in = widgets.IntText(value=_to_int(_d("MIN_WORKERS_PER_HOST", 1), 1))
cpu_reserve_in = widgets.IntText(value=_to_int(_d("CPU_RESERVE", 1), 1))
min_workers_in = widgets.IntText(value=_to_int(_d("MIN_WORKERS_PER_HOST", 1), 1))
max_workers_in = widgets.IntText(value=_to_int(_d("MAX_WORKERS_PER_HOST", 8), 8))
locust_verify_tls_cb = widgets.Checkbox(value=_to_bool(_d("LOCUST_VERIFY_TLS", True), True), description="Verify TLS certs in clients")
locust_connect_timeout_ms_in = widgets.IntText(value=_to_int(_d("LOCUST_CONNECT_TIMEOUT_MS", 10000), 10000))
locust_read_timeout_ms_in = widgets.IntText(value=_to_int(_d("LOCUST_READ_TIMEOUT_MS", 10000), 10000))
output_dir_in = widgets.Text(value=_d("OUTPUT_DIR", "./results"))
run_id_in = widgets.Text(value=_d("RUN_ID", "flb_concurrency_manual"))
preview_btn = widgets.Button(description="Preview Resolved Paths", icon="eye")
apply_btn = widgets.Button(description="Apply Selections", button_style="primary", icon="check")
print_btn = widgets.Button(description="Print Effective Config", button_style="info", icon="list")
reset_btn = widgets.Button(description="Reset From Globals", icon="history", button_style="warning")
# -----------------------------------------------------------------------------
# Discovery logic
# -----------------------------------------------------------------------------
def _make_region_clients(region):
    c = dict(cfg_home)
    c["region"] = region
    return oci.identity.IdentityClient(c), oci.core.ComputeClient(c)
def _load_ads(idc_region):
    ads = _list_all(idc_region.list_availability_domains, TENANCY_OCID)
    names = sorted([ad.name for ad in ads]) or ["AD-1"]
    return [(n, n) for n in names]
def _load_shapes(compute_region, compartment_id):
    data = _list_all(compute_region.list_shapes, compartment_id)
    active = [s for s in data if getattr(s, "lifecycle_state", "ACTIVE") in ("ACTIVE", None)]
    shape_map = {s.shape: s for s in active if getattr(s, "shape", "")}
    names = sorted(shape_map.keys())
    f = (shape_filter_in.value or "").strip().lower()
    if f:
        names = [n for n in names if f in n.lower()] or names
    return ([(n, n) for n in names] if names else [("No shapes found", "")]), shape_map
def _load_images(compute_region, compartment_id, backend_shape, generator_shape):
    def _query(shape_name):
        kwargs = {
            "operating_system": "Oracle Linux",
            "sort_by": "TIMECREATED",
            "sort_order": "DESC",
        }
        if shape_name:
            kwargs["shape"] = shape_name
        rows = _list_all(compute_region.list_images, compartment_id, **kwargs)
        return [r for r in rows if getattr(r, "lifecycle_state", "AVAILABLE") == "AVAILABLE"]
    shape_candidates = list(dict.fromkeys([s for s in [backend_shape, generator_shape] if s]))
    imgs = []
    if shape_candidates:
        lists = [_query(s) for s in shape_candidates]
        common_ids = set(getattr(im, "id", "") for im in lists[0] if getattr(im, "id", "")) if lists else set()
        for lst in lists[1:]:
            common_ids &= {getattr(im, "id", "") for im in lst if getattr(im, "id", "")}
        src = lists[0] if lists else []
        imgs = [im for im in src if getattr(im, "id", "") in common_ids]
    if not imgs:
        imgs = _query("")
    options, seen = [], set()
    for im in imgs[:300]:
        iid = getattr(im, "id", "")
        if not iid or iid in seen:
            continue
        seen.add(iid)
        name = getattr(im, "display_name", "Oracle Linux image")
        osv = getattr(im, "operating_system_version", "")
        created = getattr(im, "time_created", None)
        dt = created.strftime("%Y-%m-%d") if created else ""
        options.append((f"{name} | Oracle Linux {osv} | {dt} | {iid[:24]}...", iid))
    return options or [("No compatible Oracle Linux images discovered", "")]
def _refresh_flex_controls():
    for shape_name, ocpu_w, mem_w in [
        (backend_shape_dd.value, backend_ocpus_in, backend_mem_in),
        (generator_shape_dd.value, generator_ocpus_in, generator_mem_in),
    ]:
        if not _is_flex(shape_name):
            ocpu_w.layout.display = "none"
            mem_w.layout.display = "none"
            continue
        shp = _state["shape_map"].get(shape_name)
        if shp is not None:
            min_o, max_o, min_m, max_m, dpo = _shape_bounds(shp)
            ocpu_w.min, ocpu_w.max = min_o, max_o
            mem_w.min, mem_w.max = min_m, max_m
            if ocpu_w.value < min_o or ocpu_w.value > max_o:
                ocpu_w.value = min_o
            if mem_w.value < min_m or mem_w.value > max_m:
                mem_w.value = max(min_m, min(max_m, ocpu_w.value * dpo))
        ocpu_w.layout.display = ""
        mem_w.layout.display = ""
def _refresh_region_data(*_):
    issues_html.value = ""
    status_html.value = "<div class='ok-strip'><b>Cell 3</b>: loading region-scoped AD/shape/image data...</div>"
    try:
        idc_region, compute_region = _make_region_clients(region_dd.value)
        _safe_set_options(ad_dd, _load_ads(idc_region), preferred=_d("AD_A", "") or ad_dd.value)
        shape_opts, shape_map = _load_shapes(compute_region, comp_dd.value)
        _state["shape_map"] = shape_map
        _safe_set_options(backend_shape_dd, shape_opts, preferred=_d("BACKEND_SHAPE", "") or backend_shape_dd.value)
        _safe_set_options(generator_shape_dd, shape_opts, preferred=_d("GENERATOR_SHAPE", "") or generator_shape_dd.value)
        _refresh_flex_controls()
        _safe_set_options(image_dd, _load_images(compute_region, comp_dd.value, backend_shape_dd.value, generator_shape_dd.value), preferred=_d("IMAGE_ID", "") or image_dd.value)
        status_html.value = "<div class='ok-strip'><b>Cell 3</b>: discovery ready. Review selections and click Apply.</div>"
    except Exception as e:
        status_html.value = "<div class='issues-strip'><b>Cell 3</b>: discovery failed.</div>"
        issues_html.value = (
            "<div class='issues-strip'><b>Discovery failed.</b><br>"
            f"Error: {_html_escape(repr(e))}<br>"
            "Check IAM permissions for compartments, ADs, shapes, and images.</div>"
        )
def _refresh_images_only(*_):
    try:
        _, compute_region = _make_region_clients(region_dd.value)
        _safe_set_options(
            image_dd,
            _load_images(compute_region, comp_dd.value, backend_shape_dd.value, generator_shape_dd.value),
            preferred=image_dd.value,
        )
    except Exception as e:
        issues_html.value = f"<div class='issues-strip'><b>Image refresh failed.</b> {_html_escape(repr(e))}</div>"
# -----------------------------------------------------------------------------
# UI state sync
# -----------------------------------------------------------------------------
def _sync_topology_state():
    if lb_topology_dd.value == "single":
        lb_count_in.value = 1
        lb_count_in.disabled = True
    else:
        lb_count_in.disabled = False
def _sync_lb_visibility_state():
    is_public = lb_visibility_dd.value == "public"
    row_lb_allowed_cidr_v4.layout.display = "" if is_public else "none"
    row_lb_allowed_cidr_v6.layout.display = "" if is_public else "none"
    row_lb_private_allowed_cidr_v4.layout.display = "none" if is_public else ""
    row_lb_private_allowed_cidr_v6.layout.display = "none" if is_public else ""
    lb_private_cidr_hint.layout.display = "none" if is_public else ""
def _sync_jitter_state():
    session_ttl_jitter_pct_in.disabled = not session_ttl_jitter_enabled_cb.value
def _sync_override_state():
    vis = show_overrides_cb.value
    master_ipv6_override_in.layout.display = "" if vis else "none"
    master_private_ip_override_in.layout.display = "" if vis else "none"
def _sync_worker_state():
    fixed = workers_per_host_dd.value == "fixed"
    fixed_workers_in.disabled = not fixed
    min_workers_in.disabled = fixed
    max_workers_in.disabled = fixed
def _sync_cert_mode_state():
    mode = cert_mode_dd.value
    browse_cert_box.layout.display = "" if mode == "browse" else "none"
    generate_cert_box.layout.display = "" if mode == "generate" else "none"
    if mode == "generate":
        ecdsa_curve_dd.layout.display = "" if cert_kind_dd.value == "ecdsa" else "none"
        rsa_bits_dd.layout.display = "" if cert_kind_dd.value == "rsa" else "none"
    else:
        ecdsa_curve_dd.layout.display = "none"
        rsa_bits_dd.layout.display = "none"
def _sync_ssh_override_state():
    vis = use_ssh_override_cb.value
    ssh_pub_override_in.disabled = not vis
    ssh_priv_override_in.disabled = not vis
def _jump_filechoosers(path_text):
    p = _expand(path_text or "~")
    for fc in (fc_cert, fc_key, fc_ca, fc_ssh_pub, fc_ssh_priv):
        try:
            fc.reset(path=p)
            fc.show_hidden = bool(show_hidden_cb.value)
        except Exception:
            pass
# -----------------------------------------------------------------------------
# Resolve paths / validate
# -----------------------------------------------------------------------------
def _resolve_ssh_paths():
    if use_ssh_override_cb.value:
        return _expand((ssh_pub_override_in.value or "").strip()), _expand((ssh_priv_override_in.value or "").strip())
    return _first_existing(_fc_selected(fc_ssh_pub), _d("SSH_PUBLIC_KEY_PATH", "")), _first_existing(_fc_selected(fc_ssh_priv), _d("SSH_PRIVATE_KEY_PATH", ""))
def _resolve_cert_paths():
    if cert_mode_dd.value == "browse":
        cert_p = _first_existing(_fc_selected(fc_cert), lb_cert_path_in.value, _d("LB_CERT_PEM_PATH", ""))
        key_p = _first_existing(_fc_selected(fc_key), lb_key_path_in.value, _d("LB_KEY_PEM_PATH", ""))
        ca_candidate = _first_nonempty(_fc_selected(fc_ca), lb_ca_path_in.value, _d("LB_CA_PEM_PATH", ""))
        ca_p = _expand(ca_candidate) if ca_candidate else ""
        if not cert_p:
            raise ValueError("LB cert PEM missing.")
        if not key_p:
            raise ValueError("LB key PEM missing.")
        if ca_p and not os.path.exists(ca_p):
            raise ValueError(f"LB CA PEM path does not exist: {ca_p}")
        return cert_p, key_p, ca_p
    if "generate_self_signed_lb_pems" not in globals():
        raise ValueError("generate_self_signed_lb_pems helper not found. Re-run Cell 2.")
    kind = cert_kind_dd.value
    curve = ecdsa_curve_dd.value
    bits = int(rsa_bits_dd.value)
    cn = (cert_cn_in.value or "lb.local").strip()
    days = int(cert_days_in.value)
    out_dir = _expand((pem_output_dir_in.value or "./local-lb-pems").strip())
    base = (pem_basename_in.value or "lb_current").strip()
    if not base:
        raise ValueError("Generated PEM basename cannot be empty.")
    exp_cert = os.path.join(out_dir, f"{base}.cert.pem")
    exp_key = os.path.join(out_dir, f"{base}.key.pem")
    exp_ca = os.path.join(out_dir, f"{base}.ca.pem")
    if all(os.path.exists(p) for p in (exp_cert, exp_key, exp_ca)) and not overwrite_generate_cb.value:
        return exp_cert, exp_key, exp_ca
    old_base = _g("LB_PEM_BASENAME", "lb_current")
    try:
        globals()["LB_PEM_BASENAME"] = base
        return generate_self_signed_lb_pems(kind, curve, bits, cn, days, out_dir)
    finally:
        globals()["LB_PEM_BASENAME"] = old_base
def _preview_resolved_paths(*_):
    preview_out.clear_output()
    with preview_out:
        try:
            ssh_pub, ssh_priv = _resolve_ssh_paths()
            cert_p, key_p, ca_p = _resolve_cert_paths()
            print("Resolved paths preview:")
            print(f"- SSH public: {ssh_pub}")
            print(f"- SSH private: {ssh_priv}")
            print(f"- LB cert: {cert_p}")
            print(f"- LB key: {key_p}")
            print(f"- LB CA: {ca_p or '(empty)'}")
        except Exception as e:
            print(f"Preview failed: {repr(e)}")
def _validate_inputs():
    issues = []
    req_pairs = [
        (region_dd.value, "Region is required."),
        (comp_dd.value, "Compartment is required."),
        (ad_dd.value, "Availability Domain is required."),
        (backend_shape_dd.value, "Backend shape is required."),
        (generator_shape_dd.value, "Generator shape is required."),
        (image_dd.value, "Oracle Linux image is required."),
    ]
    issues.extend([msg for val, msg in req_pairs if not val])
    if lb_topology_dd.value == "multi" and lb_count_in.value <= 0:
        issues.append("LB count must be > 0 in multi topology.")
    if backend_count_in.value <= 0:
        issues.append("Backend instance count must be > 0.")
    if generator_count_in.value <= 0:
        issues.append("Generator instance count must be > 0.")
    if not (10 <= lb_min_mbps_in.value <= 32000):
        issues.append("LB_MIN_MBPS must be between 10 and 32000.")
    if not (10 <= lb_max_mbps_in.value <= 32000):
        issues.append("LB_MAX_MBPS must be between 10 and 32000.")
    if lb_max_mbps_in.value < lb_min_mbps_in.value:
        issues.append("LB_MAX_MBPS must be >= LB_MIN_MBPS.")
    if lb_visibility_dd.value not in {"public", "private"}:
        issues.append("LB visibility must be public or private.")
    if lb_visibility_dd.value == "public":
        try:
            _validate_cidr(lb_allowed_cidr_v4_in.value, 4)
        except Exception as e:
            issues.append(f"LB_ALLOWED_CIDR_V4 invalid: {e}")
        if enable_ipv6_frontend_cb.value:
            try:
                _validate_cidr(lb_allowed_cidr_v6_in.value, 6)
            except Exception as e:
                issues.append(f"LB_ALLOWED_CIDR_V6 invalid: {e}")
    else:
        try:
            _validate_cidr_or_auto(lb_private_allowed_cidr_v4_in.value, 4)
        except Exception as e:
            issues.append(f"LB_PRIVATE_ALLOWED_CIDR_V4 invalid: {e}")
        if enable_ipv6_frontend_cb.value:
            try:
                _validate_cidr_or_auto(lb_private_allowed_cidr_v6_in.value, 6)
            except Exception as e:
                issues.append(f"LB_PRIVATE_ALLOWED_CIDR_V6 invalid: {e}")
    num_checks = [
        (target_concurrency_in.value > 0, "TARGET_CONCURRENCY must be > 0."),
        (avg_session_sec_in.value > 0, "AVG_SESSION_SEC must be > 0."),
        (heartbeat_interval_sec_in.value > 0, "HEARTBEAT_INTERVAL_SEC must be > 0."),
        (upload_bytes_in.value > 0, "UPLOAD_BYTES must be > 0."),
        (0 <= h2_client_percent_in.value <= 100, "H2_CLIENT_PERCENT must be 0..100."),
        (0 <= session_ttl_jitter_pct_in.value <= 95, "TTL jitter percent must be 0..95."),
        (0 <= steady_state_tolerance_pct_in.value <= 100, "Steady-state tolerance percent must be 0..100."),
        (steady_state_eval_window_sec_in.value >= 30, "Steady-state eval window sec must be >= 30."),
        (ramp_up_sec_in.value > 0, "Ramp-up seconds must be > 0."),
        (run_hold_sec_in.value > 0, "Steady hold seconds must be > 0."),
        (str(health_endpoint_in.value).startswith("/"), "HEALTH_ENDPOINT_PATH must start with '/'."),
        (str(upload_endpoint_in.value).startswith("/"), "UPLOAD_ENDPOINT_PATH must start with '/'."),
        (200 <= int(upload_ack_status_in.value) <= 299, "UPLOAD_ACK_STATUS must be 200..299."),
    ]
    issues.extend([msg for ok, msg in num_checks if not ok])
    worker_checks = [
        (workers_per_host_dd.value in {"auto", "fixed"}, "WORKERS_PER_HOST must be auto or fixed."),
        (fixed_workers_in.value > 0, "Fixed workers per host must be > 0."),
        (min_workers_in.value > 0, "Min workers per host must be > 0."),
        (max_workers_in.value > 0, "Max workers per host must be > 0."),
        (max_workers_in.value >= min_workers_in.value, "Max workers must be >= min workers."),
        (locust_connect_timeout_ms_in.value >= 1000, "LOCUST_CONNECT_TIMEOUT_MS must be >= 1000."),
        (locust_read_timeout_ms_in.value >= 1000, "LOCUST_READ_TIMEOUT_MS must be >= 1000."),
        (bool(output_dir_in.value.strip()), "OUTPUT_DIR cannot be empty."),
        (bool(run_id_in.value.strip()), "RUN_ID cannot be empty."),
    ]
    issues.extend([msg for ok, msg in worker_checks if not ok])
    if _is_flex(backend_shape_dd.value):
        if backend_ocpus_in.value <= 0:
            issues.append("Backend Flex OCPUs must be > 0.")
        if backend_mem_in.value <= 0:
            issues.append("Backend Flex memory GB must be > 0.")
    if _is_flex(generator_shape_dd.value):
        if generator_ocpus_in.value <= 0:
            issues.append("Generator Flex OCPUs must be > 0.")
        if generator_mem_in.value <= 0:
            issues.append("Generator Flex memory GB must be > 0.")
    try:
        ssh_pub, ssh_priv = _resolve_ssh_paths()
        if not ssh_pub or not os.path.exists(ssh_pub):
            issues.append("Valid SSH public key path is required.")
        if not ssh_priv or not os.path.exists(ssh_priv):
            issues.append("Valid SSH private key path is required.")
    except Exception as e:
        issues.append(f"SSH path resolution failed: {e}")
    try:
        _resolve_cert_paths()
    except Exception as e:
        issues.append(f"Certificate path resolution failed: {e}")
    return issues
# -----------------------------------------------------------------------------
# Apply
# -----------------------------------------------------------------------------
def _apply(_):
    global SELECTED_REGION, REGION, COMPARTMENT_ID, AD_A, AD_B, IMAGE_ID
    global SSH_PUBLIC_KEY_PATH, SSH_PRIVATE_KEY_PATH, SSH_PUBLIC_KEY_CONTENT
    global LB_CERT_MODE, LB_CERT_PEM_PATH, LB_KEY_PEM_PATH, LB_CA_PEM_PATH
    global LB_CERT_KIND, LB_ECDSA_CURVE, LB_RSA_BITS, LB_CERT_CN, LB_CERT_DAYS, LB_PEM_OUTPUT_DIR, LB_PEM_BASENAME
    global LB_TOPOLOGY, LB_COUNT, BACKEND_COUNT, GENERATOR_COUNT
    global BACKEND_SHAPE, BACKEND_OCPUS, BACKEND_MEMORY_GB
    global GENERATOR_SHAPE, GENERATOR_OCPUS, GENERATOR_MEMORY_GB
    global LB_MIN_MBPS, LB_MAX_MBPS
    global ENABLE_IPV6_FRONTEND, USE_SSH_IPV6, SSH_ALLOWED_CIDR, SSH_ALLOWED_V6_CIDR
    global LB_VISIBILITY, LB_ALLOWED_CIDR_V4, LB_ALLOWED_CIDR_V6
    global LB_PRIVATE_ALLOWED_CIDR_V4, LB_PRIVATE_ALLOWED_CIDR_V6
    global LB_EFFECTIVE_ALLOWED_CIDR_V4, LB_EFFECTIVE_ALLOWED_CIDR_V6
    global MASTER_IPV6_OVERRIDE, MASTER_PRIVATE_IP_OVERRIDE
    global PROTOCOL_PROFILE, H2_CLIENT_PERCENT
    global TARGET_CONCURRENCY, AVG_SESSION_SEC, HEARTBEAT_INTERVAL_SEC, UPLOAD_BYTES
    global SESSION_TTL_JITTER_ENABLED, SESSION_TTL_JITTER_PCT
    global STEADY_STATE_TOLERANCE_PCT, STEADY_STATE_EVAL_WINDOW_SEC
    global RAMP_UP_SEC, RUN_HOLD_SEC
    global HEALTH_ENDPOINT_PATH, UPLOAD_ENDPOINT_PATH
    global UPLOAD_ACK_ENABLED, UPLOAD_ACK_STATUS, UPLOAD_ACK_BODY
    global UI_ENABLE, UI_EXPOSE_MODE, UI_WEB_HOST, UI_WEB_PORT, UI_ALLOWED_CIDR
    global HEADLESS_ENABLE, EXPECT_WORKERS_STRICT, EXPECTED_WORKERS_OVERRIDE, GRACE_SEC
    global WORKERS_PER_HOST, CPU_RESERVE, MIN_WORKERS_PER_HOST, MAX_WORKERS_PER_HOST
    global LOCUST_VERIFY_TLS, LOCUST_CONNECT_TIMEOUT_MS, LOCUST_READ_TIMEOUT_MS
    global LOCUST_CONNECT_TIMEOUT, LOCUST_READ_TIMEOUT
    global OUTPUT_DIR, RUN_ID
    global TARGET_OPENS_PER_SEC, TARGET_CLOSES_PER_SEC, TARGET_OPENS_PER_MIN, TARGET_CLOSES_PER_MIN
    issues = _validate_inputs()
    if issues:
        issues_html.value = "<div class='issues-strip'><b>Validation failed:</b><br>" + "<br>".join(
            [f"{i+1}. {_html_escape(x)}" for i, x in enumerate(issues)]
        ) + "</div>"
        return
    issues_html.value = ""
    try:
        ssh_pub, ssh_priv = _resolve_ssh_paths()
        cert_p, key_p, ca_p = _resolve_cert_paths()
        with open(ssh_pub, "r", encoding="utf-8") as f:
            SSH_PUBLIC_KEY_CONTENT = f.read().strip()
        if not SSH_PUBLIC_KEY_CONTENT:
            raise ValueError("SSH public key file is empty.")
        # Discovery outputs
        SELECTED_REGION = region_dd.value
        REGION = region_dd.value
        COMPARTMENT_ID = comp_dd.value
        AD_A = ad_dd.value
        AD_B = AD_A
        IMAGE_ID = image_dd.value
        # SSH
        SSH_PUBLIC_KEY_PATH = ssh_pub
        SSH_PRIVATE_KEY_PATH = ssh_priv
        # Cert
        LB_CERT_MODE = cert_mode_dd.value
        LB_CERT_PEM_PATH = cert_p
        LB_KEY_PEM_PATH = key_p
        LB_CA_PEM_PATH = ca_p
        LB_CERT_KIND = cert_kind_dd.value
        LB_ECDSA_CURVE = ecdsa_curve_dd.value
        LB_RSA_BITS = int(rsa_bits_dd.value)
        LB_CERT_CN = (cert_cn_in.value or "lb.local").strip()
        LB_CERT_DAYS = int(cert_days_in.value)
        LB_PEM_OUTPUT_DIR = _expand((pem_output_dir_in.value or "./local-lb-pems").strip())
        LB_PEM_BASENAME = (pem_basename_in.value or "lb_current").strip()
        # Infra
        LB_TOPOLOGY = lb_topology_dd.value
        LB_COUNT = 1 if LB_TOPOLOGY == "single" else int(lb_count_in.value)
        BACKEND_COUNT = int(backend_count_in.value)
        GENERATOR_COUNT = int(generator_count_in.value)
        BACKEND_SHAPE = backend_shape_dd.value
        GENERATOR_SHAPE = generator_shape_dd.value
        if _is_flex(BACKEND_SHAPE):
            BACKEND_OCPUS = float(backend_ocpus_in.value)
            BACKEND_MEMORY_GB = float(backend_mem_in.value)
        if _is_flex(GENERATOR_SHAPE):
            GENERATOR_OCPUS = float(generator_ocpus_in.value)
            GENERATOR_MEMORY_GB = float(generator_mem_in.value)
        LB_MIN_MBPS = int(lb_min_mbps_in.value)
        LB_MAX_MBPS = int(lb_max_mbps_in.value)
        ENABLE_IPV6_FRONTEND = bool(enable_ipv6_frontend_cb.value)
        USE_SSH_IPV6 = bool(use_ssh_ipv6_cb.value)
        SSH_ALLOWED_CIDR = (ssh_allowed_cidr_in.value or "0.0.0.0/0").strip()
        SSH_ALLOWED_V6_CIDR = (ssh_allowed_v6_cidr_in.value or "::/0").strip()
        LB_VISIBILITY = lb_visibility_dd.value
        LB_ALLOWED_CIDR_V4 = _validate_cidr((lb_allowed_cidr_v4_in.value or "0.0.0.0/0").strip(), 4)
        LB_ALLOWED_CIDR_V6 = _validate_cidr((lb_allowed_cidr_v6_in.value or "::/0").strip(), 6)
        LB_PRIVATE_ALLOWED_CIDR_V4 = _validate_cidr_or_auto((lb_private_allowed_cidr_v4_in.value or "auto").strip(), 4)
        LB_PRIVATE_ALLOWED_CIDR_V6 = _validate_cidr_or_auto((lb_private_allowed_cidr_v6_in.value or "auto").strip(), 6)
        if LB_VISIBILITY == "public":
            LB_EFFECTIVE_ALLOWED_CIDR_V4 = LB_ALLOWED_CIDR_V4
            LB_EFFECTIVE_ALLOWED_CIDR_V6 = LB_ALLOWED_CIDR_V6 if ENABLE_IPV6_FRONTEND else ""
        else:
            LB_EFFECTIVE_ALLOWED_CIDR_V4 = LB_PRIVATE_ALLOWED_CIDR_V4
            LB_EFFECTIVE_ALLOWED_CIDR_V6 = LB_PRIVATE_ALLOWED_CIDR_V6 if ENABLE_IPV6_FRONTEND else ""
        MASTER_IPV6_OVERRIDE = (master_ipv6_override_in.value or "").strip() if show_overrides_cb.value else ""
        MASTER_PRIVATE_IP_OVERRIDE = (master_private_ip_override_in.value or "").strip() if show_overrides_cb.value else ""
        # Session profile
        PROTOCOL_PROFILE = protocol_profile_dd.value
        H2_CLIENT_PERCENT = float(h2_client_percent_in.value)
        TARGET_CONCURRENCY = int(target_concurrency_in.value)
        AVG_SESSION_SEC = int(avg_session_sec_in.value)
        HEARTBEAT_INTERVAL_SEC = int(heartbeat_interval_sec_in.value)
        UPLOAD_BYTES = int(upload_bytes_in.value)
        SESSION_TTL_JITTER_ENABLED = bool(session_ttl_jitter_enabled_cb.value)
        SESSION_TTL_JITTER_PCT = float(session_ttl_jitter_pct_in.value)
        STEADY_STATE_TOLERANCE_PCT = float(steady_state_tolerance_pct_in.value)
        STEADY_STATE_EVAL_WINDOW_SEC = int(steady_state_eval_window_sec_in.value)
        RAMP_UP_SEC = int(ramp_up_sec_in.value)
        RUN_HOLD_SEC = int(run_hold_sec_in.value)
        HEALTH_ENDPOINT_PATH = (health_endpoint_in.value or "/healthz").strip()
        UPLOAD_ENDPOINT_PATH = (upload_endpoint_in.value or "/upload_5k").strip()
        UPLOAD_ACK_ENABLED = bool(upload_ack_enabled_cb.value)
        UPLOAD_ACK_STATUS = int(upload_ack_status_in.value)
        UPLOAD_ACK_BODY = (upload_ack_body_in.value or "ack")
        # Runtime
        UI_ENABLE = bool(ui_enable_cb.value)
        UI_EXPOSE_MODE = ui_expose_mode_dd.value
        UI_WEB_HOST = _first_nonempty(_d("UI_WEB_HOST", "0.0.0.0"), "0.0.0.0")
        UI_WEB_PORT = int(ui_web_port_in.value)
        UI_ALLOWED_CIDR = (ui_allowed_cidr_in.value or "0.0.0.0/0").strip()
        HEADLESS_ENABLE = bool(headless_enable_cb.value)
        EXPECT_WORKERS_STRICT = bool(expect_workers_strict_cb.value)
        GRACE_SEC = int(grace_sec_in.value)
        WORKERS_PER_HOST = workers_per_host_dd.value
        CPU_RESERVE = int(cpu_reserve_in.value)
        if WORKERS_PER_HOST == "fixed":
            fixed_n = int(fixed_workers_in.value)
            MIN_WORKERS_PER_HOST = fixed_n
            MAX_WORKERS_PER_HOST = fixed_n
            EXPECTED_WORKERS_OVERRIDE = fixed_n
        else:
            MIN_WORKERS_PER_HOST = int(min_workers_in.value)
            MAX_WORKERS_PER_HOST = int(max_workers_in.value)
            EXPECTED_WORKERS_OVERRIDE = _g("EXPECTED_WORKERS_OVERRIDE", None)
        LOCUST_VERIFY_TLS = bool(locust_verify_tls_cb.value)
        LOCUST_CONNECT_TIMEOUT_MS = int(locust_connect_timeout_ms_in.value)
        LOCUST_READ_TIMEOUT_MS = int(locust_read_timeout_ms_in.value)
        LOCUST_CONNECT_TIMEOUT = LOCUST_CONNECT_TIMEOUT_MS
        LOCUST_READ_TIMEOUT = LOCUST_READ_TIMEOUT_MS
        OUTPUT_DIR = os.path.abspath((output_dir_in.value or "./results").strip())
        RUN_ID = (run_id_in.value or "").strip()
        Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        TARGET_OPENS_PER_SEC = TARGET_CONCURRENCY / AVG_SESSION_SEC
        TARGET_CLOSES_PER_SEC = TARGET_OPENS_PER_SEC
        TARGET_OPENS_PER_MIN = TARGET_OPENS_PER_SEC * 60.0
        TARGET_CLOSES_PER_MIN = TARGET_CLOSES_PER_SEC * 60.0
        env_updates = {
            "OCI_CONFIG_FILE": OCI_CONFIG_FILE,
            "OCI_PROFILE": OCI_PROFILE,
            "REGION": REGION,
            "SELECTED_REGION": SELECTED_REGION,
            "COMPARTMENT_ID": COMPARTMENT_ID,
            "AD_A": AD_A,
            "AD_B": AD_B,
            "IMAGE_ID": IMAGE_ID,
            "SSH_PUBLIC_KEY_PATH": SSH_PUBLIC_KEY_PATH,
            "SSH_PRIVATE_KEY_PATH": SSH_PRIVATE_KEY_PATH,
            "LB_CERT_MODE": LB_CERT_MODE,
            "LB_CERT_PEM_PATH": LB_CERT_PEM_PATH,
            "LB_KEY_PEM_PATH": LB_KEY_PEM_PATH,
            "LB_CA_PEM_PATH": LB_CA_PEM_PATH,
            "LB_CERT_KIND": LB_CERT_KIND,
            "LB_ECDSA_CURVE": LB_ECDSA_CURVE,
            "LB_RSA_BITS": LB_RSA_BITS,
            "LB_CERT_CN": LB_CERT_CN,
            "LB_CERT_DAYS": LB_CERT_DAYS,
            "LB_PEM_OUTPUT_DIR": LB_PEM_OUTPUT_DIR,
            "LB_PEM_BASENAME": LB_PEM_BASENAME,
            "LB_TOPOLOGY": LB_TOPOLOGY,
            "LB_COUNT": LB_COUNT,
            "BACKEND_COUNT": BACKEND_COUNT,
            "GENERATOR_COUNT": GENERATOR_COUNT,
            "BACKEND_SHAPE": BACKEND_SHAPE,
            "BACKEND_OCPUS": _g("BACKEND_OCPUS", _to_float(_d("BACKEND_OCPUS", 4), 4.0)),
            "BACKEND_MEMORY_GB": _g("BACKEND_MEMORY_GB", _to_float(_d("BACKEND_MEMORY_GB", 64), 64.0)),
            "GENERATOR_SHAPE": GENERATOR_SHAPE,
            "GENERATOR_OCPUS": _g("GENERATOR_OCPUS", _to_float(_d("GENERATOR_OCPUS", 8), 8.0)),
            "GENERATOR_MEMORY_GB": _g("GENERATOR_MEMORY_GB", _to_float(_d("GENERATOR_MEMORY_GB", 128), 128.0)),
            "LB_MIN_MBPS": LB_MIN_MBPS,
            "LB_MAX_MBPS": LB_MAX_MBPS,
            "ENABLE_IPV6_FRONTEND": ENABLE_IPV6_FRONTEND,
            "USE_SSH_IPV6": USE_SSH_IPV6,
            "SSH_ALLOWED_CIDR": SSH_ALLOWED_CIDR,
            "SSH_ALLOWED_V6_CIDR": SSH_ALLOWED_V6_CIDR,
            "LB_VISIBILITY": LB_VISIBILITY,
            "LB_ALLOWED_CIDR_V4": LB_ALLOWED_CIDR_V4,
            "LB_ALLOWED_CIDR_V6": LB_ALLOWED_CIDR_V6,
            "LB_PRIVATE_ALLOWED_CIDR_V4": LB_PRIVATE_ALLOWED_CIDR_V4,
            "LB_PRIVATE_ALLOWED_CIDR_V6": LB_PRIVATE_ALLOWED_CIDR_V6,
            "LB_EFFECTIVE_ALLOWED_CIDR_V4": LB_EFFECTIVE_ALLOWED_CIDR_V4,
            "LB_EFFECTIVE_ALLOWED_CIDR_V6": LB_EFFECTIVE_ALLOWED_CIDR_V6,
            "MASTER_IPV6_OVERRIDE": MASTER_IPV6_OVERRIDE,
            "MASTER_PRIVATE_IP_OVERRIDE": MASTER_PRIVATE_IP_OVERRIDE,
            "PROTOCOL_PROFILE": PROTOCOL_PROFILE,
            "H2_CLIENT_PERCENT": H2_CLIENT_PERCENT,
            "TARGET_CONCURRENCY": TARGET_CONCURRENCY,
            "AVG_SESSION_SEC": AVG_SESSION_SEC,
            "HEARTBEAT_INTERVAL_SEC": HEARTBEAT_INTERVAL_SEC,
            "UPLOAD_BYTES": UPLOAD_BYTES,
            "SESSION_TTL_JITTER_ENABLED": SESSION_TTL_JITTER_ENABLED,
            "SESSION_TTL_JITTER_PCT": SESSION_TTL_JITTER_PCT,
            "STEADY_STATE_TOLERANCE_PCT": STEADY_STATE_TOLERANCE_PCT,
            "STEADY_STATE_EVAL_WINDOW_SEC": STEADY_STATE_EVAL_WINDOW_SEC,
            "RAMP_UP_SEC": RAMP_UP_SEC,
            "RUN_HOLD_SEC": RUN_HOLD_SEC,
            "HEALTH_ENDPOINT_PATH": HEALTH_ENDPOINT_PATH,
            "UPLOAD_ENDPOINT_PATH": UPLOAD_ENDPOINT_PATH,
            "UPLOAD_ACK_ENABLED": UPLOAD_ACK_ENABLED,
            "UPLOAD_ACK_STATUS": UPLOAD_ACK_STATUS,
            "UPLOAD_ACK_BODY": UPLOAD_ACK_BODY,
            "UI_ENABLE": UI_ENABLE,
            "UI_EXPOSE_MODE": UI_EXPOSE_MODE,
            "UI_WEB_HOST": UI_WEB_HOST,
            "UI_WEB_PORT": UI_WEB_PORT,
            "UI_ALLOWED_CIDR": UI_ALLOWED_CIDR,
            "HEADLESS_ENABLE": HEADLESS_ENABLE,
            "EXPECT_WORKERS_STRICT": EXPECT_WORKERS_STRICT,
            "GRACE_SEC": GRACE_SEC,
            "WORKERS_PER_HOST": WORKERS_PER_HOST,
            "CPU_RESERVE": CPU_RESERVE,
            "MIN_WORKERS_PER_HOST": MIN_WORKERS_PER_HOST,
            "MAX_WORKERS_PER_HOST": MAX_WORKERS_PER_HOST,
            "LOCUST_VERIFY_TLS": LOCUST_VERIFY_TLS,
            "LOCUST_CONNECT_TIMEOUT_MS": LOCUST_CONNECT_TIMEOUT_MS,
            "LOCUST_READ_TIMEOUT_MS": LOCUST_READ_TIMEOUT_MS,
            "LOCUST_CONNECT_TIMEOUT": LOCUST_CONNECT_TIMEOUT,
            "LOCUST_READ_TIMEOUT": LOCUST_READ_TIMEOUT,
            "OUTPUT_DIR": OUTPUT_DIR,
            "RUN_ID": RUN_ID,
            "TARGET_OPENS_PER_SEC": TARGET_OPENS_PER_SEC,
            "TARGET_CLOSES_PER_SEC": TARGET_CLOSES_PER_SEC,
            "TARGET_OPENS_PER_MIN": TARGET_OPENS_PER_MIN,
            "TARGET_CLOSES_PER_MIN": TARGET_CLOSES_PER_MIN,
        }
        for k, v in env_updates.items():
            _setenv(k, v)
        summary = {
            "discovery": {
                "REGION": REGION,
                "COMPARTMENT_ID": COMPARTMENT_ID,
                "AD_A": AD_A,
                "IMAGE_ID": IMAGE_ID,
            },
            "tls": {
                "LB_CERT_MODE": LB_CERT_MODE,
                "LB_CERT_PEM_PATH": LB_CERT_PEM_PATH,
                "LB_KEY_PEM_PATH": LB_KEY_PEM_PATH,
                "LB_CA_PEM_PATH": LB_CA_PEM_PATH,
            },
            "ssh": {
                "SSH_PUBLIC_KEY_PATH": SSH_PUBLIC_KEY_PATH,
                "SSH_PRIVATE_KEY_PATH": SSH_PRIVATE_KEY_PATH,
            },
            "network": {
                "LB_VISIBILITY": LB_VISIBILITY,
                "LB_ALLOWED_CIDR_V4_PUBLIC": LB_ALLOWED_CIDR_V4,
                "LB_ALLOWED_CIDR_V6_PUBLIC": LB_ALLOWED_CIDR_V6,
                "LB_PRIVATE_ALLOWED_CIDR_V4": LB_PRIVATE_ALLOWED_CIDR_V4,
                "LB_PRIVATE_ALLOWED_CIDR_V6": LB_PRIVATE_ALLOWED_CIDR_V6,
                "LB_EFFECTIVE_ALLOWED_CIDR_V4": LB_EFFECTIVE_ALLOWED_CIDR_V4,
                "LB_EFFECTIVE_ALLOWED_CIDR_V6": LB_EFFECTIVE_ALLOWED_CIDR_V6,
            },
            "session": {
                "PROTOCOL_PROFILE": PROTOCOL_PROFILE,
                "H2_CLIENT_PERCENT": H2_CLIENT_PERCENT,
                "TARGET_CONCURRENCY": TARGET_CONCURRENCY,
                "AVG_SESSION_SEC": AVG_SESSION_SEC,
                "TARGET_OPENS_PER_SEC": round(TARGET_OPENS_PER_SEC, 3),
                "UPLOAD_ACK_ENABLED": UPLOAD_ACK_ENABLED,
                "UPLOAD_ACK_STATUS": UPLOAD_ACK_STATUS,
            },
            "artifacts": {
                "OUTPUT_DIR": OUTPUT_DIR,
                "RUN_ID": RUN_ID,
            },
        }
        with apply_out:
            apply_out.clear_output()
            print("Cell 3 complete: selections applied.")
            print(json.dumps(summary, indent=2))
        status_html.value = "<div class='ok-strip'><b>Apply complete.</b> Cell 4 can now run.</div>"
    except Exception as e:
        issues_html.value = f"<div class='issues-strip'><b>Apply failed:</b> {_html_escape(repr(e))}</div>"
def _print_effective(_):
    snap = {
        "REGION": _g("REGION"),
        "COMPARTMENT_ID": _g("COMPARTMENT_ID"),
        "AD_A": _g("AD_A"),
        "IMAGE_ID": _g("IMAGE_ID"),
        "LB_TOPOLOGY": _g("LB_TOPOLOGY"),
        "LB_COUNT": _g("LB_COUNT"),
        "BACKEND_COUNT": _g("BACKEND_COUNT"),
        "GENERATOR_COUNT": _g("GENERATOR_COUNT"),
        "LB_CERT_MODE": _g("LB_CERT_MODE"),
        "LB_CERT_PEM_PATH": _g("LB_CERT_PEM_PATH"),
        "LB_KEY_PEM_PATH": _g("LB_KEY_PEM_PATH"),
        "SSH_PUBLIC_KEY_PATH": _g("SSH_PUBLIC_KEY_PATH"),
        "SSH_PRIVATE_KEY_PATH": _g("SSH_PRIVATE_KEY_PATH"),
        "PROTOCOL_PROFILE": _g("PROTOCOL_PROFILE"),
        "LB_VISIBILITY": _g("LB_VISIBILITY"),
        "LB_ALLOWED_CIDR_V4": _g("LB_ALLOWED_CIDR_V4"),
        "LB_ALLOWED_CIDR_V6": _g("LB_ALLOWED_CIDR_V6"),
        "LB_PRIVATE_ALLOWED_CIDR_V4": _g("LB_PRIVATE_ALLOWED_CIDR_V4"),
        "LB_PRIVATE_ALLOWED_CIDR_V6": _g("LB_PRIVATE_ALLOWED_CIDR_V6"),
        "LB_EFFECTIVE_ALLOWED_CIDR_V4": _g("LB_EFFECTIVE_ALLOWED_CIDR_V4"),
        "LB_EFFECTIVE_ALLOWED_CIDR_V6": _g("LB_EFFECTIVE_ALLOWED_CIDR_V6"),
        "UPLOAD_ACK_ENABLED": _g("UPLOAD_ACK_ENABLED"),
        "UPLOAD_ACK_STATUS": _g("UPLOAD_ACK_STATUS"),
        "TARGET_CONCURRENCY": _g("TARGET_CONCURRENCY"),
        "AVG_SESSION_SEC": _g("AVG_SESSION_SEC"),
        "TARGET_OPENS_PER_SEC": round(_to_float(_g("TARGET_OPENS_PER_SEC", 0.0), 0.0), 3),
        "OUTPUT_DIR": _g("OUTPUT_DIR"),
        "RUN_ID": _g("RUN_ID"),
    }
    with apply_out:
        apply_out.clear_output()
        print("Effective configuration snapshot:")
        print(json.dumps(snap, indent=2))
def _reset_from_globals(_):
    # Keep reset simple: restore widget values from current globals/env defaults and refresh discovery.
    region_dd.value = (_d("REGION", region_names[0]) if _d("REGION", region_names[0]) in region_names else region_names[0])
    comp_val = _d("COMPARTMENT_ID", TENANCY_OCID)
    if comp_val in [v for _, v in comp_options]:
        comp_dd.value = comp_val
    lb_topology_dd.value = (_d("LB_TOPOLOGY", "single") if _d("LB_TOPOLOGY", "single") in {"single", "multi"} else "single")
    lb_count_in.value = _to_int(_d("LB_COUNT", 1), 1)
    backend_count_in.value = _to_int(_d("BACKEND_COUNT", 120), 120)
    generator_count_in.value = _to_int(_d("GENERATOR_COUNT", 1), 1)
    lb_min_mbps_in.value = _to_int(_d("LB_MIN_MBPS", 100), 100)
    lb_max_mbps_in.value = _to_int(_d("LB_MAX_MBPS", 8000), 8000)
    enable_ipv6_frontend_cb.value = _to_bool(_d("ENABLE_IPV6_FRONTEND", True), True)
    use_ssh_ipv6_cb.value = _to_bool(_d("USE_SSH_IPV6", False), False)
    ssh_allowed_cidr_in.value = _d("SSH_ALLOWED_CIDR", "0.0.0.0/0")
    ssh_allowed_v6_cidr_in.value = _d("SSH_ALLOWED_V6_CIDR", "::/0")
    lb_visibility_dd.value = (_d("LB_VISIBILITY", "public") if _d("LB_VISIBILITY", "public") in {"public", "private"} else "public")
    lb_allowed_cidr_v4_in.value = _d("LB_ALLOWED_CIDR_V4", "0.0.0.0/0")
    lb_allowed_cidr_v6_in.value = _d("LB_ALLOWED_CIDR_V6", "::/0")
    lb_private_allowed_cidr_v4_in.value = _d("LB_PRIVATE_ALLOWED_CIDR_V4", "auto")
    lb_private_allowed_cidr_v6_in.value = _d("LB_PRIVATE_ALLOWED_CIDR_V6", "auto")
    show_overrides_cb.value = bool(_first_nonempty(_d("MASTER_IPV6_OVERRIDE", ""), _d("MASTER_PRIVATE_IP_OVERRIDE", "")))
    master_ipv6_override_in.value = _d("MASTER_IPV6_OVERRIDE", "")
    master_private_ip_override_in.value = _d("MASTER_PRIVATE_IP_OVERRIDE", "")
    protocol_profile_dd.value = (_d("PROTOCOL_PROFILE", "http2_preferred") if _d("PROTOCOL_PROFILE", "http2_preferred") in {"http2_preferred", "https_h1_compatible", "tcp_ppv2"} else "http2_preferred")
    h2_client_percent_in.value = _to_float(_d("H2_CLIENT_PERCENT", 100), 100.0)
    target_concurrency_in.value = _to_int(_d("TARGET_CONCURRENCY", 5000000), 5000000)
    avg_session_sec_in.value = _to_int(_d("AVG_SESSION_SEC", 1200), 1200)
    heartbeat_interval_sec_in.value = _to_int(_d("HEARTBEAT_INTERVAL_SEC", 50), 50)
    upload_bytes_in.value = _to_int(_d("UPLOAD_BYTES", 5120), 5120)
    session_ttl_jitter_enabled_cb.value = _to_bool(_d("SESSION_TTL_JITTER_ENABLED", False), False)
    session_ttl_jitter_pct_in.value = _to_float(_d("SESSION_TTL_JITTER_PCT", 10), 10.0)
    steady_state_tolerance_pct_in.value = _to_float(_d("STEADY_STATE_TOLERANCE_PCT", 10), 10.0)
    steady_state_eval_window_sec_in.value = _to_int(_d("STEADY_STATE_EVAL_WINDOW_SEC", 600), 600)
    ramp_up_sec_in.value = _to_int(_d("RAMP_UP_SEC", 1200), 1200)
    run_hold_sec_in.value = _to_int(_d("RUN_HOLD_SEC", 3600), 3600)
    health_endpoint_in.value = _d("HEALTH_ENDPOINT_PATH", "/healthz")
    upload_endpoint_in.value = _d("UPLOAD_ENDPOINT_PATH", "/upload_5k")
    upload_ack_enabled_cb.value = _to_bool(_d("UPLOAD_ACK_ENABLED", True), True)
    upload_ack_status_in.value = _to_int(_d("UPLOAD_ACK_STATUS", 200), 200)
    upload_ack_body_in.value = _d("UPLOAD_ACK_BODY", "ack")
    ui_enable_cb.value = _to_bool(_d("UI_ENABLE", True), True)
    ui_expose_mode_dd.value = (_d("UI_EXPOSE_MODE", "tunnel") if _d("UI_EXPOSE_MODE", "tunnel") in {"tunnel", "nsg"} else "tunnel")
    ui_web_port_in.value = _to_int(_d("UI_WEB_PORT", 8089), 8089)
    ui_allowed_cidr_in.value = _d("UI_ALLOWED_CIDR", "0.0.0.0/0")
    headless_enable_cb.value = _to_bool(_d("HEADLESS_ENABLE", True), True)
    expect_workers_strict_cb.value = _to_bool(_d("EXPECT_WORKERS_STRICT", False), False)
    grace_sec_in.value = _to_int(_d("GRACE_SEC", 300), 300)
    workers_per_host_dd.value = (_d("WORKERS_PER_HOST", "auto") if _d("WORKERS_PER_HOST", "auto") in {"auto", "fixed"} else "auto")
    fixed_workers_in.value = _to_int(_d("MIN_WORKERS_PER_HOST", 1), 1)
    cpu_reserve_in.value = _to_int(_d("CPU_RESERVE", 1), 1)
    min_workers_in.value = _to_int(_d("MIN_WORKERS_PER_HOST", 1), 1)
    max_workers_in.value = _to_int(_d("MAX_WORKERS_PER_HOST", 8), 8)
    locust_verify_tls_cb.value = _to_bool(_d("LOCUST_VERIFY_TLS", True), True)
    locust_connect_timeout_ms_in.value = _to_int(_d("LOCUST_CONNECT_TIMEOUT_MS", 10000), 10000)
    locust_read_timeout_ms_in.value = _to_int(_d("LOCUST_READ_TIMEOUT_MS", 10000), 10000)
    cert_mode_dd.value = (_d("LB_CERT_MODE", "browse") if _d("LB_CERT_MODE", "browse") in {"browse", "generate"} else "browse")
    lb_cert_path_in.value = _d("LB_CERT_PEM_PATH", "")
    lb_key_path_in.value = _d("LB_KEY_PEM_PATH", "")
    lb_ca_path_in.value = _d("LB_CA_PEM_PATH", "")
    cert_kind_dd.value = (_d("LB_CERT_KIND", "ecdsa") if _d("LB_CERT_KIND", "ecdsa") in {"ecdsa", "rsa"} else "ecdsa")
    rsa_bits_dd.value = (_to_int(_d("LB_RSA_BITS", 2048), 2048) if _to_int(_d("LB_RSA_BITS", 2048), 2048) in {2048, 3072, 4096} else 2048)
    cert_cn_in.value = _d("LB_CERT_CN", "lb.local")
    cert_days_in.value = _to_int(_d("LB_CERT_DAYS", 365), 365)
    pem_output_dir_in.value = _d("LB_PEM_OUTPUT_DIR", "./local-lb-pems")
    pem_basename_in.value = _d("LB_PEM_BASENAME", "lb_current")
    use_ssh_override_cb.value = False
    ssh_pub_override_in.value = _d("SSH_PUBLIC_KEY_PATH", "")
    ssh_priv_override_in.value = _d("SSH_PRIVATE_KEY_PATH", "")
    output_dir_in.value = _d("OUTPUT_DIR", "./results")
    run_id_in.value = _d("RUN_ID", "flb_concurrency_manual")
    _sync_topology_state()
    _sync_lb_visibility_state()
    _sync_jitter_state()
    _sync_override_state()
    _sync_worker_state()
    _sync_cert_mode_state()
    _sync_ssh_override_state()
    _refresh_region_data()
# -----------------------------------------------------------------------------
# Sections
# -----------------------------------------------------------------------------
guide_html = widgets.HTML(
    """
<div class='hint-strip'>
<b>Cell 3 flow:</b> 1) Discovery (region, compartment, AD, shape, image)
2) Cert mode (browse/generate) 3) SSH keys 4) Runtime 5) Preview paths 6) Apply.
</div>
"""
)
discovery_section = _section(
    "1) OCI Discovery (region / compartment / AD / shape / image)",
    widgets.HTML("<div class='hint-strip'><b>Update each run:</b> Region, compartment, AD, shapes, and Oracle Linux image.</div>"),
    widgets.HTML("<div class='hint-strip'><b>Tip:</b> Reload Discovery after region/compartment changes. Refresh Images after shape changes.</div>"),
    _row("Region", region_dd),
    _row("Compartment", comp_dd),
    _row("Availability Domain", ad_dd),
    _row("Shape filter", shape_filter_in),
    _row("Backend shape", backend_shape_dd),
    _row("Generator shape", generator_shape_dd),
    _row("Backend Flex OCPUs", backend_ocpus_in),
    _row("Backend Flex memory GB", backend_mem_in),
    _row("Generator Flex OCPUs", generator_ocpus_in),
    _row("Generator Flex memory GB", generator_mem_in),
    _row("Oracle Linux image", image_dd),
    _row("Discovery actions", widgets.HBox([reload_discovery_btn, refresh_images_btn], layout=widgets.Layout(gap="8px"))),
)
browse_cert_box = _section(
    "Browse Existing PEM Files",
    _row("Show hidden files", show_hidden_cb),
    _row("Jump picker folder", widgets.HBox([jump_path_in, jump_btn], layout=widgets.Layout(gap="8px"))),
    widgets.HTML("<b>LB certificate picker</b>"),
    fc_cert,
    widgets.HTML("<b>LB private key picker</b>"),
    fc_key,
    widgets.HTML("<b>LB CA chain picker (optional)</b>"),
    fc_ca,
    _row("LB cert path override", lb_cert_path_in),
    _row("LB key path override", lb_key_path_in),
    _row("LB CA path override", lb_ca_path_in),
)
generate_cert_box = _section(
    "Generate Self-Signed PEM Files",
    _row("Generated cert kind", cert_kind_dd),
    _row("ECDSA curve", ecdsa_curve_dd),
    _row("RSA bits", rsa_bits_dd),
    _row("Certificate CN/SAN", cert_cn_in),
    _row("Certificate validity days", cert_days_in),
    _row("Generated PEM output dir", pem_output_dir_in),
    _row("Generated PEM basename", pem_basename_in),
    _row("Overwrite existing generated files", overwrite_generate_cb),
)
cert_section = _section(
    "2) LB Certificate Mode",
    widgets.HTML("<div class='hint-strip'><b>Choose one:</b> Browse existing PEMs or generate self-signed PEMs.</div>"),
    _row("Certificate mode", cert_mode_dd),
    browse_cert_box,
    generate_cert_box,
)
ssh_section = _section(
    "3) SSH Key Selection",
    widgets.HTML("<div class='hint-strip'><b>Required:</b> Select both SSH public and private key files for this run.</div>"),
    widgets.HTML("<b>SSH public key picker (required)</b>"),
    fc_ssh_pub,
    widgets.HTML("<b>SSH private key picker (required)</b>"),
    fc_ssh_priv,
    _row("Use manual SSH path overrides", use_ssh_override_cb),
    _row("SSH public override", ssh_pub_override_in),
    _row("SSH private override", ssh_priv_override_in),
)
row_lb_allowed_cidr_v4 = _row("LB allowed IPv4 CIDR (public)", lb_allowed_cidr_v4_in)
row_lb_allowed_cidr_v6 = _row("LB allowed IPv6 CIDR (public)", lb_allowed_cidr_v6_in)
row_lb_private_allowed_cidr_v4 = _row("LB allowed IPv4 CIDR (private)", lb_private_allowed_cidr_v4_in)
row_lb_private_allowed_cidr_v6 = _row("LB allowed IPv6 CIDR (private)", lb_private_allowed_cidr_v6_in)
infra_section = _section(
    "4) Topology and Networking",
    widgets.HTML("<div class='hint-strip'><b>Set scale and network controls</b> for topology, counts, LB Mbps, and SSH CIDRs.</div>"),
    _row("LB topology", lb_topology_dd),
    _row("LB count", lb_count_in),
    _row("Backend instance count", backend_count_in),
    _row("Generator instance count", generator_count_in),
    _row("LB min Mbps", lb_min_mbps_in),
    _row("LB max Mbps", lb_max_mbps_in),
    _row("Enable IPv6 frontend", enable_ipv6_frontend_cb),
    _row("Use IPv6 for SSH", use_ssh_ipv6_cb),
    _row("LB visibility", lb_visibility_dd),
    row_lb_allowed_cidr_v4,
    row_lb_allowed_cidr_v6,
    lb_private_cidr_hint,
    row_lb_private_allowed_cidr_v4,
    row_lb_private_allowed_cidr_v6,
    _row("SSH allowed IPv4 CIDR", ssh_allowed_cidr_in),
    _row("SSH allowed IPv6 CIDR", ssh_allowed_v6_cidr_in),
    _row("Show control-plane overrides", show_overrides_cb),
    _row("MASTER_IPV6_OVERRIDE", master_ipv6_override_in),
    _row("MASTER_PRIVATE_IP_OVERRIDE", master_private_ip_override_in),
)
session_section = _section(
    "5) Session-Concurrency Profile",
    widgets.HTML("<div class='hint-strip'><b>H2-capable client percent:</b> Percent of clients that negotiate HTTP/2; the rest use HTTP/1.1 fallback.</div>"),
    widgets.HTML("<div class='hint-strip'><b>Enable TTL jitter / TTL jitter percent:</b> Randomize session end times to avoid synchronized drops.</div>"),
    widgets.HTML("<div class='hint-strip'><b>Steady-state tolerance percent:</b> Allowed deviation from target. <b>Steady-state eval window seconds:</b> Stability window. <b>Ramp-up seconds:</b> Time to reach target concurrency.</div>"),
    _row("Protocol profile", protocol_profile_dd),
    _row("H2-capable client percent", h2_client_percent_in),
    _row("Target concurrent sessions", target_concurrency_in),
    _row("Average session seconds", avg_session_sec_in),
    _row("Heartbeat interval seconds", heartbeat_interval_sec_in),
    _row("Upload bytes per heartbeat", upload_bytes_in),
    _row("Enable TTL jitter", session_ttl_jitter_enabled_cb),
    _row("TTL jitter percent", session_ttl_jitter_pct_in),
    _row("Steady-state tolerance percent", steady_state_tolerance_pct_in),
    _row("Steady-state eval window seconds", steady_state_eval_window_sec_in),
    _row("Ramp-up seconds", ramp_up_sec_in),
    _row("Steady hold seconds", run_hold_sec_in),
    _row("Health endpoint path", health_endpoint_in),
    _row("Upload endpoint path", upload_endpoint_in),
    _row("Enable upload acknowledgment", upload_ack_enabled_cb),
    _row("Upload acknowledgment status", upload_ack_status_in),
    _row("Upload acknowledgment body", upload_ack_body_in),
)
runtime_section = _section(
    "6) Runtime Controls",
    widgets.HTML("<div class='hint-strip'><b>Controls:</b> UI/headless mode, worker model, and client timeout/TLS behavior.</div>"),
    _row("Enable Locust UI", ui_enable_cb),
    _row("UI expose mode", ui_expose_mode_dd),
    _row("UI web port", ui_web_port_in),
    _row("UI allowed CIDR", ui_allowed_cidr_in),
    _row("Enable headless mode", headless_enable_cb),
    _row("Strict worker-count check", expect_workers_strict_cb),
    _row("Grace seconds", grace_sec_in),
    _row("Workers per host mode", workers_per_host_dd),
    _row("Fixed workers per host", fixed_workers_in),
    _row("CPU reserve", cpu_reserve_in),
    _row("Min workers per host", min_workers_in),
    _row("Max workers per host", max_workers_in),
    _row("Verify TLS certs in clients", locust_verify_tls_cb),
    _row("Locust connect timeout ms", locust_connect_timeout_ms_in),
    _row("Locust read timeout ms", locust_read_timeout_ms_in),
)
artifact_section = _section(
    "7) Artifacts",
    widgets.HTML("<div class='hint-strip'><b>Set a unique RUN_ID</b> per run to avoid artifact collisions.</div>"),
    _row("Output directory", output_dir_in),
    _row("Run ID", run_id_in),
)
actions_section = _section(
    "8) Apply",
    widgets.HTML("<div class='hint-strip'><b>Order:</b> Preview Resolved Paths -> Apply Selections -> Print Effective Config.</div>"),
    _row("Actions", widgets.HBox([preview_btn, apply_btn, print_btn, reset_btn], layout=widgets.Layout(gap="8px"))),
    preview_out,
)
ui = widgets.VBox(
    [
        status_html,
        guide_html,
        discovery_section,
        cert_section,
        ssh_section,
        infra_section,
        session_section,
        runtime_section,
        artifact_section,
        actions_section,
        issues_html,
        apply_out,
    ],
    layout=widgets.Layout(width="100%"),
    _dom_classes=["nb3-container"],
)
# -----------------------------------------------------------------------------
# Events + initial render
# -----------------------------------------------------------------------------
reload_discovery_btn.on_click(_refresh_region_data)
refresh_images_btn.on_click(_refresh_images_only)
region_dd.observe(_refresh_region_data, names="value")
comp_dd.observe(_refresh_region_data, names="value")
shape_filter_in.observe(lambda ch: _refresh_region_data(), names="value")
backend_shape_dd.observe(lambda ch: (_refresh_flex_controls(), _refresh_images_only()), names="value")
generator_shape_dd.observe(lambda ch: (_refresh_flex_controls(), _refresh_images_only()), names="value")
cert_mode_dd.observe(lambda ch: _sync_cert_mode_state(), names="value")
cert_kind_dd.observe(lambda ch: _sync_cert_mode_state(), names="value")
show_hidden_cb.observe(lambda ch: _jump_filechoosers(jump_path_in.value), names="value")
jump_btn.on_click(lambda _: _jump_filechoosers(jump_path_in.value))
use_ssh_override_cb.observe(lambda ch: _sync_ssh_override_state(), names="value")
lb_topology_dd.observe(lambda ch: _sync_topology_state(), names="value")
lb_visibility_dd.observe(lambda ch: _sync_lb_visibility_state(), names="value")
session_ttl_jitter_enabled_cb.observe(lambda ch: _sync_jitter_state(), names="value")
show_overrides_cb.observe(lambda ch: _sync_override_state(), names="value")
workers_per_host_dd.observe(lambda ch: _sync_worker_state(), names="value")
preview_btn.on_click(_preview_resolved_paths)
apply_btn.on_click(_apply)
print_btn.on_click(_print_effective)
reset_btn.on_click(_reset_from_globals)
_sync_topology_state()
_sync_lb_visibility_state()
_sync_jitter_state()
_sync_override_state()
_sync_worker_state()
_sync_cert_mode_state()
_sync_ssh_override_state()
_jump_filechoosers(jump_path_in.value)
_refresh_region_data()
display(ui)
print("Cell 3 loaded: region/compartment/image discovery is active. Review selections and click Apply.")


In [ ]:
# Cell 4 — Objective: Cloud-init generation (backend + generator readiness)

# Generates self-contained cloud-init artifacts for backend service readiness and generator runtime prep.

import os
import re
import json
import hashlib
import textwrap
from datetime import datetime, UTC
from pathlib import Path

# -----------------------------------------------------------------------------
# Preflight
# -----------------------------------------------------------------------------
_REQUIRED_GLOBALS = [
    "REGION",
    "HEALTH_ENDPOINT_PATH",
    "UPLOAD_ENDPOINT_PATH",
    "LOCUST_WORKDIR",
    "PROTOCOL_PROFILE",
    "H2_CLIENT_PERCENT",
    "HEARTBEAT_INTERVAL_SEC",
    "UPLOAD_BYTES",
    "AVG_SESSION_SEC",
    "LOCUST_VERIFY_TLS",
    "LOCUST_CONNECT_TIMEOUT_MS",
    "LOCUST_READ_TIMEOUT_MS",
    "OUTPUT_DIR",
    "RUN_ID",
    "UPLOAD_ACK_ENABLED",
    "UPLOAD_ACK_STATUS",
    "UPLOAD_ACK_BODY",
]
_missing = [x for x in _REQUIRED_GLOBALS if x not in globals()]
if _missing:
    raise RuntimeError(
        "Cell 4 preflight failed: run Cell 2 (and Cell 3 apply if using UI selections) first. "
        f"Missing globals: {_missing}"
    )

def _validate_http_path(name, value):
    if not isinstance(value, str) or not value.startswith("/"):
        raise ValueError(f"{name} must be an absolute URL path beginning with '/'. Current: {value!r}")
    if re.fullmatch(r"/[A-Za-z0-9._/\-]*", value) is None:
        raise ValueError(
            f"{name} has unsupported characters. Allowed: letters, numbers, '_', '-', '.', '/'. Current: {value!r}"
        )
    return value

def _nginx_escape_text(value):
    return str(value).replace("\\", "\\\\").replace('"', '\\"')

HEALTH_ENDPOINT_PATH = _validate_http_path("HEALTH_ENDPOINT_PATH", HEALTH_ENDPOINT_PATH)
UPLOAD_ENDPOINT_PATH = _validate_http_path("UPLOAD_ENDPOINT_PATH", UPLOAD_ENDPOINT_PATH)

if UPLOAD_BYTES <= 0:
    raise ValueError(f"UPLOAD_BYTES must be > 0. Current: {UPLOAD_BYTES}")
if HEARTBEAT_INTERVAL_SEC <= 0:
    raise ValueError(f"HEARTBEAT_INTERVAL_SEC must be > 0. Current: {HEARTBEAT_INTERVAL_SEC}")
if AVG_SESSION_SEC <= 0:
    raise ValueError(f"AVG_SESSION_SEC must be > 0. Current: {AVG_SESSION_SEC}")

UPLOAD_ACK_ENABLED = bool(UPLOAD_ACK_ENABLED)
UPLOAD_ACK_STATUS = int(UPLOAD_ACK_STATUS)
UPLOAD_ACK_BODY = str(UPLOAD_ACK_BODY if UPLOAD_ACK_BODY is not None else "ack")
if not (200 <= UPLOAD_ACK_STATUS <= 299):
    raise ValueError(f"UPLOAD_ACK_STATUS must be 200..299. Current: {UPLOAD_ACK_STATUS}")

REGION_STR = str(REGION).strip()
REPO_DNS_HOST = f"yum.{REGION_STR}.oci.oraclecloud.com" if REGION_STR else "yum.us-ashburn-1.oci.oraclecloud.com"

# -----------------------------------------------------------------------------
# Paths (self-contained project artifacts)
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()
CLOUD_INIT_DIR = PROJECT_ROOT / "cloud-init"
CLOUD_INIT_DIR.mkdir(parents=True, exist_ok=True)

BACKEND_CLOUD_INIT_PATH = CLOUD_INIT_DIR / "backend.sh.tftpl"
GENERATOR_CLOUD_INIT_PATH = CLOUD_INIT_DIR / "generator.sh"
CLOUD_INIT_METADATA_PATH = CLOUD_INIT_DIR / "session_concurrency_cloud_init_meta.json"

# -----------------------------------------------------------------------------
# Backend cloud-init content
# -----------------------------------------------------------------------------
if PROTOCOL_PROFILE == "tcp_ppv2":
    _listen_directive = "listen 80 proxy_protocol reuseport backlog=65535;"
    _real_ip_block = textwrap.dedent(
        """
        # Trust common private ranges when PROXY protocol is enabled.
        set_real_ip_from 10.0.0.0/8;
        set_real_ip_from 172.16.0.0/12;
        set_real_ip_from 192.168.0.0/16;
        real_ip_header proxy_protocol;
        real_ip_recursive on;
        """
    ).strip()
else:
    _listen_directive = "listen 80 reuseport backlog=65535;"
    _real_ip_block = ""

if UPLOAD_ACK_ENABLED:
    _upload_return_directive = f'return {int(UPLOAD_ACK_STATUS)} "{_nginx_escape_text(UPLOAD_ACK_BODY)}\\n";'
else:
    _upload_return_directive = "return 204;"

backend_cloud_init = textwrap.dedent(
    f"""#!/bin/bash
set -euxo pipefail

if command -v firewall-cmd >/dev/null 2>&1; then
  systemctl stop firewalld || true
  systemctl disable firewalld || true
fi

DNS_TEST_HOST="{REPO_DNS_HOST}"

wait_for_dns() {{
  local i
  for i in $(seq 1 24); do
    if getent hosts "$DNS_TEST_HOST" >/dev/null 2>&1; then
      return 0
    fi
    sleep 5
  done
  return 1
}}

retry_cmd() {{
  local attempts="$1"
  shift
  local n=1
  while true; do
    if "$@"; then
      return 0
    fi
    if [ "$n" -ge "$attempts" ]; then
      return 1
    fi
    sleep $((n * 5))
    n=$((n + 1))
  done
}}

# Safe kernel tuning for long-lived and bursty connection patterns.
cat >/etc/sysctl.d/99-flb-session-concurrency.conf <<'SYSCTL'
net.core.somaxconn=262144
net.core.netdev_max_backlog=250000
net.ipv4.tcp_max_syn_backlog=262144
net.ipv4.ip_local_port_range=1024 65535
net.ipv4.tcp_fin_timeout=15
net.ipv4.tcp_tw_reuse=1
fs.file-max=1000000
SYSCTL
sysctl --system || true

wait_for_dns || true

if command -v dnf >/dev/null 2>&1; then
  retry_cmd 8 dnf -y --setopt=timeout=30 --setopt=retries=10 --setopt=skip_if_unavailable=true --disablerepo=ol9_ksplice makecache || true
  retry_cmd 8 dnf -y --setopt=timeout=30 --setopt=retries=10 --setopt=skip_if_unavailable=true --disablerepo=ol9_ksplice install nginx || true
elif command -v yum >/dev/null 2>&1; then
  retry_cmd 8 yum -y --setopt=timeout=30 --setopt=retries=10 --setopt=skip_if_unavailable=true makecache || true
  retry_cmd 8 yum -y --setopt=timeout=30 --setopt=retries=10 --setopt=skip_if_unavailable=true install nginx || true
elif command -v apt-get >/dev/null 2>&1; then
  retry_cmd 8 apt-get update -y -o Acquire::Retries=10 || true
  retry_cmd 8 apt-get install -y nginx -o Acquire::Retries=10 || true
fi

if ! command -v nginx >/dev/null 2>&1; then
  echo "ERROR: nginx installation failed after retries." >&2
  if command -v dnf >/dev/null 2>&1; then
    dnf repolist || true
  fi
  exit 1
fi

cat >/etc/nginx/nginx.conf <<'NGINX'
user nginx;
worker_processes auto;
worker_rlimit_nofile 1048576;

events {{
    worker_connections 131072;
    multi_accept on;
    accept_mutex off;
}}

http {{
    include /etc/nginx/mime.types;
    default_type application/octet-stream;

    sendfile on;
    tcp_nopush on;
    tcp_nodelay on;

    access_log off;
    server_tokens off;

    keepalive_timeout 75;
    keepalive_requests 100000;

    client_header_timeout 30;
    client_body_timeout 30;
    send_timeout 30;

    server {{
        {_listen_directive}
        server_name _;

        {_real_ip_block}

        location = {HEALTH_ENDPOINT_PATH} {{
            default_type text/plain;
            return 200 "ok\\n";
        }}

        # Accept periodic ~5KB uploads for session heartbeat traffic.
        location = {UPLOAD_ENDPOINT_PATH} {{
            limit_except POST PUT {{ deny all; }}
            client_max_body_size 64k;
            default_type text/plain;
            {_upload_return_directive}
        }}

        location / {{
            default_type text/plain;
            return 200 "ready\\n";
        }}
    }}
}}
NGINX

nginx -t
systemctl enable nginx || true
systemctl restart nginx

echo "backend_ready $(date -u +%Y-%m-%dT%H:%M:%SZ)" >/var/log/flb_backend_ready.log
"""
).strip() + "\n"

# -----------------------------------------------------------------------------
# Generator cloud-init content
# -----------------------------------------------------------------------------
_generator_template = r"""#!/bin/bash
set -euxo pipefail

if command -v firewall-cmd >/dev/null 2>&1; then
  systemctl stop firewalld || true
  systemctl disable firewalld || true
fi

DNS_TEST_HOST="__REPO_DNS_HOST__"

wait_for_dns() {
  local i
  for i in $(seq 1 24); do
    if getent hosts "$DNS_TEST_HOST" >/dev/null 2>&1; then
      return 0
    fi
    sleep 5
  done
  return 1
}

retry_cmd() {
  local attempts="$1"
  shift
  local n=1
  while true; do
    if "$@"; then
      return 0
    fi
    if [ "$n" -ge "$attempts" ]; then
      return 1
    fi
    sleep $((n * 5))
    n=$((n + 1))
  done
}

wait_for_dns || true

# Runtime + optional build deps (build deps only used if wheels are unavailable).
if command -v dnf >/dev/null 2>&1; then
  retry_cmd 8 dnf -y --setopt=timeout=30 --setopt=retries=10 --setopt=skip_if_unavailable=true --disablerepo=ol9_ksplice makecache || true
  retry_cmd 8 dnf -y --setopt=timeout=30 --setopt=retries=10 --setopt=skip_if_unavailable=true --disablerepo=ol9_ksplice \
    install python3 python3-pip python3-virtualenv python3-devel gcc gcc-c++ make libffi-devel openssl-devel tmux curl jq git || true
elif command -v yum >/dev/null 2>&1; then
  retry_cmd 8 yum -y --setopt=timeout=30 --setopt=retries=10 --setopt=skip_if_unavailable=true makecache || true
  retry_cmd 8 yum -y --setopt=timeout=30 --setopt=retries=10 --setopt=skip_if_unavailable=true \
    install python3 python3-pip python3-virtualenv python3-devel gcc gcc-c++ make libffi-devel openssl-devel tmux curl jq git || true
elif command -v apt-get >/dev/null 2>&1; then
  retry_cmd 8 apt-get update -y -o Acquire::Retries=10 || true
  retry_cmd 8 apt-get install -y python3 python3-pip python3-venv python3-dev build-essential libffi-dev libssl-dev tmux curl jq git -o Acquire::Retries=10 || true
fi

if ! command -v python3 >/dev/null 2>&1; then
  echo "ERROR: python3 is required on generator but not installed." >&2
  exit 1
fi

WORKDIR="__LOCUST_WORKDIR__"
VENV_DIR="$WORKDIR/.venv"
mkdir -p "$WORKDIR" "$WORKDIR/logs" "$WORKDIR/results"

if ! python3 -m venv "$VENV_DIR"; then
  if command -v virtualenv >/dev/null 2>&1; then
    virtualenv -p python3 "$VENV_DIR"
  else
    echo "ERROR: failed to create virtualenv via python3 -m venv and virtualenv is unavailable." >&2
    exit 1
  fi
fi

VENV_PY="$VENV_DIR/bin/python"
VENV_PIP="$VENV_DIR/bin/pip"
VENV_LOCUST="$VENV_DIR/bin/locust"

if [ ! -x "$VENV_PY" ] || [ ! -x "$VENV_PIP" ]; then
  echo "ERROR: virtualenv bootstrap incomplete at $VENV_DIR." >&2
  exit 1
fi

PY_MAJMIN="$($VENV_PY - <<'PY'
import sys
print(f"{sys.version_info.major}.{sys.version_info.minor}")
PY
)"
echo "Detected Python in venv: ${PY_MAJMIN}"

# Locust compatibility:
# - Locust >= 2.34.1 drops Python 3.9 support.
# - Oracle Linux 9 commonly ships Python 3.9.
if $VENV_PY - <<'PY'
import sys
raise SystemExit(0 if sys.version_info >= (3, 10) else 1)
PY
then
  LOCUST_SPEC="locust>=2.42,<3"
else
  LOCUST_SPEC="locust==2.33.2"
fi
echo "Using Locust spec: ${LOCUST_SPEC}"

# Per Locust install guidance for wheel/build issues, prefer binary wheels.
export PIP_DISABLE_PIP_VERSION_CHECK=1
retry_cmd 6 "$VENV_PY" -m pip install --upgrade pip setuptools wheel
retry_cmd 6 "$VENV_PY" -m pip install --prefer-binary --no-cache-dir "${LOCUST_SPEC}" "httpx[http2]>=0.28,<1"

# Validate runtime modules from venv interpreter.
"$VENV_PY" - <<'PY'
import sys
try:
    import locust
    import httpx
    import requests
except Exception as e:
    print(f"ERROR: Failed module import validation: {e}", file=sys.stderr)
    raise SystemExit(1)

print("locust_version", locust.__version__)
print("httpx_version", httpx.__version__)
print("requests_version", requests.__version__)
PY

if [ ! -x "$VENV_LOCUST" ]; then
  echo "ERROR: locust binary missing in venv: $VENV_LOCUST" >&2
  exit 1
fi

"$VENV_LOCUST" --version

"$VENV_PY" - <<'PY'
from pathlib import Path
p = Path("__LOCUST_WORKDIR__") / "payload___UPLOAD_BYTES__.bin"
size = __UPLOAD_BYTES__
if (not p.exists()) or p.stat().st_size != size:
    p.write_bytes(b"x" * size)
print(f"payload ready: {p} ({size} bytes)")
PY

cat > "$WORKDIR/session_profile.env" <<'ENV'
export LOCUST_MODE=session_concurrency
export LOCUST_VENV_DIR=__LOCUST_WORKDIR__/.venv
export LOCUST_BIN=__LOCUST_WORKDIR__/.venv/bin/locust
export LOCUST_PYTHON=__LOCUST_WORKDIR__/.venv/bin/python
export PATH=__LOCUST_WORKDIR__/.venv/bin:$PATH
export LOCUST_PROTOCOL_PROFILE=__PROTOCOL_PROFILE__
export LOCUST_H2_CLIENT_PERCENT=__H2_CLIENT_PERCENT__
export LOCUST_HEALTH_PATH=__HEALTH_ENDPOINT_PATH__
export LOCUST_UPLOAD_PATH=__UPLOAD_ENDPOINT_PATH__
export LOCUST_HEARTBEAT_INTERVAL_SEC=__HEARTBEAT_INTERVAL_SEC__
export LOCUST_UPLOAD_BYTES=__UPLOAD_BYTES__
export LOCUST_SESSION_SEC=__AVG_SESSION_SEC__
export LOCUST_VERIFY_TLS=__LOCUST_VERIFY_TLS__
export LOCUST_CONNECT_TIMEOUT_MS=__LOCUST_CONNECT_TIMEOUT_MS__
export LOCUST_READ_TIMEOUT_MS=__LOCUST_READ_TIMEOUT_MS__
ENV

cat > "$WORKDIR/protocol_smoke.py" <<'PY'
import os
import requests
import httpx

base = os.environ.get("TARGET_URL", "").strip().rstrip("/")
health_path = os.environ.get("LOCUST_HEALTH_PATH", "/healthz")
verify_tls = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
connect_s = max(float(os.environ.get("LOCUST_CONNECT_TIMEOUT_MS", "8000")) / 1000.0, 1.0)
read_s = max(float(os.environ.get("LOCUST_READ_TIMEOUT_MS", "15000")) / 1000.0, 1.0)

if not base:
    print("TARGET_URL is empty; set it before running protocol_smoke.py")
    raise SystemExit(2)

url = f"{base}{health_path}"

r1 = requests.get(url, verify=verify_tls, timeout=(connect_s, read_s))
print("H1 status:", r1.status_code)

with httpx.Client(http2=True, verify=verify_tls, timeout=httpx.Timeout(connect=connect_s, read=read_s, write=read_s, pool=read_s)) as c:
    r2 = c.get(url)
    print("H2-preferred status:", r2.status_code, "http_version:", r2.http_version)
PY

cat > "$WORKDIR/locustfile.py" <<'PY'
import os
import time
import httpx
from locust import HttpUser, User, task, constant, events

HEALTH_PATH = os.environ.get("LOCUST_HEALTH_PATH", "/healthz")
UPLOAD_PATH = os.environ.get("LOCUST_UPLOAD_PATH", "/upload_5k")
HEARTBEAT_SEC = max(float(os.environ.get("LOCUST_HEARTBEAT_INTERVAL_SEC", "50")), 0.1)
UPLOAD_BYTES = max(int(os.environ.get("LOCUST_UPLOAD_BYTES", "5120")), 1)
VERIFY_TLS = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
CONNECT_TIMEOUT_S = max(float(os.environ.get("LOCUST_CONNECT_TIMEOUT_MS", "8000")) / 1000.0, 0.1)
READ_TIMEOUT_S = max(float(os.environ.get("LOCUST_READ_TIMEOUT_MS", "15000")) / 1000.0, 0.1)
H2_PCT = min(max(float(os.environ.get("LOCUST_H2_CLIENT_PERCENT", "100")), 0.0), 100.0)

H2_WEIGHT = int(round(H2_PCT))
H1_WEIGHT = int(round(100.0 - H2_PCT))
if H1_WEIGHT == 0 and H2_WEIGHT == 0:
    H1_WEIGHT = 1

PAYLOAD = b"x" * UPLOAD_BYTES

def _full_url(base, path):
    base = (base or "").rstrip("/")
    return f"{base}{path}"

class H1SessionUser(HttpUser):
    weight = H1_WEIGHT
    wait_time = constant(HEARTBEAT_SEC)

    @task(1)
    def heartbeat_upload(self):
        self.client.post(
            UPLOAD_PATH,
            data=PAYLOAD,
            headers={"Content-Type": "application/octet-stream"},
            verify=VERIFY_TLS,
            timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
            name=f"H1 {UPLOAD_PATH}",
        )

class H2SessionUser(User):
    weight = H2_WEIGHT
    wait_time = constant(HEARTBEAT_SEC)
    abstract = False

    def on_start(self):
        self.base_url = (os.environ.get("LOCUST_DEFAULT_HOST") or os.environ.get("TARGET_URL") or "").strip().rstrip("/")
        self._timeout = httpx.Timeout(connect=CONNECT_TIMEOUT_S, read=READ_TIMEOUT_S, write=READ_TIMEOUT_S, pool=READ_TIMEOUT_S)
        self._client = httpx.Client(http2=True, verify=VERIFY_TLS, timeout=self._timeout)

    def on_stop(self):
        try:
            self._client.close()
        except Exception:
            pass

    @task(1)
    def heartbeat_upload_h2(self):
        if not self.base_url:
            return

        start = time.perf_counter()
        resp = None
        exc = None
        response_length = 0

        try:
            resp = self._client.post(
                _full_url(self.base_url, UPLOAD_PATH),
                content=PAYLOAD,
                headers={"Content-Type": "application/octet-stream"},
            )
            response_length = len(resp.content or b"")
            resp.raise_for_status()
        except Exception as e:
            exc = e

        elapsed_ms = (time.perf_counter() - start) * 1000.0
        events.request.fire(
            request_type="HTTP",
            name=f"H2 {UPLOAD_PATH}",
            response_time=elapsed_ms,
            response_length=response_length,
            response=resp,
            context={},
            exception=exc,
        )
PY

if id opc >/dev/null 2>&1; then
  chown -R opc:opc "$WORKDIR"
fi

echo "generator_ready $(date -u +%Y-%m-%dT%H:%M:%SZ)" > "$WORKDIR/generator_ready.txt"
"""

def _render_generator_script(template):
    replacements = {
        "__REPO_DNS_HOST__": REPO_DNS_HOST,
        "__LOCUST_WORKDIR__": LOCUST_WORKDIR,
        "__PROTOCOL_PROFILE__": PROTOCOL_PROFILE,
        "__H2_CLIENT_PERCENT__": str(float(H2_CLIENT_PERCENT)),
        "__HEALTH_ENDPOINT_PATH__": HEALTH_ENDPOINT_PATH,
        "__UPLOAD_ENDPOINT_PATH__": UPLOAD_ENDPOINT_PATH,
        "__HEARTBEAT_INTERVAL_SEC__": str(int(HEARTBEAT_INTERVAL_SEC)),
        "__UPLOAD_BYTES__": str(int(UPLOAD_BYTES)),
        "__AVG_SESSION_SEC__": str(int(AVG_SESSION_SEC)),
        "__LOCUST_VERIFY_TLS__": "true" if bool(LOCUST_VERIFY_TLS) else "false",
        "__LOCUST_CONNECT_TIMEOUT_MS__": str(int(LOCUST_CONNECT_TIMEOUT_MS)),
        "__LOCUST_READ_TIMEOUT_MS__": str(int(LOCUST_READ_TIMEOUT_MS)),
    }
    out = template
    for k, v in replacements.items():
        out = out.replace(k, v)
    return out

generator_cloud_init = _render_generator_script(_generator_template)

# -----------------------------------------------------------------------------
# Persist artifacts
# -----------------------------------------------------------------------------
BACKEND_CLOUD_INIT_PATH.write_text(backend_cloud_init, encoding="utf-8")
GENERATOR_CLOUD_INIT_PATH.write_text(generator_cloud_init, encoding="utf-8")

try:
    GENERATOR_CLOUD_INIT_PATH.chmod(0o755)
except Exception:
    pass

def _sha256(path_obj):
    h = hashlib.sha256()
    with open(path_obj, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

meta = {
    "RUN_ID": RUN_ID,
    "generated_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "region": REGION_STR,
    "repo_dns_host": REPO_DNS_HOST,
    "protocol_profile": PROTOCOL_PROFILE,
    "health_endpoint": HEALTH_ENDPOINT_PATH,
    "upload_endpoint": UPLOAD_ENDPOINT_PATH,
    "locust_workdir": LOCUST_WORKDIR,
    "upload_bytes": int(UPLOAD_BYTES),
    "heartbeat_interval_sec": int(HEARTBEAT_INTERVAL_SEC),
    "avg_session_sec": int(AVG_SESSION_SEC),
    "h2_client_percent": float(H2_CLIENT_PERCENT),
    "upload_ack_enabled": bool(UPLOAD_ACK_ENABLED),
    "upload_ack_status": int(UPLOAD_ACK_STATUS),
    "upload_ack_body": str(UPLOAD_ACK_BODY),
    "backend_cloud_init_path": str(BACKEND_CLOUD_INIT_PATH),
    "generator_cloud_init_path": str(GENERATOR_CLOUD_INIT_PATH),
    "backend_sha256": _sha256(BACKEND_CLOUD_INIT_PATH),
    "generator_sha256": _sha256(GENERATOR_CLOUD_INIT_PATH),
}
CLOUD_INIT_METADATA_PATH.write_text(json.dumps(meta, indent=2), encoding="utf-8")

# -----------------------------------------------------------------------------
# Export for downstream cells (terraform/tfvars)
# -----------------------------------------------------------------------------
globals()["PROJECT_ROOT"] = str(PROJECT_ROOT)
globals()["CLOUD_INIT_DIR"] = str(CLOUD_INIT_DIR)
globals()["BACKEND_CLOUD_INIT_PATH"] = str(BACKEND_CLOUD_INIT_PATH)
globals()["GENERATOR_CLOUD_INIT_PATH"] = str(GENERATOR_CLOUD_INIT_PATH)
globals()["CLOUD_INIT_METADATA_PATH"] = str(CLOUD_INIT_METADATA_PATH)

os.environ["CLOUD_INIT_DIR"] = str(CLOUD_INIT_DIR)
os.environ["BACKEND_CLOUD_INIT_PATH"] = str(BACKEND_CLOUD_INIT_PATH)
os.environ["GENERATOR_CLOUD_INIT_PATH"] = str(GENERATOR_CLOUD_INIT_PATH)
os.environ["CLOUD_INIT_METADATA_PATH"] = str(CLOUD_INIT_METADATA_PATH)

print("Cell 4 complete: cloud-init artifacts generated for backend and generator readiness.")
print(
    json.dumps(
        {
            "BACKEND_CLOUD_INIT_PATH": str(BACKEND_CLOUD_INIT_PATH),
            "GENERATOR_CLOUD_INIT_PATH": str(GENERATOR_CLOUD_INIT_PATH),
            "CLOUD_INIT_METADATA_PATH": str(CLOUD_INIT_METADATA_PATH),
            "REGION": REGION_STR,
            "REPO_DNS_HOST": REPO_DNS_HOST,
            "PROTOCOL_PROFILE": PROTOCOL_PROFILE,
            "HEALTH_ENDPOINT_PATH": HEALTH_ENDPOINT_PATH,
            "UPLOAD_ENDPOINT_PATH": UPLOAD_ENDPOINT_PATH,
            "UPLOAD_BYTES": int(UPLOAD_BYTES),
            "UPLOAD_ACK_ENABLED": bool(UPLOAD_ACK_ENABLED),
            "UPLOAD_ACK_STATUS": int(UPLOAD_ACK_STATUS),
            "UPLOAD_ACK_BODY": str(UPLOAD_ACK_BODY),
            "HEARTBEAT_INTERVAL_SEC": int(HEARTBEAT_INTERVAL_SEC),
            "LOCUST_WORKDIR": LOCUST_WORKDIR,
        },
        indent=2,
    )
)
print("NEXT: Run Cell 5 to generate Terraform using these cloud-init artifacts.")


In [ ]:
# Cell 5 — Objective: Terraform generation (visibility-aware LB + stateless NSG rules + Service Gateway for private-subnet Oracle services access)

import os
import json
import ipaddress
from pathlib import Path


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def _validate_cidr(value: str, version: int, label: str) -> str:
    try:
        net = ipaddress.ip_network(str(value).strip(), strict=False)
    except Exception as e:
        raise ValueError(f"{label} invalid CIDR: {value!r} ({e})") from e
    if net.version != version:
        raise ValueError(f"{label} must be IPv{version} CIDR, got: {value!r}")
    return str(net)


# -----------------------------------------------------------------------------
# Preflight
# -----------------------------------------------------------------------------
_REQUIRED_GLOBALS = [
    "COMPARTMENT_ID",
    "AD_A",
    "IMAGE_ID",
    "BACKEND_COUNT",
    "GENERATOR_COUNT",
    "BACKEND_SHAPE",
    "GENERATOR_SHAPE",
    "BACKEND_OCPUS",
    "BACKEND_MEMORY_GB",
    "GENERATOR_OCPUS",
    "GENERATOR_MEMORY_GB",
    "LB_COUNT",
    "LB_MIN_MBPS",
    "LB_MAX_MBPS",
    "SSH_ALLOWED_CIDR",
    "SSH_ALLOWED_V6_CIDR",
    "ENABLE_IPV6_FRONTEND",
    "UI_ENABLE",
    "UI_WEB_PORT",
    "UI_ALLOWED_CIDR",
    "PROTOCOL_PROFILE",
    "AVG_SESSION_SEC",
    "HEALTH_ENDPOINT_PATH",
    "BACKEND_CLOUD_INIT_PATH",
    "GENERATOR_CLOUD_INIT_PATH",
    "RUN_ID",
    "LB_VISIBILITY",
    "LB_ALLOWED_CIDR_V4",
    "LB_ALLOWED_CIDR_V6",
    "LB_PRIVATE_ALLOWED_CIDR_V4",
    "LB_PRIVATE_ALLOWED_CIDR_V6",
]
_missing = [x for x in _REQUIRED_GLOBALS if x not in globals()]
if _missing:
    raise RuntimeError(
        "Cell 5 preflight failed: required globals are missing. Run Cells 2-4 first. "
        f"Missing: {_missing}"
    )

if not COMPARTMENT_ID:
    raise ValueError("COMPARTMENT_ID is empty. In Cell 3, choose compartment and click Apply.")
if not AD_A:
    raise ValueError("AD_A is empty. In Cell 3, choose AD and click Apply.")
if not IMAGE_ID:
    raise ValueError("IMAGE_ID is empty. In Cell 3, choose image and click Apply.")

_backend_cloud_init_path = str(Path(BACKEND_CLOUD_INIT_PATH).expanduser().resolve())
_generator_cloud_init_path = str(Path(GENERATOR_CLOUD_INIT_PATH).expanduser().resolve())

if not os.path.exists(_backend_cloud_init_path):
    raise FileNotFoundError(f"BACKEND_CLOUD_INIT_PATH not found: {_backend_cloud_init_path}")
if not os.path.exists(_generator_cloud_init_path):
    raise FileNotFoundError(f"GENERATOR_CLOUD_INIT_PATH not found: {_generator_cloud_init_path}")


# -----------------------------------------------------------------------------
# Visibility + CIDR resolution
# -----------------------------------------------------------------------------
_lb_visibility = str(LB_VISIBILITY).strip().lower()
if _lb_visibility not in {"public", "private"}:
    raise ValueError(f"LB_VISIBILITY must be public|private, got: {LB_VISIBILITY!r}")

_lb_public_cidr_v4 = _validate_cidr(LB_ALLOWED_CIDR_V4, 4, "LB_ALLOWED_CIDR_V4")
_lb_public_cidr_v6 = _validate_cidr(LB_ALLOWED_CIDR_V6, 6, "LB_ALLOWED_CIDR_V6")

_lb_private_raw_v4 = str(LB_PRIVATE_ALLOWED_CIDR_V4).strip()
_lb_private_raw_v6 = str(LB_PRIVATE_ALLOWED_CIDR_V6).strip()

_generator_subnet_v4 = "10.0.3.0/24"

if _lb_visibility == "public":
    _lb_ingress_cidr_v4 = _lb_public_cidr_v4
    _lb_ingress_cidr_v4_display = _lb_public_cidr_v4
    if bool(ENABLE_IPV6_FRONTEND):
        _lb_ingress_cidr_v6_expr = json.dumps(_lb_public_cidr_v6)
        _lb_ingress_cidr_v6_display = _lb_public_cidr_v6
    else:
        _lb_ingress_cidr_v6_expr = '""'
        _lb_ingress_cidr_v6_display = ""
else:
    if _lb_private_raw_v4.lower() in {"", "auto"}:
        _lb_ingress_cidr_v4 = _generator_subnet_v4
        _lb_ingress_cidr_v4_display = "auto(10.0.3.0/24)"
    else:
        _lb_ingress_cidr_v4 = _validate_cidr(_lb_private_raw_v4, 4, "LB_PRIVATE_ALLOWED_CIDR_V4")
        _lb_ingress_cidr_v4_display = _lb_ingress_cidr_v4

    if bool(ENABLE_IPV6_FRONTEND):
        if _lb_private_raw_v6.lower() in {"", "auto"}:
            _lb_ingress_cidr_v6_expr = "local.gens_pub_ipv6"
            _lb_ingress_cidr_v6_display = "auto(local.gens_pub_ipv6)"
        else:
            _lb_private_v6 = _validate_cidr(_lb_private_raw_v6, 6, "LB_PRIVATE_ALLOWED_CIDR_V6")
            _lb_ingress_cidr_v6_expr = json.dumps(_lb_private_v6)
            _lb_ingress_cidr_v6_display = _lb_private_v6
    else:
        _lb_ingress_cidr_v6_expr = '""'
        _lb_ingress_cidr_v6_display = ""

_lb_is_private = _lb_visibility == "private"
_lb_subnet_id_expr = "oci_core_subnet.lb_priv.id" if _lb_is_private else "oci_core_subnet.lb_pub.id"
_lb_ipv6_subnet_expr = "local.lb_priv_ipv6" if _lb_is_private else "local.lb_pub_ipv6"
_lb_is_private_tf = "true" if _lb_is_private else "false"


# -----------------------------------------------------------------------------
# Derived toggles / blocks
# -----------------------------------------------------------------------------
_backend_shape_config_block = ""
if str(BACKEND_SHAPE).endswith(".Flex"):
    _backend_shape_config_block = """
  shape_config {
    ocpus         = var.backend_ocpus
    memory_in_gbs = var.backend_memory_gb
  }"""

_generator_shape_config_block = ""
if str(GENERATOR_SHAPE).endswith(".Flex"):
    _generator_shape_config_block = """
  shape_config {
    ocpus         = var.generator_ocpus
    memory_in_gbs = var.generator_memory_gb
  }"""

_enable_tcp_ppv2_default = str(PROTOCOL_PROFILE).lower() == "tcp_ppv2"
_listener_idle_timeout_sec = max(1, min(7200, int(AVG_SESSION_SEC)))


# -----------------------------------------------------------------------------
# Terraform main.tf
# -----------------------------------------------------------------------------
terraform_main_template = r'''terraform {
  required_providers {
    oci = {
      source  = "oracle/oci"
      version = ">= 8.10.0"
    }
  }
}

provider "oci" {
  config_file_profile = var.oci_profile
  region              = var.region
}

# ----------------------------------------------------------------------------
# Variables
# ----------------------------------------------------------------------------
variable "oci_profile"            { type = string }
variable "region"                 { type = string }
variable "compartment_id"         { type = string }
variable "ad_a"                   { type = string }
variable "image_id"               { type = string }
variable "ssh_public_key_content" { type = string }

variable "backend_count"          { type = number }
variable "generator_count"        { type = number }

variable "backend_shape"          { type = string }
variable "backend_ocpus"          { type = number }
variable "backend_memory_gb"      { type = number }

variable "generator_shape"        { type = string }
variable "generator_ocpus"        { type = number }
variable "generator_memory_gb"    { type = number }

variable "lb_count"               { type = number }
variable "lb_min_mbps"            { type = number }
variable "lb_max_mbps"            { type = number }

variable "enable_ipv6_frontend"   { type = bool }
variable "enable_tcp_ppv2_listener" {
  type    = bool
  default = __ENABLE_TCP_PPV2_DEFAULT__
}

variable "ssh_allowed_cidr"       { type = string }
variable "ssh_allowed_v6_cidr"    { type = string }

variable "ui_enable"              { type = bool }
variable "ui_web_port"            { type = number }
variable "ui_allowed_cidr"        { type = string }

variable "health_endpoint_path"   { type = string }

variable "lb_cert_pem"            { type = string }
variable "lb_key_pem"             { type = string }
variable "lb_ca_pem"              { type = string }

variable "backend_cloud_init_path"   { type = string }
variable "generator_cloud_init_path" { type = string }

variable "bastion_plugin_name" {
  type    = string
  default = "Bastion"
}

# ----------------------------------------------------------------------------
# Networking
# ----------------------------------------------------------------------------
resource "oci_core_vcn" "vcn" {
  cidr_block     = "10.0.0.0/16"
  compartment_id = var.compartment_id
  display_name   = "flb-concurrency-vcn"
  is_ipv6enabled = true
}

locals {
  vcn_ipv6_base      = oci_core_vcn.vcn.ipv6cidr_blocks[0]
  lb_priv_ipv6       = cidrsubnet(local.vcn_ipv6_base, 8, 1)
  backends_priv_ipv6 = cidrsubnet(local.vcn_ipv6_base, 8, 2)
  gens_pub_ipv6      = cidrsubnet(local.vcn_ipv6_base, 8, 3)
  lb_pub_ipv6        = cidrsubnet(local.vcn_ipv6_base, 8, 4)

  lb_ingress_cidr_v4 = __LB_INGRESS_CIDR_V4__
  lb_ingress_cidr_v6 = __LB_INGRESS_CIDR_V6_EXPR__
}

data "oci_core_services" "all_services" {}

locals {
  osn_services = [
    for s in data.oci_core_services.all_services.services : s
    if can(regex("all-.*-services-in-oracle-services-network", lower(s.cidr_block)))
  ]
  osn_service = length(local.osn_services) > 0 ? local.osn_services[0] : data.oci_core_services.all_services.services[0]
}

resource "oci_core_default_security_list" "default_sl" {
  manage_default_resource_id = oci_core_vcn.vcn.default_security_list_id

  egress_security_rules {
    protocol         = "all"
    destination      = "0.0.0.0/0"
    destination_type = "CIDR_BLOCK"
    stateless        = true
  }
  egress_security_rules {
    protocol         = "all"
    destination      = "::/0"
    destination_type = "CIDR_BLOCK"
    stateless        = true
  }
  ingress_security_rules {
    protocol    = "all"
    source      = "0.0.0.0/0"
    source_type = "CIDR_BLOCK"
    stateless   = true
  }
  ingress_security_rules {
    protocol    = "all"
    source      = "::/0"
    source_type = "CIDR_BLOCK"
    stateless   = true
  }
}

resource "oci_core_internet_gateway" "igw" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "flb-concurrency-igw"
}

resource "oci_core_nat_gateway" "nat" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "flb-concurrency-nat"
}

resource "oci_core_service_gateway" "sgw" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "flb-concurrency-sgw"

  services {
    service_id = local.osn_service.id
  }

  lifecycle {
    precondition {
      condition     = length(local.osn_services) > 0
      error_message = "No OSN service entry found in oci_core_services for this region."
    }
  }
}

resource "oci_core_route_table" "rt_public" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "flb-concurrency-rt-public"

  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }
  route_rules {
    destination       = "::/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }
}

resource "oci_core_route_table" "rt_private" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "flb-concurrency-rt-private"

  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_nat_gateway.nat.id
  }

  route_rules {
    destination       = local.osn_service.cidr_block
    destination_type  = "SERVICE_CIDR_BLOCK"
    network_entity_id = oci_core_service_gateway.sgw.id
  }
}

resource "oci_core_subnet" "lb_priv" {
  cidr_block                 = "10.0.1.0/24"
  ipv6cidr_block             = local.lb_priv_ipv6
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "flb-concurrency-lb-priv"
  route_table_id             = oci_core_route_table.rt_private.id
  prohibit_public_ip_on_vnic = true
}

resource "oci_core_subnet" "backends_priv" {
  cidr_block                 = "10.0.2.0/24"
  ipv6cidr_block             = local.backends_priv_ipv6
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "flb-concurrency-backends-priv"
  route_table_id             = oci_core_route_table.rt_private.id
  prohibit_public_ip_on_vnic = true
}

resource "oci_core_subnet" "gens_pub" {
  cidr_block                 = "10.0.3.0/24"
  ipv6cidr_block             = local.gens_pub_ipv6
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "flb-concurrency-generators-pub"
  route_table_id             = oci_core_route_table.rt_public.id
  prohibit_public_ip_on_vnic = false
}

resource "oci_core_subnet" "lb_pub" {
  cidr_block                 = "10.0.4.0/24"
  ipv6cidr_block             = local.lb_pub_ipv6
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "flb-concurrency-lb-pub"
  route_table_id             = oci_core_route_table.rt_public.id
  prohibit_public_ip_on_vnic = false
}

# ----------------------------------------------------------------------------
# NSGs
# ----------------------------------------------------------------------------
resource "oci_core_network_security_group" "nsg_lb" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "nsg-flb-lb"
}

resource "oci_core_network_security_group" "nsg_backends" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "nsg-flb-backends"
}

resource "oci_core_network_security_group" "nsg_generators" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "nsg-flb-generators"
}

resource "oci_core_network_security_group_security_rule" "lb_ingress_443_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "CIDR_BLOCK"
  source                    = local.lb_ingress_cidr_v4
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 443
      max = 443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_ingress_443_v6" {
  count                     = (var.enable_ipv6_frontend && trimspace(local.lb_ingress_cidr_v6) != "") ? 1 : 0
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "CIDR_BLOCK"
  source                    = local.lb_ingress_cidr_v6
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 443
      max = 443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_ingress_8443_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "CIDR_BLOCK"
  source                    = local.lb_ingress_cidr_v4
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 8443
      max = 8443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_ingress_8443_v6" {
  count                     = (var.enable_ipv6_frontend && trimspace(local.lb_ingress_cidr_v6) != "") ? 1 : 0
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "CIDR_BLOCK"
  source                    = local.lb_ingress_cidr_v6
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 8443
      max = 8443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_ingress_9443_v4" {
  count                     = var.enable_tcp_ppv2_listener ? 1 : 0
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "CIDR_BLOCK"
  source                    = local.lb_ingress_cidr_v4
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 9443
      max = 9443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_ingress_9443_v6" {
  count                     = (var.enable_tcp_ppv2_listener && var.enable_ipv6_frontend && trimspace(local.lb_ingress_cidr_v6) != "") ? 1 : 0
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "CIDR_BLOCK"
  source                    = local.lb_ingress_cidr_v6
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 9443
      max = 9443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_egress_to_clients_src443_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "CIDR_BLOCK"
  destination               = local.lb_ingress_cidr_v4
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 443
      max = 443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_egress_to_clients_src443_v6" {
  count                     = (var.enable_ipv6_frontend && trimspace(local.lb_ingress_cidr_v6) != "") ? 1 : 0
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "CIDR_BLOCK"
  destination               = local.lb_ingress_cidr_v6
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 443
      max = 443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_egress_to_clients_src8443_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "CIDR_BLOCK"
  destination               = local.lb_ingress_cidr_v4
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 8443
      max = 8443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_egress_to_clients_src8443_v6" {
  count                     = (var.enable_ipv6_frontend && trimspace(local.lb_ingress_cidr_v6) != "") ? 1 : 0
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "CIDR_BLOCK"
  destination               = local.lb_ingress_cidr_v6
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 8443
      max = 8443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_egress_to_clients_src9443_v4" {
  count                     = var.enable_tcp_ppv2_listener ? 1 : 0
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "CIDR_BLOCK"
  destination               = local.lb_ingress_cidr_v4
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 9443
      max = 9443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_egress_to_clients_src9443_v6" {
  count                     = (var.enable_tcp_ppv2_listener && var.enable_ipv6_frontend && trimspace(local.lb_ingress_cidr_v6) != "") ? 1 : 0
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "CIDR_BLOCK"
  destination               = local.lb_ingress_cidr_v6
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 9443
      max = 9443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_egress_80_to_backends" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "NETWORK_SECURITY_GROUP"
  destination               = oci_core_network_security_group.nsg_backends.id
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 80
      max = 80
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_ingress_from_backends_src80" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_backends.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 80
      max = 80
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_egress_to_generators_src443" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "NETWORK_SECURITY_GROUP"
  destination               = oci_core_network_security_group.nsg_generators.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 443
      max = 443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_egress_to_generators_src8443" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "NETWORK_SECURITY_GROUP"
  destination               = oci_core_network_security_group.nsg_generators.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 8443
      max = 8443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "lb_egress_to_generators_src9443" {
  count                     = var.enable_tcp_ppv2_listener ? 1 : 0
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "NETWORK_SECURITY_GROUP"
  destination               = oci_core_network_security_group.nsg_generators.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 9443
      max = 9443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "backends_ingress_80_from_lb" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_lb.id
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 80
      max = 80
    }
  }
}

resource "oci_core_network_security_group_security_rule" "backends_egress_all_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "backends_egress_all_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "::/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "gens_ingress_ssh_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "CIDR_BLOCK"
  source                    = var.ssh_allowed_cidr
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 22
      max = 22
    }
  }
}

resource "oci_core_network_security_group_security_rule" "gens_ingress_ssh_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "CIDR_BLOCK"
  source                    = var.ssh_allowed_v6_cidr
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 22
      max = 22
    }
  }
}

resource "oci_core_network_security_group_security_rule" "gens_ingress_locust_control" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_generators.id
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 5557
      max = 5558
    }
  }
}

resource "oci_core_network_security_group_security_rule" "gens_ingress_locust_control_return" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_generators.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 5557
      max = 5558
    }
  }
}

resource "oci_core_network_security_group_security_rule" "gens_ingress_ui" {
  count                     = var.ui_enable ? 1 : 0
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "CIDR_BLOCK"
  source                    = var.ui_allowed_cidr
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = var.ui_web_port
      max = var.ui_web_port
    }
  }
}

resource "oci_core_network_security_group_security_rule" "gens_ingress_from_lb_src443" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_lb.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 443
      max = 443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "gens_ingress_from_lb_src8443" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_lb.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 8443
      max = 8443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "gens_ingress_from_lb_src9443" {
  count                     = var.enable_tcp_ppv2_listener ? 1 : 0
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_lb.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 9443
      max = 9443
    }
  }
}

resource "oci_core_network_security_group_security_rule" "gens_egress_all_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "gens_egress_all_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "::/0"
  stateless                 = true
}

# ----------------------------------------------------------------------------
# Compute instances
# ----------------------------------------------------------------------------
resource "oci_core_instance" "backend" {
  count               = var.backend_count
  availability_domain = var.ad_a
  compartment_id      = var.compartment_id
  shape               = var.backend_shape
__BACKEND_SHAPE_CONFIG__
  source_details {
    source_type = "image"
    source_id   = var.image_id
  }

  create_vnic_details {
    subnet_id        = oci_core_subnet.backends_priv.id
    assign_public_ip = false
    nsg_ids          = [oci_core_network_security_group.nsg_backends.id]
  }

  agent_config {
    are_all_plugins_disabled = false
    is_management_disabled   = false
    is_monitoring_disabled   = false
    plugins_config {
      name          = var.bastion_plugin_name
      desired_state = "ENABLED"
    }
  }

  display_name = "flb-backend-${count.index}"

  metadata = {
    ssh_authorized_keys = var.ssh_public_key_content
    user_data           = base64encode(file(var.backend_cloud_init_path))
  }
}

resource "oci_core_instance" "generator" {
  count               = var.generator_count
  availability_domain = var.ad_a
  compartment_id      = var.compartment_id
  shape               = var.generator_shape
__GENERATOR_SHAPE_CONFIG__
  source_details {
    source_type = "image"
    source_id   = var.image_id
  }

  create_vnic_details {
    subnet_id        = oci_core_subnet.gens_pub.id
    assign_public_ip = true
    nsg_ids          = [oci_core_network_security_group.nsg_generators.id]
  }

  display_name = "flb-generator-${count.index}"

  metadata = {
    ssh_authorized_keys = var.ssh_public_key_content
    user_data           = filebase64(var.generator_cloud_init_path)
  }
}

# ----------------------------------------------------------------------------
# Load balancer + cert + listeners
# ----------------------------------------------------------------------------
resource "oci_load_balancer_load_balancer" "lb" {
  count                      = var.lb_count
  compartment_id             = var.compartment_id
  display_name               = "flb-session-lb-${count.index}"
  shape                      = "flexible"
  is_private                 = __LB_IS_PRIVATE__
  subnet_ids                 = [__LB_SUBNET_ID__]
  network_security_group_ids = [oci_core_network_security_group.nsg_lb.id]

  ip_mode         = var.enable_ipv6_frontend ? "IPV6" : null
  ipv6subnet_cidr = var.enable_ipv6_frontend ? __LB_IPV6_SUBNET_EXPR__ : null

  shape_details {
    minimum_bandwidth_in_mbps = var.lb_min_mbps
    maximum_bandwidth_in_mbps = var.lb_max_mbps
  }
}

resource "oci_load_balancer_certificate" "lb_cert" {
  count              = var.lb_count
  load_balancer_id   = oci_load_balancer_load_balancer.lb[count.index].id
  certificate_name   = "flb-cert-${count.index}"
  public_certificate = var.lb_cert_pem
  private_key        = var.lb_key_pem
  ca_certificate     = trimspace(var.lb_ca_pem) != "" ? var.lb_ca_pem : null
}

resource "oci_load_balancer_backend_set" "bs_http" {
  count            = var.lb_count
  load_balancer_id = oci_load_balancer_load_balancer.lb[count.index].id
  name             = "bs-http"
  policy           = "ROUND_ROBIN"

  health_checker {
    protocol          = "HTTP"
    port              = 80
    url_path          = var.health_endpoint_path
    retries           = 3
    interval_ms       = 10000
    timeout_in_millis = 5000
    return_code       = 200
  }
}

resource "oci_load_balancer_backend_set" "bs_tcp_ppv2" {
  count            = var.enable_tcp_ppv2_listener ? var.lb_count : 0
  load_balancer_id = oci_load_balancer_load_balancer.lb[count.index].id
  name             = "bs-tcp-ppv2"
  policy           = "ROUND_ROBIN"

  health_checker {
    protocol          = "TCP"
    port              = 80
    retries           = 3
    interval_ms       = 10000
    timeout_in_millis = 5000
  }
}

resource "oci_load_balancer_backend" "bs_http_backends" {
  count            = var.lb_count * var.backend_count
  load_balancer_id = oci_load_balancer_load_balancer.lb[floor(count.index / var.backend_count)].id
  backendset_name  = oci_load_balancer_backend_set.bs_http[floor(count.index / var.backend_count)].name
  ip_address       = oci_core_instance.backend[count.index % var.backend_count].private_ip
  port             = 80
  weight           = 1
}

resource "oci_load_balancer_backend" "bs_tcp_ppv2_backends" {
  count            = var.enable_tcp_ppv2_listener ? (var.lb_count * var.backend_count) : 0
  load_balancer_id = oci_load_balancer_load_balancer.lb[floor(count.index / var.backend_count)].id
  backendset_name  = oci_load_balancer_backend_set.bs_tcp_ppv2[floor(count.index / var.backend_count)].name
  ip_address       = oci_core_instance.backend[count.index % var.backend_count].private_ip
  port             = 80
  weight           = 1
}

resource "oci_load_balancer_listener" "h2_443" {
  count                    = var.lb_count
  load_balancer_id         = oci_load_balancer_load_balancer.lb[count.index].id
  name                     = "h2-443"
  port                     = 443
  protocol                 = "HTTP2"
  default_backend_set_name = oci_load_balancer_backend_set.bs_http[count.index].name

  ssl_configuration {
    certificate_name        = oci_load_balancer_certificate.lb_cert[count.index].certificate_name
    verify_peer_certificate = false
    protocols               = ["TLSv1.3", "TLSv1.2"]
    cipher_suite_name       = "oci-default-http2-tls-12-13-ssl-cipher-suite-v1"
    server_order_preference = "ENABLED"
  }

  connection_configuration {
    idle_timeout_in_seconds = __IDLE_TIMEOUT__
  }

  depends_on = [oci_load_balancer_certificate.lb_cert]
}

resource "oci_load_balancer_listener" "h1_https_8443" {
  count                    = var.lb_count
  load_balancer_id         = oci_load_balancer_load_balancer.lb[count.index].id
  name                     = "https-h1-8443"
  port                     = 8443
  protocol                 = "HTTP"
  default_backend_set_name = oci_load_balancer_backend_set.bs_http[count.index].name

  ssl_configuration {
    certificate_name        = oci_load_balancer_certificate.lb_cert[count.index].certificate_name
    verify_peer_certificate = false
    protocols               = ["TLSv1.3", "TLSv1.2"]
    cipher_suite_name       = "oci-modern-ssl-cipher-suite-v1"
    server_order_preference = "ENABLED"
  }

  connection_configuration {
    idle_timeout_in_seconds = __IDLE_TIMEOUT__
  }

  depends_on = [oci_load_balancer_certificate.lb_cert]
}

resource "oci_load_balancer_listener" "tcp_ppv2_9443" {
  count                    = var.enable_tcp_ppv2_listener ? var.lb_count : 0
  load_balancer_id         = oci_load_balancer_load_balancer.lb[count.index].id
  name                     = "tcp-ppv2-9443"
  port                     = 9443
  protocol                 = "TCP"
  default_backend_set_name = oci_load_balancer_backend_set.bs_tcp_ppv2[count.index].name

  ssl_configuration {
    certificate_name        = oci_load_balancer_certificate.lb_cert[count.index].certificate_name
    verify_peer_certificate = false
    protocols               = ["TLSv1.3", "TLSv1.2"]
    cipher_suite_name       = "oci-modern-ssl-cipher-suite-v1"
    server_order_preference = "ENABLED"
  }

  connection_configuration {
    idle_timeout_in_seconds            = __IDLE_TIMEOUT__
    backend_tcp_proxy_protocol_version = 2
  }

  depends_on = [oci_load_balancer_certificate.lb_cert]
}

# ----------------------------------------------------------------------------
# Outputs
# ----------------------------------------------------------------------------
output "lb_id" {
  value = oci_load_balancer_load_balancer.lb[0].id
}

output "lb_ids" {
  value = [for lb in oci_load_balancer_load_balancer.lb : lb.id]
}

output "lb_ip_addresses" {
  value = [for d in oci_load_balancer_load_balancer.lb[0].ip_address_details : d.ip_address]
}

output "lb_ip_addresses_multi" {
  value = [for lb in oci_load_balancer_load_balancer.lb : [for d in lb.ip_address_details : d.ip_address]]
}

output "lb_ip_addresses_flat" {
  value = flatten([for lb in oci_load_balancer_load_balancer.lb : [for d in lb.ip_address_details : d.ip_address]])
}

output "backend_private_ips" {
  value = [for i in oci_core_instance.backend : i.private_ip]
}

output "generator_public_ips" {
  value = [for i in oci_core_instance.generator : i.public_ip]
}

output "service_gateway_id" {
  value = oci_core_service_gateway.sgw.id
}

output "osn_service_cidr_block" {
  value = local.osn_service.cidr_block
}
'''

terraform_main = (
    terraform_main_template
    .replace("__BACKEND_SHAPE_CONFIG__", _backend_shape_config_block)
    .replace("__GENERATOR_SHAPE_CONFIG__", _generator_shape_config_block)
    .replace("__ENABLE_TCP_PPV2_DEFAULT__", "true" if _enable_tcp_ppv2_default else "false")
    .replace("__IDLE_TIMEOUT__", str(_listener_idle_timeout_sec))
    .replace("__LB_IS_PRIVATE__", _lb_is_private_tf)
    .replace("__LB_SUBNET_ID__", _lb_subnet_id_expr)
    .replace("__LB_IPV6_SUBNET_EXPR__", _lb_ipv6_subnet_expr)
    .replace("__LB_INGRESS_CIDR_V4__", json.dumps(_lb_ingress_cidr_v4))
    .replace("__LB_INGRESS_CIDR_V6_EXPR__", _lb_ingress_cidr_v6_expr)
)

main_tf_path = Path.cwd() / "main.tf"
main_tf_path.write_text(terraform_main, encoding="utf-8")


# -----------------------------------------------------------------------------
# Export context for Cell 6+
# -----------------------------------------------------------------------------
globals()["TERRAFORM_MAIN_PATH"] = str(main_tf_path)
os.environ["TERRAFORM_MAIN_PATH"] = str(main_tf_path)

os.environ["BACKEND_CLOUD_INIT_PATH"] = _backend_cloud_init_path
os.environ["GENERATOR_CLOUD_INIT_PATH"] = _generator_cloud_init_path

summary = {
    "main_tf": str(main_tf_path),
    "backend_cloud_init_path": _backend_cloud_init_path,
    "generator_cloud_init_path": _generator_cloud_init_path,
    "lb_profile": {
        "visibility": _lb_visibility,
        "is_private": _lb_is_private,
        "subnet": "lb_priv" if _lb_is_private else "lb_pub",
        "http2_listener": "443",
        "https_h1_listener": "8443",
        "tcp_ppv2_listener": "9443 (toggle: enable_tcp_ppv2_listener)",
        "idle_timeout_sec": _listener_idle_timeout_sec,
        "allowed_cidr_v4_effective": _lb_ingress_cidr_v4_display,
        "allowed_cidr_v6_effective": _lb_ingress_cidr_v6_display,
    },
    "networking": {
        "service_gateway_enabled": True,
        "private_route_table_includes_osn_service_cidr": True,
    },
    "security": "All NSG/default SL rules generated as stateless.",
}

print("Cell 5 complete: Terraform main.tf generated (visibility-aware LB + stateless security rules + service gateway route).")
print(json.dumps(summary, indent=2))
print("NEXT: Run Cell 6 to generate terraform.tfvars with full variable mapping.")


In [ ]:
# Cell 6 — Objective: Generate terraform.tfvars (complete + validated mapping to main.tf)

import os
import re
import json
from pathlib import Path


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def _hcl_bool(v: bool) -> str:
    return "true" if bool(v) else "false"


def _hcl_string(v: str) -> str:
    return json.dumps(str(v))


def _require_global(name: str):
    if name not in globals():
        raise RuntimeError(f"Missing required global: {name}. Run prior cells first.")
    return globals()[name]


def _resolve_path(path_value: str, label: str, required: bool = True) -> str:
    p = str(path_value or "").strip()
    if not p:
        if required:
            raise ValueError(f"{label} is empty.")
        return ""
    rp = str(Path(p).expanduser().resolve())
    if required and not os.path.exists(rp):
        raise FileNotFoundError(f"{label} not found: {rp}")
    return rp


def _read_text(path_value: str, label: str, required: bool = True) -> str:
    p = _resolve_path(path_value, label, required=required)
    if not p:
        return ""
    with open(p, "r", encoding="utf-8") as f:
        return f.read().strip()


def _heredoc(var_name: str, marker: str, value: str) -> str:
    return f"{var_name} = <<{marker}\n{value}\n{marker}"


# -----------------------------------------------------------------------------
# Preflight globals from Cells 2-5
# -----------------------------------------------------------------------------
_REQUIRED_GLOBALS = [
    "OCI_PROFILE",
    "REGION",
    "COMPARTMENT_ID",
    "AD_A",
    "IMAGE_ID",
    "SSH_PUBLIC_KEY_PATH",
    "BACKEND_COUNT",
    "GENERATOR_COUNT",
    "BACKEND_SHAPE",
    "BACKEND_OCPUS",
    "BACKEND_MEMORY_GB",
    "GENERATOR_SHAPE",
    "GENERATOR_OCPUS",
    "GENERATOR_MEMORY_GB",
    "LB_COUNT",
    "LB_MIN_MBPS",
    "LB_MAX_MBPS",
    "ENABLE_IPV6_FRONTEND",
    "SSH_ALLOWED_CIDR",
    "SSH_ALLOWED_V6_CIDR",
    "UI_ENABLE",
    "UI_WEB_PORT",
    "UI_ALLOWED_CIDR",
    "HEALTH_ENDPOINT_PATH",
    "LB_CERT_PEM_PATH",
    "LB_KEY_PEM_PATH",
    "LB_CA_PEM_PATH",
    "BACKEND_CLOUD_INIT_PATH",
    "GENERATOR_CLOUD_INIT_PATH",
    "PROTOCOL_PROFILE",
    "LB_VISIBILITY",
]
for _name in _REQUIRED_GLOBALS:
    _require_global(_name)

if not str(COMPARTMENT_ID).strip():
    raise ValueError("COMPARTMENT_ID is empty. Run Cell 3 and click Apply.")
if not str(AD_A).strip():
    raise ValueError("AD_A is empty. Run Cell 3 and click Apply.")
if not str(IMAGE_ID).strip():
    raise ValueError("IMAGE_ID is empty. Run Cell 3 and click Apply.")

if int(BACKEND_COUNT) < 1 or int(GENERATOR_COUNT) < 1:
    raise ValueError("BACKEND_COUNT and GENERATOR_COUNT must both be >= 1.")
if int(LB_COUNT) < 1:
    raise ValueError("LB_COUNT must be >= 1.")

if int(LB_MAX_MBPS) < int(LB_MIN_MBPS):
    raise ValueError(f"LB_MAX_MBPS ({LB_MAX_MBPS}) must be >= LB_MIN_MBPS ({LB_MIN_MBPS}).")

_protocol_profile = str(PROTOCOL_PROFILE).strip().lower()
if _protocol_profile not in {"http2_preferred", "https_h1_compatible", "tcp_ppv2"}:
    raise ValueError(f"Unsupported PROTOCOL_PROFILE: {PROTOCOL_PROFILE!r}")

_lb_visibility = str(LB_VISIBILITY).strip().lower()
if _lb_visibility not in {"public", "private"}:
    raise ValueError(f"LB_VISIBILITY must be public|private. Current: {LB_VISIBILITY!r}")


# -----------------------------------------------------------------------------
# Validate expected variables exist in main.tf
# -----------------------------------------------------------------------------
main_tf_path = Path(globals().get("TERRAFORM_MAIN_PATH", Path.cwd() / "main.tf")).expanduser().resolve()
if not main_tf_path.exists():
    raise FileNotFoundError(f"main.tf not found: {main_tf_path}")

main_tf_text = main_tf_path.read_text(encoding="utf-8")
main_tf_vars = set(re.findall(r'variable\s+"([^"]+)"', main_tf_text))

_expected_vars = {
    "oci_profile",
    "region",
    "compartment_id",
    "ad_a",
    "image_id",
    "ssh_public_key_content",
    "backend_count",
    "generator_count",
    "backend_shape",
    "backend_ocpus",
    "backend_memory_gb",
    "generator_shape",
    "generator_ocpus",
    "generator_memory_gb",
    "lb_count",
    "lb_min_mbps",
    "lb_max_mbps",
    "enable_ipv6_frontend",
    "enable_tcp_ppv2_listener",
    "ssh_allowed_cidr",
    "ssh_allowed_v6_cidr",
    "ui_enable",
    "ui_web_port",
    "ui_allowed_cidr",
    "health_endpoint_path",
    "lb_cert_pem",
    "lb_key_pem",
    "lb_ca_pem",
    "backend_cloud_init_path",
    "generator_cloud_init_path",
}
_missing_in_main = sorted(_expected_vars - main_tf_vars)
if _missing_in_main:
    raise RuntimeError(
        "main.tf variable mismatch. Missing expected variables: "
        + ", ".join(_missing_in_main)
    )


# -----------------------------------------------------------------------------
# Resolve paths and read file content
# -----------------------------------------------------------------------------
ssh_public_key_path = _resolve_path(SSH_PUBLIC_KEY_PATH, "SSH_PUBLIC_KEY_PATH", required=True)
backend_cloud_init_path = _resolve_path(BACKEND_CLOUD_INIT_PATH, "BACKEND_CLOUD_INIT_PATH", required=True)
generator_cloud_init_path = _resolve_path(GENERATOR_CLOUD_INIT_PATH, "GENERATOR_CLOUD_INIT_PATH", required=True)

lb_cert_pem_path = _resolve_path(LB_CERT_PEM_PATH, "LB_CERT_PEM_PATH", required=True)
lb_key_pem_path = _resolve_path(LB_KEY_PEM_PATH, "LB_KEY_PEM_PATH", required=True)
lb_ca_pem_path = _resolve_path(LB_CA_PEM_PATH, "LB_CA_PEM_PATH", required=False) if str(LB_CA_PEM_PATH).strip() else ""

ssh_public_key_content = _read_text(ssh_public_key_path, "SSH public key", required=True)
lb_cert_pem = _read_text(lb_cert_pem_path, "LB cert PEM", required=True)
lb_key_pem = _read_text(lb_key_pem_path, "LB key PEM", required=True)
lb_ca_pem = _read_text(lb_ca_pem_path, "LB CA PEM", required=False) if lb_ca_pem_path else ""

if not ssh_public_key_content:
    raise ValueError(f"SSH public key file is empty: {ssh_public_key_path}")
if not lb_cert_pem:
    raise ValueError(f"LB certificate PEM is empty: {lb_cert_pem_path}")
if not lb_key_pem:
    raise ValueError(f"LB private key PEM is empty: {lb_key_pem_path}")

enable_tcp_ppv2_listener = _protocol_profile == "tcp_ppv2"


# -----------------------------------------------------------------------------
# Build terraform.tfvars (complete mapping)
# -----------------------------------------------------------------------------
lines = [
    f"oci_profile = {_hcl_string(OCI_PROFILE)}",
    f"region = {_hcl_string(REGION)}",
    f"compartment_id = {_hcl_string(COMPARTMENT_ID)}",
    f"ad_a = {_hcl_string(AD_A)}",
    f"image_id = {_hcl_string(IMAGE_ID)}",
    f"backend_count = {int(BACKEND_COUNT)}",
    f"generator_count = {int(GENERATOR_COUNT)}",
    f"backend_shape = {_hcl_string(BACKEND_SHAPE)}",
    f"backend_ocpus = {float(BACKEND_OCPUS)}",
    f"backend_memory_gb = {float(BACKEND_MEMORY_GB)}",
    f"generator_shape = {_hcl_string(GENERATOR_SHAPE)}",
    f"generator_ocpus = {float(GENERATOR_OCPUS)}",
    f"generator_memory_gb = {float(GENERATOR_MEMORY_GB)}",
    f"lb_count = {int(LB_COUNT)}",
    f"lb_min_mbps = {int(LB_MIN_MBPS)}",
    f"lb_max_mbps = {int(LB_MAX_MBPS)}",
    f"enable_ipv6_frontend = {_hcl_bool(ENABLE_IPV6_FRONTEND)}",
    f"enable_tcp_ppv2_listener = {_hcl_bool(enable_tcp_ppv2_listener)}",
    f"ssh_allowed_cidr = {_hcl_string(SSH_ALLOWED_CIDR)}",
    f"ssh_allowed_v6_cidr = {_hcl_string(SSH_ALLOWED_V6_CIDR)}",
    f"ui_enable = {_hcl_bool(UI_ENABLE)}",
    f"ui_web_port = {int(UI_WEB_PORT)}",
    f"ui_allowed_cidr = {_hcl_string(UI_ALLOWED_CIDR)}",
    f"health_endpoint_path = {_hcl_string(HEALTH_ENDPOINT_PATH)}",
    f"backend_cloud_init_path = {_hcl_string(backend_cloud_init_path)}",
    f"generator_cloud_init_path = {_hcl_string(generator_cloud_init_path)}",
]

if "bastion_plugin_name" in main_tf_vars:
    bastion_name = str(globals().get("BASTION_PLUGIN_NAME", "Bastion")).strip() or "Bastion"
    lines.append(f"bastion_plugin_name = {_hcl_string(bastion_name)}")

lines += [
    _heredoc("ssh_public_key_content", "EOSSH", ssh_public_key_content),
    _heredoc("lb_cert_pem", "EOCERT", lb_cert_pem),
    _heredoc("lb_key_pem", "EOKEY", lb_key_pem),
    _heredoc("lb_ca_pem", "EOCA", lb_ca_pem),
]

tfvars_text = "\n\n".join(lines) + "\n"

tfvars_path = (main_tf_path.parent / "terraform.tfvars").resolve()
tfvars_path.write_text(tfvars_text, encoding="utf-8")

globals()["TERRAFORM_TFVARS_PATH"] = str(tfvars_path)
os.environ["TERRAFORM_TFVARS_PATH"] = str(tfvars_path)

summary = {
    "terraform_tfvars": str(tfvars_path),
    "main_tf": str(main_tf_path),
    "protocol_profile": _protocol_profile,
    "enable_tcp_ppv2_listener": enable_tcp_ppv2_listener,
    "lb_visibility": _lb_visibility,
    "lb_count": int(LB_COUNT),
    "backend_count": int(BACKEND_COUNT),
    "generator_count": int(GENERATOR_COUNT),
    "enable_ipv6_frontend": bool(ENABLE_IPV6_FRONTEND),
    "note": "LB visibility/CIDR behavior is compiled into main.tf by Cell 5; tfvars keeps runtime inputs only.",
}

print("Cell 6 complete: terraform.tfvars generated with full main.tf variable mapping.")
print(json.dumps(summary, indent=2))
print("NEXT: Run Cell 7 for terraform init/apply.")


In [ ]:
# Cell 7 — Objective: Initialize, validate, plan, and apply Terraform (non-interactive)

import os
import json
import shutil
import subprocess
from pathlib import Path


def _run(cmd: list[str], cwd: Path, env: dict) -> str:
    proc = subprocess.run(cmd, cwd=str(cwd), env=env, text=True, capture_output=True)
    if proc.returncode != 0:
        print(f"\nFAILED: {' '.join(cmd)}")
        if proc.stdout:
            print("\n--- stdout ---")
            print(proc.stdout)
        if proc.stderr:
            print("\n--- stderr ---")
            print(proc.stderr)
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {' '.join(cmd)}")
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    return proc.stdout or ""


# -----------------------------------------------------------------------------
# Preflight
# -----------------------------------------------------------------------------
terraform_bin = shutil.which("terraform")
if not terraform_bin:
    raise EnvironmentError("Terraform CLI not found in PATH. Install Terraform and rerun Cell 7.")

main_tf_path = Path(globals().get("TERRAFORM_MAIN_PATH", Path.cwd() / "main.tf")).expanduser().resolve()
tfvars_path = Path(globals().get("TERRAFORM_TFVARS_PATH", Path.cwd() / "terraform.tfvars")).expanduser().resolve()

if not main_tf_path.exists():
    raise FileNotFoundError(f"main.tf not found: {main_tf_path}")
if not tfvars_path.exists():
    raise FileNotFoundError(f"terraform.tfvars not found: {tfvars_path}")

workdir = main_tf_path.parent.resolve()
plan_path = workdir / "tfplan"
outputs_json_path = workdir / "terraform_outputs.json"

# Use a local writable TF data dir (helps avoid sync-folder/plugin state issues)
tf_data_dir = Path(globals().get("TF_DATA_DIR", "/private/tmp/flb_tf_data")).expanduser().resolve()
tf_data_dir.mkdir(parents=True, exist_ok=True)

if plan_path.exists():
    plan_path.unlink()

env = os.environ.copy()
env["TF_IN_AUTOMATION"] = "1"
env["TF_DATA_DIR"] = str(tf_data_dir)
env["TF_CLI_ARGS"] = "-no-color"

print("Terraform binary:", terraform_bin)
print("Terraform working dir:", workdir)
print("Using tfvars:", tfvars_path)
print("Using TF_DATA_DIR:", tf_data_dir)

# -----------------------------------------------------------------------------
# Terraform workflow
# -----------------------------------------------------------------------------
print("\n[1/5] terraform version")
_run([terraform_bin, "version"], cwd=workdir, env=env)

print("\n[2/5] terraform init (reconfigure)")
_run([terraform_bin, "init", "-input=false", "-reconfigure"], cwd=workdir, env=env)

print("\n[3/5] terraform validate")
_run([terraform_bin, "validate"], cwd=workdir, env=env)

print("\n[4/5] terraform plan")
_run(
    [
        terraform_bin,
        "plan",
        "-input=false",
        "-lock-timeout=10m",
        f"-var-file={tfvars_path}",
        f"-out={plan_path}",
    ],
    cwd=workdir,
    env=env,
)

print("\n[5/5] terraform apply")
_run(
    [terraform_bin, "apply", "-input=false", "-auto-approve", "-lock-timeout=10m", str(plan_path)],
    cwd=workdir,
    env=env,
)

print("\n[post] terraform output -json")
outputs_json = _run([terraform_bin, "output", "-json"], cwd=workdir, env=env)

# Validate JSON before persisting
json.loads(outputs_json)
outputs_json_path.write_text(outputs_json, encoding="utf-8")

# Export for downstream cells
globals()["TERRAFORM_PLAN_PATH"] = str(plan_path)
globals()["TERRAFORM_OUTPUTS_JSON_PATH"] = str(outputs_json_path)
globals()["TF_DATA_DIR"] = str(tf_data_dir)

os.environ["TERRAFORM_PLAN_PATH"] = str(plan_path)
os.environ["TERRAFORM_OUTPUTS_JSON_PATH"] = str(outputs_json_path)
os.environ["TF_DATA_DIR"] = str(tf_data_dir)

summary = {
    "workdir": str(workdir),
    "main_tf": str(main_tf_path),
    "terraform_tfvars": str(tfvars_path),
    "tf_data_dir": str(tf_data_dir),
    "tfplan": str(plan_path),
    "terraform_outputs_json": str(outputs_json_path),
}
print("Cell 7 complete: terraform init/validate/plan/apply succeeded.")
print(json.dumps(summary, indent=2))
print("NEXT: Run Cell 8 to parse and persist selected Terraform outputs for downstream test orchestration.")


In [ ]:
# Cell 8 — Objective: Parse Terraform outputs and persist run targets for downstream orchestration

import os
import json
import shutil
import ipaddress
import subprocess
from pathlib import Path
from datetime import datetime, UTC


def _run(cmd, cwd: Path, env: dict) -> str:
    proc = subprocess.run(cmd, cwd=str(cwd), env=env, text=True, capture_output=True)
    if proc.returncode != 0:
        print(f"\nFAILED: {' '.join(cmd)}")
        if proc.stdout:
            print("\n--- stdout ---")
            print(proc.stdout)
        if proc.stderr:
            print("\n--- stderr ---")
            print(proc.stderr)
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {' '.join(cmd)}")
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    return proc.stdout or ""


def _normalize_list(v):
    if v is None:
        return []
    if isinstance(v, list):
        return [str(x) for x in v]
    return [str(v)]


def _safe_ip(s: str):
    try:
        return ipaddress.ip_address(str(s))
    except Exception:
        return None


def _url_host(ip: str) -> str:
    obj = _safe_ip(ip)
    if obj and obj.version == 6:
        return f"[{ip}]"
    return ip


def _tf_value(tf_outputs: dict, name: str, required: bool = True, default=None):
    node = tf_outputs.get(name)
    if node is None:
        if required:
            raise KeyError(f"Required Terraform output missing: {name}")
        return default
    if isinstance(node, dict) and "value" in node:
        return node["value"]
    if required:
        raise KeyError(f"Terraform output does not contain 'value': {name}")
    return default


def _load_outputs(terraform_bin: str, workdir: Path, outputs_json_path: Path, env: dict) -> dict:
    # Prefer existing file from Cell 7
    if outputs_json_path.exists() and outputs_json_path.stat().st_size > 0:
        return json.loads(outputs_json_path.read_text(encoding="utf-8"))

    print("[preflight] terraform_outputs.json missing/empty, regenerating via terraform output -json")
    try:
        out = _run([terraform_bin, "output", "-json"], cwd=workdir, env=env)
    except Exception:
        print("[preflight] terraform output failed; running terraform init -reconfigure and retrying output")
        _run([terraform_bin, "init", "-input=false", "-reconfigure"], cwd=workdir, env=env)
        out = _run([terraform_bin, "output", "-json"], cwd=workdir, env=env)

    outputs_json_path.write_text(out, encoding="utf-8")
    return json.loads(out)


# -----------------------------------------------------------------------------
# Preflight
# -----------------------------------------------------------------------------
terraform_bin = shutil.which("terraform")
if not terraform_bin:
    raise EnvironmentError("Terraform CLI not found in PATH. Install Terraform and rerun Cell 8.")

main_tf_path = Path(globals().get("TERRAFORM_MAIN_PATH", Path.cwd() / "main.tf")).expanduser().resolve()
if not main_tf_path.exists():
    raise FileNotFoundError(f"main.tf not found: {main_tf_path}")

workdir = main_tf_path.parent.resolve()
outputs_json_path = Path(
    globals().get("TERRAFORM_OUTPUTS_JSON_PATH", workdir / "terraform_outputs.json")
).expanduser().resolve()

tf_data_dir = Path(
    globals().get("TF_DATA_DIR", os.environ.get("TF_DATA_DIR", "/private/tmp/flb_tf_data"))
).expanduser().resolve()
tf_data_dir.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["TF_IN_AUTOMATION"] = "1"
env["TF_DATA_DIR"] = str(tf_data_dir)
env["TF_CLI_ARGS"] = "-no-color"

tf_outputs = _load_outputs(terraform_bin, workdir, outputs_json_path, env)


# -----------------------------------------------------------------------------
# Extract outputs
# -----------------------------------------------------------------------------
lb_id = _tf_value(tf_outputs, "lb_id", required=False, default="")
lb_ids = _normalize_list(_tf_value(tf_outputs, "lb_ids", required=False, default=[]))

lb_ips_flat = _tf_value(tf_outputs, "lb_ip_addresses_flat", required=False, default=None)
if lb_ips_flat is None:
    lb_ips_flat = _tf_value(tf_outputs, "lb_ip_addresses", required=False, default=[])
lb_ips_flat = _normalize_list(lb_ips_flat)

backend_private_ips = _normalize_list(_tf_value(tf_outputs, "backend_private_ips", required=False, default=[]))
generator_public_ips = _normalize_list(_tf_value(tf_outputs, "generator_public_ips", required=False, default=[]))

service_gateway_id = str(_tf_value(tf_outputs, "service_gateway_id", required=False, default="") or "")
osn_service_cidr_block = str(_tf_value(tf_outputs, "osn_service_cidr_block", required=False, default="") or "")

if len(lb_ips_flat) == 0:
    raise RuntimeError("No LB IP addresses found in Terraform outputs.")
if len(generator_public_ips) == 0:
    raise RuntimeError("No generator public IPs found in Terraform outputs.")
if len(backend_private_ips) == 0:
    raise RuntimeError("No backend private IPs found in Terraform outputs.")

lb_ipv4 = []
lb_ipv6 = []
for ip in lb_ips_flat:
    obj = _safe_ip(ip)
    if not obj:
        continue
    if obj.version == 4:
        lb_ipv4.append(ip)
    elif obj.version == 6:
        lb_ipv6.append(ip)

lb_visibility = str(globals().get("LB_VISIBILITY", "public")).strip().lower()
if lb_visibility not in {"public", "private"}:
    lb_visibility = "public"

if lb_visibility == "public":
    public_v4 = [ip for ip in lb_ipv4 if not _safe_ip(ip).is_private]
    lb_primary_ip = public_v4[0] if public_v4 else (lb_ipv4[0] if lb_ipv4 else lb_ips_flat[0])
else:
    private_v4 = [ip for ip in lb_ipv4 if _safe_ip(ip).is_private]
    lb_primary_ip = private_v4[0] if private_v4 else (lb_ipv4[0] if lb_ipv4 else lb_ips_flat[0])

lb_primary_ipv4 = lb_ipv4[0] if lb_ipv4 else ""
lb_primary_ipv6 = lb_ipv6[0] if lb_ipv6 else ""

# -----------------------------------------------------------------------------
# Session/profile context from globals (Cell 2/3)
# -----------------------------------------------------------------------------
protocol_profile = str(globals().get("PROTOCOL_PROFILE", "http2_preferred"))
health_path = str(globals().get("HEALTH_ENDPOINT_PATH", "/healthz"))
upload_path = str(globals().get("UPLOAD_ENDPOINT_PATH", "/upload_5k"))
enable_ipv6_frontend = bool(globals().get("ENABLE_IPV6_FRONTEND", True))

upload_ack_enabled = bool(globals().get("UPLOAD_ACK_ENABLED", True))
upload_ack_status = int(globals().get("UPLOAD_ACK_STATUS", 200))
upload_ack_body = str(globals().get("UPLOAD_ACK_BODY", "ack"))

target_concurrency = int(globals().get("TARGET_CONCURRENCY", 0) or 0)
avg_session_sec = int(globals().get("AVG_SESSION_SEC", 0) or 0)
target_opens_per_sec = float(
    globals().get(
        "TARGET_OPENS_PER_SEC",
        (target_concurrency / avg_session_sec) if (target_concurrency > 0 and avg_session_sec > 0) else 0.0,
    )
)

primary_host = _url_host(lb_primary_ip)
target_url_443 = f"https://{primary_host}:443"
target_url_8443 = f"https://{primary_host}:8443"
target_url_9443 = f"https://{primary_host}:9443"

if protocol_profile == "http2_preferred":
    locust_default_host = target_url_443
elif protocol_profile == "https_h1_compatible":
    locust_default_host = target_url_8443
else:
    locust_default_host = ""

# -----------------------------------------------------------------------------
# Persist selected outputs for downstream cells
# -----------------------------------------------------------------------------
run_id = str(globals().get("RUN_ID", f"flb_run_{datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')}"))
output_dir = Path(globals().get("OUTPUT_DIR", workdir / "results")).expanduser().resolve()
run_dir = (output_dir / run_id).resolve()
run_dir.mkdir(parents=True, exist_ok=True)

selected_outputs_path = run_dir / "selected_outputs.json"
selected_env_path = run_dir / "selected_outputs.env"

selected = {
    "run_id": run_id,
    "generated_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "terraform": {
        "workdir": str(workdir),
        "tf_data_dir": str(tf_data_dir),
        "outputs_json": str(outputs_json_path),
        "lb_visibility": lb_visibility,
        "lb_id": lb_id,
        "lb_ids": lb_ids,
        "lb_ip_addresses": lb_ips_flat,
        "lb_ip_addresses_v4": lb_ipv4,
        "lb_ip_addresses_v6": lb_ipv6,
        "lb_primary_ip": lb_primary_ip,
        "lb_primary_ipv4": lb_primary_ipv4,
        "lb_primary_ipv6": lb_primary_ipv6,
        "backend_private_ips": backend_private_ips,
        "generator_public_ips": generator_public_ips,
        "service_gateway_id": service_gateway_id,
        "osn_service_cidr_block": osn_service_cidr_block,
    },
    "profile": {
        "protocol_profile": protocol_profile,
        "enable_ipv6_frontend": enable_ipv6_frontend,
        "health_endpoint_path": health_path,
        "upload_endpoint_path": upload_path,
        "upload_ack_enabled": upload_ack_enabled,
        "upload_ack_status": upload_ack_status,
        "upload_ack_body": upload_ack_body,
        "target_url_443": target_url_443,
        "target_url_8443": target_url_8443,
        "target_url_9443": target_url_9443,
        "locust_default_host": locust_default_host,
    },
    "session": {
        "target_concurrency": target_concurrency,
        "avg_session_sec": avg_session_sec,
        "target_opens_per_sec": round(target_opens_per_sec, 3),
        "target_closes_per_sec": round(target_opens_per_sec, 3),
    },
}

selected_outputs_path.write_text(json.dumps(selected, indent=2), encoding="utf-8")

env_lines = [
    f'export RUN_ID="{run_id}"',
    f'export LB_VISIBILITY="{lb_visibility}"',
    f'export LB_PRIMARY_IP="{lb_primary_ip}"',
    f'export LB_PRIMARY_IPV4="{lb_primary_ipv4}"',
    f'export LB_PRIMARY_IPV6="{lb_primary_ipv6}"',
    f'export LB_IP_ADDRESSES="{",".join(lb_ips_flat)}"',
    f'export GENERATOR_PUBLIC_IPS="{",".join(generator_public_ips)}"',
    f'export BACKEND_PRIVATE_IPS="{",".join(backend_private_ips)}"',
    f'export TARGET_URL_443="{target_url_443}"',
    f'export TARGET_URL_8443="{target_url_8443}"',
    f'export TARGET_URL_9443="{target_url_9443}"',
    f'export HEALTH_ENDPOINT_PATH="{health_path}"',
    f'export UPLOAD_ENDPOINT_PATH="{upload_path}"',
    f'export PROTOCOL_PROFILE="{protocol_profile}"',
    f'export LOCUST_DEFAULT_HOST="{locust_default_host}"',
]
selected_env_path.write_text("\n".join(env_lines) + "\n", encoding="utf-8")

# Export globals/env for downstream cells
globals()["RUN_OUTPUT_DIR"] = str(run_dir)
globals()["SELECTED_OUTPUTS_PATH"] = str(selected_outputs_path)
globals()["SELECTED_OUTPUTS_ENV_PATH"] = str(selected_env_path)

globals()["LB_VISIBILITY"] = lb_visibility
globals()["LB_PRIMARY_IP"] = lb_primary_ip
globals()["LB_PRIMARY_IPV4"] = lb_primary_ipv4
globals()["LB_PRIMARY_IPV6"] = lb_primary_ipv6
globals()["LB_IP_ADDRESSES"] = lb_ips_flat
globals()["LB_IP_ADDRESSES_V4"] = lb_ipv4
globals()["LB_IP_ADDRESSES_V6"] = lb_ipv6

globals()["GENERATOR_PUBLIC_IPS"] = generator_public_ips
globals()["BACKEND_PRIVATE_IPS"] = backend_private_ips
globals()["SERVICE_GATEWAY_ID"] = service_gateway_id
globals()["OSN_SERVICE_CIDR_BLOCK"] = osn_service_cidr_block

globals()["TARGET_URL_443"] = target_url_443
globals()["TARGET_URL_8443"] = target_url_8443
globals()["TARGET_URL_9443"] = target_url_9443
globals()["LOCUST_DEFAULT_HOST"] = locust_default_host

os.environ["RUN_OUTPUT_DIR"] = str(run_dir)
os.environ["SELECTED_OUTPUTS_PATH"] = str(selected_outputs_path)
os.environ["SELECTED_OUTPUTS_ENV_PATH"] = str(selected_env_path)

os.environ["LB_VISIBILITY"] = lb_visibility
os.environ["LB_PRIMARY_IP"] = lb_primary_ip
os.environ["LB_PRIMARY_IPV4"] = lb_primary_ipv4
os.environ["LB_PRIMARY_IPV6"] = lb_primary_ipv6
os.environ["LB_IP_ADDRESSES"] = ",".join(lb_ips_flat)
os.environ["GENERATOR_PUBLIC_IPS"] = ",".join(generator_public_ips)
os.environ["BACKEND_PRIVATE_IPS"] = ",".join(backend_private_ips)
os.environ["SERVICE_GATEWAY_ID"] = service_gateway_id
os.environ["OSN_SERVICE_CIDR_BLOCK"] = osn_service_cidr_block

os.environ["TARGET_URL_443"] = target_url_443
os.environ["TARGET_URL_8443"] = target_url_8443
os.environ["TARGET_URL_9443"] = target_url_9443
os.environ["LOCUST_DEFAULT_HOST"] = locust_default_host

summary = {
    "selected_outputs_json": str(selected_outputs_path),
    "selected_outputs_env": str(selected_env_path),
    "lb_visibility": lb_visibility,
    "lb_primary_ip": lb_primary_ip,
    "lb_primary_ipv4": lb_primary_ipv4,
    "lb_primary_ipv6": lb_primary_ipv6,
    "generator_public_ips": generator_public_ips,
    "backend_private_ips_count": len(backend_private_ips),
    "protocol_profile": protocol_profile,
    "locust_default_host": locust_default_host,
    "target_opens_per_sec": round(target_opens_per_sec, 3),
}
print("Cell 8 complete: Terraform outputs parsed and persisted for downstream orchestration.")
print(json.dumps(summary, indent=2))
print("NEXT: Continue with your SSH helper / Locust orchestration cells.")


In [ ]:
# Cell 9 — Objective: Generator/Backend preflight + SSH helper setup (before Locust run)

import os
import json
import shlex
import subprocess
from pathlib import Path
from datetime import datetime, UTC


def _run_local(cmd: list[str], input_text: str | None = None, check: bool = True) -> subprocess.CompletedProcess:
    proc = subprocess.run(cmd, input=input_text, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {' '.join(cmd)}")
    return proc


def _ssh_script(generator_ip: str, key_path: str, script: str, timeout_sec: int = 30, check: bool = True) -> subprocess.CompletedProcess:
    cmd = [
        "ssh",
        "-o",
        "StrictHostKeyChecking=no",
        "-o",
        "BatchMode=yes",
        "-o",
        f"ConnectTimeout={timeout_sec}",
        "-i",
        key_path,
        f"opc@{generator_ip}",
        "bash -s",
    ]
    return _run_local(cmd, input_text=script, check=check)


# -----------------------------------------------------------------------------
# Preflight inputs
# -----------------------------------------------------------------------------
selected_outputs_path = Path(
    globals().get(
        "SELECTED_OUTPUTS_PATH",
        Path(globals().get("RUN_OUTPUT_DIR", Path.cwd() / "results")) / "selected_outputs.json",
    )
).expanduser().resolve()

if not selected_outputs_path.exists():
    raise FileNotFoundError(
        f"selected_outputs.json not found: {selected_outputs_path}. "
        "Run Cell 8 first."
    )

selected = json.loads(selected_outputs_path.read_text(encoding="utf-8"))
tf = selected.get("terraform", {})
profile = selected.get("profile", {})

generator_public_ips = [str(x) for x in tf.get("generator_public_ips", [])]
backend_private_ips = [str(x) for x in tf.get("backend_private_ips", [])]

if not generator_public_ips:
    raise RuntimeError("No generator public IPs found in selected outputs.")
if not backend_private_ips:
    raise RuntimeError("No backend private IPs found in selected outputs.")

generator_ip = generator_public_ips[0]
lb_primary_ip = str(tf.get("lb_primary_ip", ""))
locust_default_host = str(profile.get("locust_default_host", globals().get("LOCUST_DEFAULT_HOST", "")))
health_path = str(profile.get("health_endpoint_path", globals().get("HEALTH_ENDPOINT_PATH", "/healthz")))
upload_path = str(profile.get("upload_endpoint_path", globals().get("UPLOAD_ENDPOINT_PATH", "/upload_5k")))

upload_ack_enabled = bool(globals().get("UPLOAD_ACK_ENABLED", True))
upload_ack_status = int(globals().get("UPLOAD_ACK_STATUS", 200 if upload_ack_enabled else 204))
upload_ack_body_expected = str(globals().get("UPLOAD_ACK_BODY", "ack" if upload_ack_enabled else "")).strip()

locust_workdir = str(globals().get("LOCUST_WORKDIR", "/home/opc/locustwork"))
ssh_private_key_path = str(
    Path(globals().get("SSH_PRIVATE_KEY_PATH", os.environ.get("SSH_PRIVATE_KEY_PATH", "~/.ssh/SandboxKey")))
    .expanduser()
    .resolve()
)
if not Path(ssh_private_key_path).exists():
    raise FileNotFoundError(f"SSH private key not found: {ssh_private_key_path}")

run_dir = selected_outputs_path.parent.resolve()
run_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 1) Generator readiness check (Locust + Python)
# -----------------------------------------------------------------------------
locust_check_script = f"""
set -euo pipefail
echo "GEN_HOSTNAME $(hostname)"
LOCUST_WORKDIR={shlex.quote(locust_workdir)}

if command -v locust >/dev/null 2>&1; then
  echo "LOCUST_BIN $(command -v locust)"
  echo "LOCUST_VERSION $(locust --version | head -n1)"
elif [ -x "${{LOCUST_WORKDIR}}/.venv/bin/locust" ]; then
  echo "LOCUST_BIN ${{LOCUST_WORKDIR}}/.venv/bin/locust"
  echo "LOCUST_VERSION $(${{LOCUST_WORKDIR}}/.venv/bin/locust --version | head -n1)"
else
  echo "LOCUST_MISSING true"
  exit 22
fi

python3 --version | sed 's/^/PYTHON_VERSION /'
"""

print("[1/3] Checking generator readiness (Locust/Python)...")
locust_check = _ssh_script(generator_ip, ssh_private_key_path, locust_check_script, timeout_sec=20, check=True)
(run_dir / "cell9_generator_readiness.log").write_text(
    (locust_check.stdout or "") + "\n" + (locust_check.stderr or ""),
    encoding="utf-8",
)

if "LOCUST_VERSION " not in (locust_check.stdout or ""):
    raise RuntimeError("Locust version line not found in generator readiness output.")
if "LOCUST_MISSING true" in (locust_check.stdout or ""):
    raise RuntimeError("Locust is not installed on generator.")

# -----------------------------------------------------------------------------
# 2) Backend readiness check from generator (health + upload ack)
# -----------------------------------------------------------------------------
backend_list_literal = " ".join(shlex.quote(ip) for ip in backend_private_ips)

backend_check_script = f"""
set -euo pipefail
BACKENDS=({backend_list_literal})
HEALTH_PATH={shlex.quote(health_path)}
UPLOAD_PATH={shlex.quote(upload_path)}

ok=0
fail=0

for ip in "${{BACKENDS[@]}}"; do
  code=$(curl -sS -o /dev/null -w "%{{http_code}}" --max-time 5 "http://${{ip}}${{HEALTH_PATH}}" || true)
  echo "HEALTH ${{ip}} ${{code}}"
  if [ "${{code}}" = "200" ]; then
    ok=$((ok+1))
  else
    fail=$((fail+1))
  fi
done

first_ip="${{BACKENDS[0]}}"
ack_code=$(curl -sS -o /tmp/flb_ack_body -w "%{{http_code}}" --max-time 10 -X POST --data-binary @/etc/hosts "http://${{first_ip}}${{UPLOAD_PATH}}" || true)
echo "ACK_CODE ${{ack_code}}"

python3 - <<'PY'
import json, pathlib
p = pathlib.Path('/tmp/flb_ack_body')
body = p.read_text(encoding='utf-8') if p.exists() else ""
print("ACK_BODY_JSON", json.dumps(body))
PY

echo "SUMMARY ${{ok}} ${{fail}}"
"""

print("[2/3] Checking backend readiness from generator (health/upload)...")
backend_check = _ssh_script(generator_ip, ssh_private_key_path, backend_check_script, timeout_sec=30, check=True)
(run_dir / "cell9_backend_readiness.log").write_text(
    (backend_check.stdout or "") + "\n" + (backend_check.stderr or ""),
    encoding="utf-8",
)

health_results: dict[str, str] = {}
ack_code = None
ack_body = ""
summary_ok = None
summary_fail = None

for raw in (backend_check.stdout or "").splitlines():
    line = raw.strip()
    if line.startswith("HEALTH "):
        parts = line.split()
        if len(parts) >= 3:
            health_results[parts[1]] = parts[2]
    elif line.startswith("ACK_CODE "):
        ack_code = line.split(maxsplit=1)[1].strip()
    elif line.startswith("ACK_BODY_JSON "):
        ack_body = json.loads(line.split(" ", 1)[1])
    elif line.startswith("SUMMARY "):
        p = line.split()
        if len(p) >= 3:
            summary_ok = int(p[1])
            summary_fail = int(p[2])

if len(health_results) != len(backend_private_ips):
    raise RuntimeError(
        f"Health-check result count mismatch. Expected {len(backend_private_ips)}, got {len(health_results)}."
    )

non_200 = {ip: code for ip, code in health_results.items() if code != "200"}
if non_200:
    raise RuntimeError(f"Backend health check failures: {non_200}")

expected_ack_status = str(upload_ack_status if upload_ack_enabled else 204)
if ack_code != expected_ack_status:
    raise RuntimeError(
        f"Upload ack status mismatch. Expected {expected_ack_status}, got {ack_code}. "
        f"Ack body: {ack_body!r}"
    )

if upload_ack_enabled and upload_ack_body_expected:
    if ack_body.strip() != upload_ack_body_expected:
        raise RuntimeError(
            f"Upload ack body mismatch. Expected {upload_ack_body_expected!r}, got {ack_body.strip()!r}"
        )

# -----------------------------------------------------------------------------
# 3) Optional LB edge check from local machine
# -----------------------------------------------------------------------------
lb_health_check = {"url": "", "http_code": "", "warning": ""}

if locust_default_host:
    lb_health_url = f"{locust_default_host}{health_path}"
    print(f"[3/3] Optional LB edge check from local: {lb_health_url}")
    curl_cmd = [
        "curl",
        "-k",
        "-sS",
        "-o",
        "/dev/null",
        "-w",
        "%{http_code}",
        "--max-time",
        "10",
        lb_health_url,
    ]
    curl_proc = _run_local(curl_cmd, check=False)
    lb_health_check["url"] = lb_health_url
    lb_health_check["http_code"] = (curl_proc.stdout or "").strip()
    if lb_health_check["http_code"] != "200":
        lb_health_check["warning"] = (
            "LB edge health check from local machine did not return 200. "
            "This does not block internal generator/backend readiness."
        )
else:
    lb_health_check["warning"] = "LOCUST_DEFAULT_HOST is empty (expected in tcp_ppv2 mode)."

# -----------------------------------------------------------------------------
# Persist helper artifacts
# -----------------------------------------------------------------------------
ssh_helper_path = run_dir / "ssh_helpers.sh"
ssh_helper = f"""#!/usr/bin/env bash
# Generated by Cell 9 preflight
export FLB_GENERATOR_IP="{generator_ip}"
export FLB_SSH_KEY="{ssh_private_key_path}"
export FLB_LB_PRIMARY_IP="{lb_primary_ip}"
export FLB_LOCUST_DEFAULT_HOST="{locust_default_host}"

flb_ssh_gen() {{
  ssh -o StrictHostKeyChecking=no -o BatchMode=yes -o ConnectTimeout=12 -i "$FLB_SSH_KEY" "opc@$FLB_GENERATOR_IP" "$@"
}}

flb_tail_locust() {{
  flb_ssh_gen "tail -n 200 -f /home/opc/locustwork/locust.log"
}}
"""
ssh_helper_path.write_text(ssh_helper, encoding="utf-8")
ssh_helper_path.chmod(0o700)

preflight_summary = {
    "generated_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "run_dir": str(run_dir),
    "generator_ip": generator_ip,
    "backend_count": len(backend_private_ips),
    "lb_primary_ip": lb_primary_ip,
    "locust_default_host": locust_default_host,
    "health_path": health_path,
    "upload_path": upload_path,
    "upload_ack_enabled": upload_ack_enabled,
    "upload_ack_expected_status": expected_ack_status,
    "upload_ack_expected_body": upload_ack_body_expected,
    "generator_readiness_log": str(run_dir / "cell9_generator_readiness.log"),
    "backend_readiness_log": str(run_dir / "cell9_backend_readiness.log"),
    "ssh_helpers": str(ssh_helper_path),
    "health_ok_count": summary_ok if summary_ok is not None else len(backend_private_ips),
    "health_fail_count": summary_fail if summary_fail is not None else 0,
    "upload_ack_status_actual": ack_code,
    "upload_ack_body_actual": ack_body.strip(),
    "lb_health_check": lb_health_check,
}
preflight_summary_path = run_dir / "cell9_preflight_summary.json"
preflight_summary_path.write_text(json.dumps(preflight_summary, indent=2), encoding="utf-8")

# Export globals/env for downstream run cells
globals()["GENERATOR_IP"] = generator_ip
globals()["GENERATOR_PUBLIC_IP"] = generator_ip
globals()["BACKEND_PRIVATE_IPS"] = backend_private_ips
globals()["SSH_HELPERS_PATH"] = str(ssh_helper_path)
globals()["CELL9_PREFLIGHT_SUMMARY_PATH"] = str(preflight_summary_path)

os.environ["GENERATOR_IP"] = generator_ip
os.environ["GENERATOR_PUBLIC_IP"] = generator_ip
os.environ["SSH_HELPERS_PATH"] = str(ssh_helper_path)
os.environ["CELL9_PREFLIGHT_SUMMARY_PATH"] = str(preflight_summary_path)

print("Cell 9 complete: generator/backend preflight passed and SSH helpers were generated.")
print(json.dumps(preflight_summary, indent=2))
print("NEXT: Run Cell 10 to start/monitor Locust orchestration.")


In [ ]:
# Cell 10 — Objective: Protocol-aware sanity checks
# Aligned with PLAN.md + CPS reference approach:
# - Run sanity checks from generator
# - Use curl first (H1/H2), optional venv-httpx fallback for H2 evidence
# - Do NOT launch Locust in this cell

import os
import json
import shlex
import subprocess
from pathlib import Path
from datetime import datetime, UTC


def _run_local(cmd, input_text=None, check=True):
    proc = subprocess.run(cmd, input=input_text, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {' '.join(cmd)}")
    return proc


def _ssh_script(generator_ip, key_path, script, timeout_sec=30, check=True):
    cmd = [
        "ssh",
        "-o", "StrictHostKeyChecking=no",
        "-o", "BatchMode=yes",
        "-o", f"ConnectTimeout={timeout_sec}",
        "-i", key_path,
        f"opc@{generator_ip}",
        "bash -s",
    ]
    return _run_local(cmd, input_text=script, check=check)


# -----------------------------------------------------------------------------
# Inputs / preflight
# -----------------------------------------------------------------------------
selected_outputs_path = Path(
    globals().get("SELECTED_OUTPUTS_PATH", "")
    or (Path(globals().get("RUN_OUTPUT_DIR", Path.cwd() / "results")) / "selected_outputs.json")
).expanduser().resolve()

if not selected_outputs_path.exists():
    raise FileNotFoundError(f"selected_outputs.json not found: {selected_outputs_path}. Run Cell 8 first.")

selected = json.loads(selected_outputs_path.read_text(encoding="utf-8"))
tf = selected.get("terraform", {})
profile = selected.get("profile", {})

generator_ips = [str(x) for x in tf.get("generator_public_ips", [])]
if not generator_ips:
    raise RuntimeError("No generator public IP found in selected outputs.")
generator_ip = generator_ips[0]

run_id = str(selected.get("run_id", globals().get("RUN_ID", f"flb_run_{datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')}")))
run_dir = selected_outputs_path.parent.resolve()
run_dir.mkdir(parents=True, exist_ok=True)

ssh_private_key_path = str(
    Path(globals().get("SSH_PRIVATE_KEY_PATH", os.environ.get("SSH_PRIVATE_KEY_PATH", "~/.ssh/SandboxKey")))
    .expanduser()
    .resolve()
)
if not Path(ssh_private_key_path).exists():
    raise FileNotFoundError(f"SSH private key not found: {ssh_private_key_path}")

protocol_profile = str(globals().get("PROTOCOL_PROFILE", profile.get("protocol_profile", "http2_preferred"))).strip().lower()
health_path = str(globals().get("HEALTH_ENDPOINT_PATH", profile.get("health_endpoint_path", "/healthz"))).strip()

target_url_443 = str(profile.get("target_url_443", "")).strip()
target_url_8443 = str(profile.get("target_url_8443", "")).strip()
target_url_9443 = str(profile.get("target_url_9443", "")).strip()

if not target_url_443 and protocol_profile != "tcp_ppv2":
    raise RuntimeError("target_url_443 missing in selected outputs.")

health_url_443 = f"{target_url_443}{health_path}" if target_url_443 else ""
health_url_8443 = f"{target_url_8443}{health_path}" if target_url_8443 else ""

venv_python = str(globals().get("LOCUST_WORKDIR", "/home/opc/locustwork")).rstrip("/") + "/.venv/bin/python"

# -----------------------------------------------------------------------------
# Remote sanity checks from generator
# -----------------------------------------------------------------------------
check_script = "\n".join(
    [
        "set -euo pipefail",
        f"export URL443={shlex.quote(health_url_443)}",
        f"export URL8443={shlex.quote(health_url_8443)}",
        f"export URL9443={shlex.quote(target_url_9443)}",
        f"export PROFILE={shlex.quote(protocol_profile)}",
        f"export VENV_PY={shlex.quote(venv_python)}",
        "",
        'echo "GEN_HOSTNAME $(hostname)"',
        'if command -v tmux >/dev/null 2>&1; then echo "TMUX_AVAILABLE true"; else echo "TMUX_AVAILABLE false"; fi',
        'echo "CURL_VERSION $(curl --version | head -n1)"',
        'if curl --help all 2>/dev/null | grep -q -- "--http2"; then echo "CURL_HTTP2_FLAG true"; else echo "CURL_HTTP2_FLAG false"; fi',
        "",
        "h1_head() {",
        "  label=\"$1\"; url=\"$2\"",
        "  if [ -z \"$url\" ]; then echo \"H1_HEAD_${label} SKIP_EMPTY_URL\"; return 0; fi",
        "  line=$(curl -skI --http1.1 --connect-timeout 5 --max-time 10 \"$url\" | head -n1 || true)",
        "  if [ -z \"$line\" ]; then line=\"EMPTY\"; fi",
        "  echo \"H1_HEAD_${label} ${line}\"",
        "}",
        "",
        "h2_head() {",
        "  label=\"$1\"; url=\"$2\"",
        "  if [ -z \"$url\" ]; then echo \"H2_HEAD_${label} SKIP_EMPTY_URL\"; return 0; fi",
        "  if ! curl --help all 2>/dev/null | grep -q -- \"--http2\"; then",
        "    echo \"H2_HEAD_${label} SKIP_NO_HTTP2_FLAG\"",
        "    return 0",
        "  fi",
        "  line=$(curl -skI --http2 --connect-timeout 5 --max-time 10 \"$url\" | head -n1 || true)",
        "  if [ -z \"$line\" ]; then line=\"EMPTY\"; fi",
        "  echo \"H2_HEAD_${label} ${line}\"",
        "}",
        "",
        "h1_head 443 \"$URL443\"",
        "h2_head 443 \"$URL443\"",
        "h1_head 8443 \"$URL8443\"",
        "h2_head 8443 \"$URL8443\"",
        "",
        "# Optional fallback proof for H2 via venv httpx (if available)",
        "if [ -x \"$VENV_PY\" ]; then",
        "  \"$VENV_PY\" - <<'PY'",
        "import os",
        "u = os.environ.get('URL443', '').strip()",
        "if not u:",
        "    print('H2_HTTPX_443 SKIP_EMPTY_URL')",
        "    raise SystemExit(0)",
        "try:",
        "    import httpx",
        "except Exception as e:",
        "    print('H2_HTTPX_443 SKIP_NO_HTTPX error={}:{}'.format(type(e).__name__, e))",
        "    raise SystemExit(0)",
        "try:",
        "    t = httpx.Timeout(connect=5.0, read=10.0, write=10.0, pool=10.0)",
        "    with httpx.Client(http2=True, verify=False, timeout=t) as c:",
        "        r = c.get(u)",
        "    print('H2_HTTPX_443 status={} version={}'.format(r.status_code, r.http_version))",
        "except Exception as e:",
        "    print('H2_HTTPX_443 status=ERR version=ERR error={}:{}'.format(type(e).__name__, e))",
        "PY",
        "else",
        "  echo \"H2_HTTPX_443 SKIP_NO_VENV\"",
        "fi",
        "",
        "# tcp_ppv2 profile: L4 TCP connect check to 9443",
        "python3 - <<'PY'",
        "import os, socket",
        "u = os.environ.get('URL9443', '').strip()",
        "profile = os.environ.get('PROFILE', '').strip().lower()",
        "if profile != 'tcp_ppv2':",
        "    print('TCP_9443_CONNECT SKIP_PROFILE')",
        "    raise SystemExit(0)",
        "if not u:",
        "    print('TCP_9443_CONNECT fail error=MissingURL')",
        "    raise SystemExit(0)",
        "try:",
        "    host = u.split('://', 1)[-1].strip('/')",
        "    if host.startswith('['):",
        "        h, p = host.rsplit(']:', 1)",
        "        h = h[1:]",
        "    else:",
        "        h, p = host.rsplit(':', 1)",
        "    p = int(p)",
        "    with socket.create_connection((h, p), timeout=5):",
        "        pass",
        "    print('TCP_9443_CONNECT ok')",
        "except Exception as e:",
        "    print('TCP_9443_CONNECT fail error={}:{}'.format(type(e).__name__, e))",
        "PY",
    ]
)

print("Running protocol sanity checks from generator...")
check_proc = _ssh_script(generator_ip, ssh_private_key_path, check_script, timeout_sec=60, check=True)

log_path = run_dir / "cell10_protocol_sanity.log"
log_path.write_text((check_proc.stdout or "") + "\n" + (check_proc.stderr or ""), encoding="utf-8")

# -----------------------------------------------------------------------------
# Parse + enforce
# -----------------------------------------------------------------------------
lines = [ln.strip() for ln in (check_proc.stdout or "").splitlines() if ln.strip()]

def _get(prefix):
    for ln in lines:
        if ln.startswith(prefix):
            return ln
    return ""

def _is_200_status_line(ln):
    return (" 200 " in ln) or ln.endswith(" 200")

def _is_http2_200_head(ln):
    # curl header line example: HTTP/2 200
    return ("HTTP/2" in ln) and _is_200_status_line(ln)

h1_443 = _get("H1_HEAD_443 ")
h2_443 = _get("H2_HEAD_443 ")
h1_8443 = _get("H1_HEAD_8443 ")
h2_8443 = _get("H2_HEAD_8443 ")
h2_httpx_443 = _get("H2_HTTPX_443 ")
tcp_9443 = _get("TCP_9443_CONNECT ")

tmux_available = _get("TMUX_AVAILABLE ").endswith("true")
curl_http2_supported = _get("CURL_HTTP2_FLAG ").endswith("true")

issues = []

if protocol_profile == "http2_preferred":
    h2_ok_by_curl = _is_http2_200_head(h2_443)
    h2_ok_by_httpx = ("status=200" in h2_httpx_443 and "HTTP/2" in h2_httpx_443)
    if not (h2_ok_by_curl or h2_ok_by_httpx):
        issues.append(
            "HTTP/2 check on 443 failed. "
            f"curl_line={h2_443!r}, httpx_line={h2_httpx_443!r}"
        )
    if not _is_200_status_line(h1_8443):
        issues.append(f"HTTP/1.1 compatibility check on 8443 failed: {h1_8443!r}")
elif protocol_profile == "https_h1_compatible":
    if not _is_200_status_line(h1_8443):
        issues.append(f"HTTP/1.1 check on 8443 failed: {h1_8443!r}")
    # keep 443 path sanity for visibility
    if not _is_200_status_line(h1_443):
        issues.append(f"HTTP/1.1 check on 443 failed: {h1_443!r}")
elif protocol_profile == "tcp_ppv2":
    if " ok" not in tcp_9443:
        issues.append(f"TCP 9443 connectivity check failed: {tcp_9443!r}")
else:
    issues.append(f"Unknown protocol profile: {protocol_profile!r}")

summary = {
    "generated_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "run_id": run_id,
    "generator_ip": generator_ip,
    "protocol_profile": protocol_profile,
    "health_url_443": health_url_443,
    "health_url_8443": health_url_8443,
    "checks": {
        "h1_head_443": h1_443,
        "h2_head_443": h2_443,
        "h1_head_8443": h1_8443,
        "h2_head_8443": h2_8443,
        "h2_httpx_443": h2_httpx_443,
        "tcp_9443_connect": tcp_9443,
    },
    "tmux_available": tmux_available,
    "curl_http2_supported": curl_http2_supported,
    "issues": issues,
    "log_path": str(log_path),
}

summary_path = run_dir / "cell10_protocol_sanity_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

globals()["CELL10_PROTOCOL_SANITY_LOG"] = str(log_path)
globals()["CELL10_PROTOCOL_SANITY_SUMMARY"] = str(summary_path)
globals()["TMUX_AVAILABLE_ON_GENERATOR"] = tmux_available

os.environ["CELL10_PROTOCOL_SANITY_LOG"] = str(log_path)
os.environ["CELL10_PROTOCOL_SANITY_SUMMARY"] = str(summary_path)
os.environ["TMUX_AVAILABLE_ON_GENERATOR"] = "true" if tmux_available else "false"

if issues:
    print(json.dumps(summary, indent=2))
    raise RuntimeError("Cell 10 protocol sanity failed. See issues above and artifact log.")

print("Cell 10 complete: protocol sanity checks passed.")
print(json.dumps(summary, indent=2))
print("NEXT: Run Cell 11 (generator workload prep) and Cell 12 (Locust orchestration/launch).")


In [ ]:
# Cell 11 — Objective: Prepare generators for workload execution (venv locust + locustfile + tmux readiness), in parallel
# - Implements spec-aligned session model:
#   * Session lifecycle close at ~AVG_SESSION_SEC (with optional jitter)
#   * Heartbeat pacing at ~HEARTBEAT_INTERVAL_SEC
#   * HTTP/2 preferred with HTTP/1.1 capability mix via H2_CLIENT_PERCENT

import os
import json
import base64
import shlex
import subprocess
from pathlib import Path
from datetime import datetime, UTC
from concurrent.futures import ThreadPoolExecutor, as_completed


def _run_local(cmd: list[str], input_text: str | None = None, check: bool = True) -> subprocess.CompletedProcess:
    proc = subprocess.run(cmd, input=input_text, text=True, capture_output=True)
    if check and proc.returncode != 0:
        out = (proc.stdout or "").strip()
        err = (proc.stderr or "").strip()
        raise RuntimeError(f"Command failed ({proc.returncode}): {' '.join(cmd)}\nstdout:\n{out}\nstderr:\n{err}")
    return proc


def _ssh_script(generator_ip: str, key_path: str, script: str, timeout_sec: int = 60, check: bool = True) -> subprocess.CompletedProcess:
    cmd = [
        "ssh",
        "-o", "StrictHostKeyChecking=no",
        "-o", "BatchMode=yes",
        "-o", f"ConnectTimeout={timeout_sec}",
        "-i", key_path,
        f"opc@{generator_ip}",
        "bash -s",
    ]
    return _run_local(cmd, input_text=script, check=check)


def _parse_prep_markers(text: str) -> dict:
    d = {}
    for raw in (text or "").splitlines():
        line = raw.strip()
        if line.startswith("__PREP_HOSTNAME__ "):
            d["hostname"] = line.split(" ", 1)[1].strip()
        elif line.startswith("__PREP_TMUX__ "):
            d["tmux_available"] = line.split(" ", 1)[1].strip().lower() == "true"
        elif line.startswith("__PREP_LOCUST_BIN__ "):
            d["locust_bin"] = line.split(" ", 1)[1].strip()
        elif line.startswith("__PREP_LOCUST_VER__ "):
            d["locust_version"] = line.split(" ", 1)[1].strip()
        elif line.startswith("__PREP_PYTHON_VER__ "):
            d["python_version"] = line.split(" ", 1)[1].strip()
        elif line.startswith("__PREP_HTTPX_VER__ "):
            d["httpx_version"] = line.split(" ", 1)[1].strip()
        elif line.startswith("__PREP_NPROC__ "):
            try:
                d["nproc"] = int(line.split(" ", 1)[1].strip())
            except Exception:
                d["nproc"] = 1
        elif line.startswith("__PREP_LB_HEALTH__ "):
            d["lb_health_code"] = line.split(" ", 1)[1].strip()
        elif line.startswith("__PREP_READY__ "):
            d["ready"] = line.split(" ", 1)[1].strip().lower() == "true"
    return d


# -----------------------------------------------------------------------------
# Inputs / preflight
# -----------------------------------------------------------------------------
selected_outputs_path = Path(
    globals().get("SELECTED_OUTPUTS_PATH", "")
    or (Path(globals().get("RUN_OUTPUT_DIR", Path.cwd() / "results")) / "selected_outputs.json")
).expanduser().resolve()

if not selected_outputs_path.exists():
    raise FileNotFoundError(f"selected_outputs.json not found: {selected_outputs_path}. Run Cell 8 first.")

selected = json.loads(selected_outputs_path.read_text(encoding="utf-8"))
tf = selected.get("terraform", {})
profile = selected.get("profile", {})

generator_ips = [str(x) for x in tf.get("generator_public_ips", [])]
if not generator_ips:
    raise RuntimeError("No generator public IPs found in selected outputs.")

run_id = str(selected.get("run_id", globals().get("RUN_ID", f"flb_run_{datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')}")))
run_dir = selected_outputs_path.parent.resolve()
run_dir.mkdir(parents=True, exist_ok=True)

ssh_private_key_path = str(
    Path(globals().get("SSH_PRIVATE_KEY_PATH", os.environ.get("SSH_PRIVATE_KEY_PATH", "~/.ssh/SandboxKey")))
    .expanduser()
    .resolve()
)
if not Path(ssh_private_key_path).exists():
    raise FileNotFoundError(f"SSH private key not found: {ssh_private_key_path}")

locust_workdir = str(globals().get("LOCUST_WORKDIR", "/home/opc/locustwork"))
locust_default_host = str(globals().get("LOCUST_DEFAULT_HOST", profile.get("locust_default_host", ""))).strip()
if not locust_default_host:
    raise ValueError("LOCUST_DEFAULT_HOST is empty. Run Cell 8 and ensure selected outputs are loaded.")

protocol_profile = str(globals().get("PROTOCOL_PROFILE", profile.get("protocol_profile", "http2_preferred"))).strip().lower()
if protocol_profile not in {"http2_preferred", "https_h1_compatible", "tcp_ppv2"}:
    raise ValueError(f"Unsupported PROTOCOL_PROFILE: {protocol_profile!r}")

health_path = str(globals().get("HEALTH_ENDPOINT_PATH", profile.get("health_endpoint_path", "/healthz"))).strip()
upload_path = str(globals().get("UPLOAD_ENDPOINT_PATH", profile.get("upload_endpoint_path", "/upload_5k"))).strip()

locust_verify_tls = bool(globals().get("LOCUST_VERIFY_TLS", False))
locust_connect_timeout_ms = int(globals().get("LOCUST_CONNECT_TIMEOUT_MS", 8000))
locust_read_timeout_ms = int(globals().get("LOCUST_READ_TIMEOUT_MS", 15000))

h2_client_percent_raw = float(globals().get("H2_CLIENT_PERCENT", 100.0))
h2_client_percent_raw = max(0.0, min(100.0, h2_client_percent_raw))

if protocol_profile == "http2_preferred":
    h2_client_percent = h2_client_percent_raw
elif protocol_profile in {"https_h1_compatible", "tcp_ppv2"}:
    h2_client_percent = 0.0
else:
    h2_client_percent = h2_client_percent_raw

heartbeat_interval_sec = int(globals().get("HEARTBEAT_INTERVAL_SEC", 50))
upload_bytes = int(globals().get("UPLOAD_BYTES", 5120))
avg_session_sec = int(globals().get("AVG_SESSION_SEC", 1200))

session_close_mode = str(globals().get("SESSION_CLOSE_MODE", "lifecycle_ttl")).strip().lower()
if session_close_mode != "lifecycle_ttl":
    raise ValueError(f"Unsupported SESSION_CLOSE_MODE: {session_close_mode!r} (expected 'lifecycle_ttl').")

session_ttl_jitter_enabled = bool(globals().get("SESSION_TTL_JITTER_ENABLED", True))
session_ttl_jitter_pct = float(globals().get("SESSION_TTL_JITTER_PCT", 10.0))
if session_ttl_jitter_pct < 0.0 or session_ttl_jitter_pct > 95.0:
    raise ValueError(f"SESSION_TTL_JITTER_PCT must be within [0,95]. Current: {session_ttl_jitter_pct}")

if heartbeat_interval_sec <= 0:
    raise ValueError(f"HEARTBEAT_INTERVAL_SEC must be > 0. Current: {heartbeat_interval_sec}")
if avg_session_sec <= 0:
    raise ValueError(f"AVG_SESSION_SEC must be > 0. Current: {avg_session_sec}")
if upload_bytes <= 0:
    raise ValueError(f"UPLOAD_BYTES must be > 0. Current: {upload_bytes}")

prep_concurrency = int(globals().get("PREP_CONCURRENCY", 5))
prep_concurrency = max(1, min(prep_concurrency, len(generator_ips)))
require_tmux = bool(globals().get("REQUIRE_TMUX", True))

workers_mode = str(globals().get("WORKERS_PER_HOST", "auto")).strip().lower()
cpu_reserve = int(globals().get("CPU_RESERVE", 1))
min_workers_per_host = int(globals().get("MIN_WORKERS_PER_HOST", 1))
max_workers_per_host = int(globals().get("MAX_WORKERS_PER_HOST", 64))
fixed_workers_per_host = int(globals().get("FIXED_WORKERS_PER_HOST", min_workers_per_host))
if max_workers_per_host < min_workers_per_host:
    raise ValueError("MAX_WORKERS_PER_HOST must be >= MIN_WORKERS_PER_HOST")

# -----------------------------------------------------------------------------
# Canonical locustfile for this scenario (H1 + H2 cohorts, lifecycle TTL, exact heartbeat pacing)
# -----------------------------------------------------------------------------
locustfile_code = r'''import os
import time
import random
import httpx
from locust import HttpUser, User, task, constant_pacing, events
from locust.exception import StopUser

PROTOCOL_PROFILE = os.environ.get("LOCUST_PROTOCOL_PROFILE", "http2_preferred").strip().lower()
HEALTH_PATH = os.environ.get("LOCUST_HEALTH_PATH", "/healthz")
UPLOAD_PATH = os.environ.get("LOCUST_UPLOAD_PATH", "/upload_5k")
HEARTBEAT_SEC = max(float(os.environ.get("LOCUST_HEARTBEAT_INTERVAL_SEC", "50")), 0.1)
UPLOAD_BYTES = max(int(os.environ.get("LOCUST_UPLOAD_BYTES", "5120")), 1)

SESSION_SEC = max(float(os.environ.get("LOCUST_SESSION_SEC", "1200")), 1.0)
SESSION_CLOSE_MODE = os.environ.get("LOCUST_SESSION_CLOSE_MODE", "lifecycle_ttl").strip().lower()
TTL_JITTER_ENABLED = os.environ.get("LOCUST_SESSION_TTL_JITTER_ENABLED", "true").strip().lower() in ("1", "true", "yes", "y", "on")
TTL_JITTER_PCT = min(max(float(os.environ.get("LOCUST_SESSION_TTL_JITTER_PCT", "10")), 0.0), 95.0)

VERIFY_TLS = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
CONNECT_TIMEOUT_S = max(float(os.environ.get("LOCUST_CONNECT_TIMEOUT_MS", "8000")) / 1000.0, 0.1)
READ_TIMEOUT_S = max(float(os.environ.get("LOCUST_READ_TIMEOUT_MS", "15000")) / 1000.0, 0.1)

DEFAULT_HOST = (os.environ.get("LOCUST_DEFAULT_HOST") or os.environ.get("TARGET_URL") or "").strip()
if not DEFAULT_HOST:
    DEFAULT_HOST = "https://127.0.0.1"

H2_PCT = min(max(float(os.environ.get("LOCUST_H2_CLIENT_PERCENT", "100")), 0.0), 100.0)

if PROTOCOL_PROFILE == "http2_preferred":
    H2_WEIGHT = int(round(H2_PCT))
    H1_WEIGHT = int(round(100.0 - H2_PCT))
elif PROTOCOL_PROFILE in {"https_h1_compatible", "tcp_ppv2"}:
    H2_WEIGHT = 0
    H1_WEIGHT = 100
else:
    H2_WEIGHT = int(round(H2_PCT))
    H1_WEIGHT = int(round(100.0 - H2_PCT))

if H1_WEIGHT == 0 and H2_WEIGHT == 0:
    H1_WEIGHT = 1

PAYLOAD = b"x" * UPLOAD_BYTES


def _full_url(base, path):
    base = (base or "").rstrip("/")
    return f"{base}{path}"


def _ttl_seconds():
    if not TTL_JITTER_ENABLED or TTL_JITTER_PCT <= 0.0:
        return SESSION_SEC
    delta = random.uniform(-TTL_JITTER_PCT / 100.0, TTL_JITTER_PCT / 100.0)
    return max(1.0, SESSION_SEC * (1.0 + delta))


class _SessionLifecycleMixin:
    def _init_session_lifecycle(self):
        self._session_started_mono = time.monotonic()
        self._session_ttl_sec = _ttl_seconds()
        self._session_deadline_mono = self._session_started_mono + self._session_ttl_sec

    def _maybe_stop_for_ttl(self):
        if SESSION_CLOSE_MODE != "lifecycle_ttl":
            return
        if time.monotonic() >= self._session_deadline_mono:
            raise StopUser()


class H1SessionUser(_SessionLifecycleMixin, HttpUser):
    weight = H1_WEIGHT
    host = DEFAULT_HOST
    # Locust docs: constant_pacing keeps task start cadence near the configured interval.
    wait_time = constant_pacing(HEARTBEAT_SEC)

    def on_start(self):
        self._init_session_lifecycle()

    @task(1)
    def heartbeat_upload(self):
        self._maybe_stop_for_ttl()
        self.client.post(
            UPLOAD_PATH,
            data=PAYLOAD,
            headers={"Content-Type": "application/octet-stream"},
            verify=VERIFY_TLS,
            timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
            name=f"H1 {UPLOAD_PATH}",
        )
        self._maybe_stop_for_ttl()


class H2SessionUser(_SessionLifecycleMixin, User):
    weight = H2_WEIGHT
    wait_time = constant_pacing(HEARTBEAT_SEC)
    abstract = False

    def on_start(self):
        self._init_session_lifecycle()
        self.base_url = DEFAULT_HOST.rstrip("/")
        self._timeout = httpx.Timeout(
            connect=CONNECT_TIMEOUT_S,
            read=READ_TIMEOUT_S,
            write=READ_TIMEOUT_S,
            pool=max(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
        )

        # Keep one long-lived H2 connection per session (one_session_one_connection model).
        keepalive_expiry = max(HEARTBEAT_SEC * 2.0, self._session_ttl_sec + HEARTBEAT_SEC)
        self._limits = httpx.Limits(
            max_connections=1,
            max_keepalive_connections=1,
            keepalive_expiry=keepalive_expiry,
        )

        self._client = httpx.Client(
            http2=True,
            verify=VERIFY_TLS,
            timeout=self._timeout,
            limits=self._limits,
        )

    def on_stop(self):
        try:
            self._client.close()
        except Exception:
            pass

    @task(1)
    def heartbeat_upload_h2(self):
        self._maybe_stop_for_ttl()
        if not self.base_url:
            return

        start = time.perf_counter()
        resp = None
        exc = None
        response_length = 0

        try:
            resp = self._client.post(
                _full_url(self.base_url, UPLOAD_PATH),
                content=PAYLOAD,
                headers={"Content-Type": "application/octet-stream"},
            )
            response_length = len(resp.content or b"")
            resp.raise_for_status()
        except Exception as e:
            exc = e

        elapsed_ms = (time.perf_counter() - start) * 1000.0
        events.request.fire(
            request_type="HTTP",
            name=f"H2 {UPLOAD_PATH}",
            response_time=elapsed_ms,
            response_length=response_length,
            response=resp,
            context={},
            exception=exc,
        )

        self._maybe_stop_for_ttl()
'''
locustfile_b64 = base64.b64encode(locustfile_code.encode("utf-8")).decode("ascii")

session_profile_lines = [
    "export LOCUST_MODE=session_concurrency",
    f"export LOCUST_WORKDIR={shlex.quote(locust_workdir)}",
    f"export LOCUST_VENV_DIR={shlex.quote(locust_workdir + '/.venv')}",
    f"export LOCUST_BIN={shlex.quote(locust_workdir + '/.venv/bin/locust')}",
    f"export LOCUST_PYTHON={shlex.quote(locust_workdir + '/.venv/bin/python')}",
    f"export PATH={shlex.quote(locust_workdir + '/.venv/bin')}:$PATH",
    f"export LOCUST_DEFAULT_HOST={shlex.quote(locust_default_host)}",
    f"export TARGET_URL={shlex.quote(locust_default_host)}",
    f"export LOCUST_PROTOCOL_PROFILE={shlex.quote(protocol_profile)}",
    f"export LOCUST_H2_CLIENT_PERCENT={h2_client_percent}",
    f"export LOCUST_HEALTH_PATH={shlex.quote(health_path)}",
    f"export LOCUST_UPLOAD_PATH={shlex.quote(upload_path)}",
    f"export LOCUST_HEARTBEAT_INTERVAL_SEC={heartbeat_interval_sec}",
    f"export LOCUST_UPLOAD_BYTES={upload_bytes}",
    f"export LOCUST_SESSION_SEC={avg_session_sec}",
    f"export LOCUST_SESSION_CLOSE_MODE={shlex.quote(session_close_mode)}",
    f"export LOCUST_SESSION_TTL_JITTER_ENABLED={'true' if session_ttl_jitter_enabled else 'false'}",
    f"export LOCUST_SESSION_TTL_JITTER_PCT={session_ttl_jitter_pct}",
    f"export LOCUST_VERIFY_TLS={'true' if locust_verify_tls else 'false'}",
    f"export LOCUST_CONNECT_TIMEOUT_MS={locust_connect_timeout_ms}",
    f"export LOCUST_READ_TIMEOUT_MS={locust_read_timeout_ms}",
]
session_profile_content = "\n".join(session_profile_lines) + "\n"


def _prepare_host(host: str) -> dict:
    remote_script = "\n".join([
        "set -euo pipefail",
        f"WORKDIR={shlex.quote(locust_workdir)}",
        "VENV_DIR=\"$WORKDIR/.venv\"",
        "VENV_PY=\"$VENV_DIR/bin/python\"",
        "VENV_PIP=\"$VENV_DIR/bin/pip\"",
        "VENV_LOCUST=\"$VENV_DIR/bin/locust\"",
        "mkdir -p \"$WORKDIR\" \"$WORKDIR/logs\" \"$WORKDIR/results\"",
        "",
        "# Best-effort generator-side socket tuning for high session churn",
        "if command -v sudo >/dev/null 2>&1; then",
        "  sudo -n sysctl -w net.core.somaxconn=262144 || true",
        "  sudo -n sysctl -w net.core.netdev_max_backlog=250000 || true",
        "  sudo -n sysctl -w net.ipv4.tcp_max_syn_backlog=262144 || true",
        "  sudo -n sysctl -w net.ipv4.ip_local_port_range='1024 65535' || true",
        "  sudo -n sysctl -w net.ipv4.tcp_fin_timeout=15 || true",
        "  sudo -n sysctl -w net.ipv4.tcp_tw_reuse=1 || true",
        "  sudo -n sysctl -w fs.file-max=1000000 || true",
        "fi",
        "",
        "# Ensure tmux exists (best effort install)",
        "if ! command -v tmux >/dev/null 2>&1; then",
        "  if command -v dnf >/dev/null 2>&1; then sudo -n dnf -y install tmux || true; fi",
        "  if ! command -v tmux >/dev/null 2>&1 && command -v yum >/dev/null 2>&1; then sudo -n yum -y install tmux || true; fi",
        "  if ! command -v tmux >/dev/null 2>&1 && command -v apt-get >/dev/null 2>&1; then sudo -n apt-get update -y || true; sudo -n apt-get install -y tmux || true; fi",
        "fi",
        "if command -v tmux >/dev/null 2>&1; then TMUX_OK=true; else TMUX_OK=false; fi",
        "",
        "# Ensure venv",
        "if [ ! -x \"$VENV_PY\" ]; then",
        "  python3 -m venv \"$VENV_DIR\" || true",
        "fi",
        "if [ ! -x \"$VENV_PY\" ]; then",
        "  if command -v virtualenv >/dev/null 2>&1; then virtualenv -p python3 \"$VENV_DIR\"; fi",
        "fi",
        "if [ ! -x \"$VENV_PY\" ] || [ ! -x \"$VENV_PIP\" ]; then",
        "  echo 'ERROR: failed to prepare virtualenv' >&2",
        "  exit 41",
        "fi",
        "",
        "# Install/repair locust stack only if missing",
        "NEED_INSTALL=0",
        "if [ ! -x \"$VENV_LOCUST\" ]; then NEED_INSTALL=1; fi",
        "if ! \"$VENV_PY\" - <<'PY'\nimport sys\ntry:\n import locust, httpx, requests\n from locust import constant_pacing\n from locust.exception import StopUser\nexcept Exception:\n raise SystemExit(1)\nraise SystemExit(0)\nPY",
        "then NEED_INSTALL=1; fi",
        "",
        "if [ \"$NEED_INSTALL\" -eq 1 ]; then",
        "  \"$VENV_PY\" -m pip install --upgrade pip setuptools wheel",
        "  if \"$VENV_PY\" - <<'PY'\nimport sys\nraise SystemExit(0 if sys.version_info >= (3,10) else 1)\nPY",
        "  then LOCUST_SPEC='locust>=2.42,<3'; else LOCUST_SPEC='locust==2.33.2'; fi",
        "  \"$VENV_PY\" -m pip install --prefer-binary --no-cache-dir \"$LOCUST_SPEC\" \"httpx[http2]>=0.28,<1\" \"requests>=2.31,<3\"",
        "fi",
        "",
        "# Final module verification",
        "\"$VENV_PY\" - <<'PY'\nimport sys, locust, httpx, requests\nfrom locust import constant_pacing\nfrom locust.exception import StopUser\nprint('VERIFY_LOCUST', locust.__version__)\nprint('VERIFY_HTTPX', httpx.__version__)\nprint('VERIFY_REQUESTS', requests.__version__)\nprint('VERIFY_PYTHON', sys.version.split()[0])\nPY",
        "",
        "# Write canonical locustfile",
        "cat > \"$WORKDIR/locustfile.py.b64\" <<'B64'\n" + locustfile_b64 + "\nB64",
        "\"$VENV_PY\" - <<'PY'\nimport base64, pathlib\nw = pathlib.Path('" + locust_workdir + "')\nb64 = (w / 'locustfile.py.b64').read_text(encoding='utf-8').strip()\n(w / 'locustfile.py').write_text(base64.b64decode(b64).decode('utf-8'), encoding='utf-8')\n(w / 'locustfile.py.b64').unlink(missing_ok=True)\nprint('LOCUSTFILE_WRITTEN', str(w / 'locustfile.py'))\nPY",
        "\"$VENV_PY\" -m py_compile \"$WORKDIR/locustfile.py\"",
        "",
        "# Write session profile env",
        "cat > \"$WORKDIR/session_profile.env\" <<'ENV'\n" + session_profile_content + "ENV",
        "",
        "# Optional LB sanity from generator",
        "LB_CODE=SKIP",
        "if [ -n " + shlex.quote(locust_default_host) + " ]; then",
        "  LB_CODE=$(curl -skS -o /dev/null -w '%{http_code}' --connect-timeout 5 --max-time 10 " + shlex.quote(locust_default_host + health_path) + " || true)",
        "fi",
        "",
        "LOCUST_VER=$($VENV_LOCUST --version | head -n1 || true)",
        "PY_VER=$($VENV_PY --version 2>&1 | head -n1 || true)",
        "HTTPX_VER=$($VENV_PY - <<'PY'\nimport httpx\nprint(httpx.__version__)\nPY\n)",
        "NPROC=$(nproc 2>/dev/null || getconf _NPROCESSORS_ONLN 2>/dev/null || echo 1)",
        "echo \"__PREP_HOSTNAME__ $(hostname)\"",
        "echo \"__PREP_TMUX__ ${TMUX_OK}\"",
        "echo \"__PREP_LOCUST_BIN__ ${VENV_LOCUST}\"",
        "echo \"__PREP_LOCUST_VER__ ${LOCUST_VER}\"",
        "echo \"__PREP_PYTHON_VER__ ${PY_VER}\"",
        "echo \"__PREP_HTTPX_VER__ ${HTTPX_VER}\"",
        "echo \"__PREP_NPROC__ ${NPROC}\"",
        "echo \"__PREP_LB_HEALTH__ ${LB_CODE}\"",
        "echo \"__PREP_READY__ true\"",
    ])

    proc = _ssh_script(host, ssh_private_key_path, remote_script, timeout_sec=240, check=False)

    host_log = run_dir / f"cell11_prepare_{host.replace(':', '_')}.log"
    host_log.write_text((proc.stdout or "") + "\n" + (proc.stderr or ""), encoding="utf-8")

    parsed = _parse_prep_markers(proc.stdout or "")
    result = {
        "host": host,
        "return_code": proc.returncode,
        "log_path": str(host_log),
        "stdout_tail": (proc.stdout or "")[-3000:],
        "stderr_tail": (proc.stderr or "")[-2000:],
        **parsed,
    }
    result["ok"] = bool(result.get("ready", False) and proc.returncode == 0)
    if "nproc" not in result:
        result["nproc"] = 1
    return result


# -----------------------------------------------------------------------------
# Parallel host preparation
# -----------------------------------------------------------------------------
print(f"Preparing {len(generator_ips)} generator(s) in parallel (concurrency={prep_concurrency}) ...")

host_results = []
with ThreadPoolExecutor(max_workers=prep_concurrency) as ex:
    fut_map = {ex.submit(_prepare_host, host): host for host in generator_ips}
    for fut in as_completed(fut_map):
        host = fut_map[fut]
        try:
            res = fut.result()
        except Exception as e:
            res = {
                "host": host,
                "ok": False,
                "return_code": -1,
                "error": repr(e),
                "nproc": 1,
                "tmux_available": False,
            }
        host_results.append(res)
        print(f" - {host}: ok={res.get('ok')} tmux={res.get('tmux_available')} nproc={res.get('nproc')}")

host_results_sorted = sorted(host_results, key=lambda x: x["host"])
failed_hosts = [r for r in host_results_sorted if not r.get("ok", False)]
tmux_missing_hosts = [r["host"] for r in host_results_sorted if not r.get("tmux_available", False)]

if require_tmux and tmux_missing_hosts:
    raise RuntimeError(
        "Cell 11 failed: tmux is required but missing on hosts: "
        + ", ".join(tmux_missing_hosts)
        + ". Check host logs in run artifacts."
    )

if failed_hosts:
    short = [{"host": r["host"], "return_code": r.get("return_code"), "log_path": r.get("log_path")} for r in failed_hosts]
    raise RuntimeError(f"Cell 11 failed on one or more hosts: {json.dumps(short, indent=2)}")

# -----------------------------------------------------------------------------
# Worker plan (for Cell 12 orchestration)
# -----------------------------------------------------------------------------
def _workers_for_host(nproc: int) -> int:
    if workers_mode == "fixed":
        base = fixed_workers_per_host
    else:
        base = max(1, int(nproc) - cpu_reserve)
    return max(min_workers_per_host, min(max_workers_per_host, int(base)))


workers_plan = {}
for r in host_results_sorted:
    workers_plan[r["host"]] = _workers_for_host(int(r.get("nproc", 1)))

expected_workers = int(sum(workers_plan.values()))
master_ip = generator_ips[0]

prep_summary = {
    "generated_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "run_id": run_id,
    "locust_workdir": locust_workdir,
    "master_ip": master_ip,
    "generator_ips": generator_ips,
    "host_results": host_results_sorted,
    "session_model": {
        "session_close_mode": session_close_mode,
        "avg_session_sec": avg_session_sec,
        "heartbeat_interval_sec": heartbeat_interval_sec,
        "session_ttl_jitter_enabled": session_ttl_jitter_enabled,
        "session_ttl_jitter_pct": session_ttl_jitter_pct,
        "upload_bytes": upload_bytes,
        "protocol_profile": protocol_profile,
        "h2_client_percent_effective": h2_client_percent,
    },
    "workers_policy": {
        "mode": workers_mode,
        "cpu_reserve": cpu_reserve,
        "min_workers_per_host": min_workers_per_host,
        "max_workers_per_host": max_workers_per_host,
        "fixed_workers_per_host": fixed_workers_per_host,
    },
    "workers_plan": workers_plan,
    "expected_workers": expected_workers,
    "require_tmux": require_tmux,
    "tmux_missing_hosts": tmux_missing_hosts,
}
prep_summary_path = run_dir / "cell11_prep_summary.json"
prep_summary_path.write_text(json.dumps(prep_summary, indent=2), encoding="utf-8")

# Export globals/env for launch cell
globals()["LOCUST_MASTER_IP"] = master_ip
globals()["LOCUST_GENERATOR_IPS"] = generator_ips
globals()["LOCUST_WORKDIR"] = locust_workdir
globals()["LOCUST_WORKERS_PLAN"] = workers_plan
globals()["EXPECTED_WORKERS"] = expected_workers
globals()["CELL11_PREP_SUMMARY_PATH"] = str(prep_summary_path)
globals()["TMUX_AVAILABLE_ON_GENERATORS"] = all(r.get("tmux_available", False) for r in host_results_sorted)

os.environ["LOCUST_MASTER_IP"] = master_ip
os.environ["LOCUST_GENERATOR_IPS_JSON"] = json.dumps(generator_ips)
os.environ["LOCUST_WORKERS_PLAN_JSON"] = json.dumps(workers_plan)
os.environ["EXPECTED_WORKERS"] = str(expected_workers)
os.environ["CELL11_PREP_SUMMARY_PATH"] = str(prep_summary_path)
os.environ["TMUX_AVAILABLE_ON_GENERATORS"] = "true" if globals()["TMUX_AVAILABLE_ON_GENERATORS"] else "false"

print("Cell 11 complete: generators prepared (venv locust + spec-aligned locustfile + session env) and worker plan computed.")
print(json.dumps({
    "prep_summary": str(prep_summary_path),
    "master_ip": master_ip,
    "generator_count": len(generator_ips),
    "tmux_available_on_all_generators": globals()["TMUX_AVAILABLE_ON_GENERATORS"],
    "expected_workers": expected_workers,
    "workers_plan": workers_plan,
    "session_model": prep_summary["session_model"],
}, indent=2))
print("NEXT: Run Cell 12 to launch Locust orchestration (master + workers) using this prep plan.")


In [ ]:
# Cell 12 — Objective: Start Locust orchestration (master + workers) using Cell 11 prep outputs
# - Supports UI mode and headless mode
# - Uses tmux when available, falls back to nohup
# - Verifies master listener(s) and worker process counts
# - Keeps summary concise while printing exact operator run guidance

import os
import json
import time
import base64
import shlex
import subprocess
from pathlib import Path
from datetime import datetime, UTC
from concurrent.futures import ThreadPoolExecutor, as_completed


def _run_local(cmd: list[str], input_text: str | None = None, check: bool = True) -> subprocess.CompletedProcess:
    proc = subprocess.run(cmd, input=input_text, text=True, capture_output=True)
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed ({proc.returncode}): {' '.join(cmd)}\n"
            f"stdout:\n{(proc.stdout or '').strip()}\n"
            f"stderr:\n{(proc.stderr or '').strip()}"
        )
    return proc


def _ssh_script(host: str, key_path: str, script: str, timeout_sec: int = 60, check: bool = True) -> subprocess.CompletedProcess:
    cmd = [
        "ssh",
        "-o", "StrictHostKeyChecking=no",
        "-o", "BatchMode=yes",
        "-o", f"ConnectTimeout={timeout_sec}",
        "-i", key_path,
        f"opc@{host}",
        "bash -s",
    ]
    return _run_local(cmd, input_text=script, check=check)


def _remote_write_file(host: str, key_path: str, remote_path: str, content: str, mode: str = "0755") -> None:
    b64 = base64.b64encode(content.encode("utf-8")).decode("ascii")
    script = "\n".join(
        [
            "set -euo pipefail",
            f"mkdir -p {shlex.quote(str(Path(remote_path).parent))}",
            f"echo {shlex.quote(b64)} | base64 -d > {shlex.quote(remote_path)}",
            f"chmod {mode} {shlex.quote(remote_path)}",
        ]
    )
    _ssh_script(host, key_path, script, timeout_sec=30, check=True)


def _count_workers(host: str, key_path: str) -> int:
    script = "set -euo pipefail; pgrep -fc 'locust.*--worker' || true"
    proc = _ssh_script(host, key_path, script, timeout_sec=15, check=False)
    try:
        return int((proc.stdout or "0").strip().splitlines()[-1])
    except Exception:
        return 0


def _master_status(host: str, key_path: str, ui_mode: bool, ui_port: int, master_port: int) -> dict:
    script_lines = [
        "set -euo pipefail",
        "MASTER_PROC=$(pgrep -fc 'locust.*--master' || true)",
        f"MASTER_LISTEN=$(ss -lnt 2>/dev/null | awk '{{print $4}}' | grep -c ':{master_port}$' || true)",
    ]
    if ui_mode:
        script_lines.append(f"UI_LISTEN=$(ss -lnt 2>/dev/null | awk '{{print $4}}' | grep -c ':{ui_port}$' || true)")
    else:
        script_lines.append("UI_LISTEN=0")
    script_lines += [
        'echo "__MASTER_PROC__ ${MASTER_PROC}"',
        'echo "__MASTER_LISTEN__ ${MASTER_LISTEN}"',
        'echo "__UI_LISTEN__ ${UI_LISTEN}"',
    ]
    proc = _ssh_script(host, key_path, "\n".join(script_lines), timeout_sec=15, check=True)
    status = {"master_proc": 0, "master_listen": 0, "ui_listen": 0}
    for ln in (proc.stdout or "").splitlines():
        ln = ln.strip()
        if ln.startswith("__MASTER_PROC__ "):
            status["master_proc"] = int(ln.split(" ", 1)[1].strip())
        elif ln.startswith("__MASTER_LISTEN__ "):
            status["master_listen"] = int(ln.split(" ", 1)[1].strip())
        elif ln.startswith("__UI_LISTEN__ "):
            status["ui_listen"] = int(ln.split(" ", 1)[1].strip())
    return status


def _detect_master_private_ip(host: str, key_path: str) -> str:
    script = r"""
set -euo pipefail
ip="$(ip -4 route get 1.1.1.1 2>/dev/null | awk '{for(i=1;i<=NF;i++) if($i=="src") print $(i+1)}' | head -n1)"
if [ -n "$ip" ]; then
  echo "$ip"
  exit 0
fi
hostname -I 2>/dev/null | tr ' ' '\n' | awk '($1 ~ /^10\./)||($1 ~ /^192\.168\./)||($1 ~ /^172\.(1[6-9]|2[0-9]|3[0-1])\./){print; exit}'
"""
    proc = _ssh_script(host, key_path, script, timeout_sec=15, check=False)
    return (proc.stdout or "").strip()


def _calc_row(users: int, avg_session_sec: int, heartbeat_sec: int) -> dict:
    ramp = users / avg_session_sec
    steady_rps = users / heartbeat_sec
    return {
        "users": int(users),
        "ramp_users_per_sec": round(ramp, 3),
        "expected_steady_heartbeat_rps": round(steady_rps, 3),
        "ramp_derivation": f"{users} / {avg_session_sec} = {round(ramp, 3)} users/s",
        "rps_derivation": f"{users} / {heartbeat_sec} = {round(steady_rps, 3)} rps",
    }


# -----------------------------------------------------------------------------
# Inputs / preflight
# -----------------------------------------------------------------------------
prep_summary_path = Path(globals().get("CELL11_PREP_SUMMARY_PATH", "")).expanduser().resolve()
if not prep_summary_path.exists():
    raise FileNotFoundError(f"Cell 11 summary not found: {prep_summary_path}. Run Cell 11 first.")

prep = json.loads(prep_summary_path.read_text(encoding="utf-8"))

master_ip = str(prep.get("master_ip", "")).strip()
generator_ips = [str(x) for x in prep.get("generator_ips", [])]
workers_plan = {str(k): int(v) for k, v in prep.get("workers_plan", {}).items()}
expected_workers = int(prep.get("expected_workers", 0))
session_model = prep.get("session_model", {}) or {}

if not master_ip or not generator_ips or not workers_plan:
    raise RuntimeError("Cell 11 outputs are incomplete (master/generators/workers_plan missing).")

ssh_private_key_path = str(
    Path(globals().get("SSH_PRIVATE_KEY_PATH", os.environ.get("SSH_PRIVATE_KEY_PATH", "~/.ssh/SandboxKey")))
    .expanduser()
    .resolve()
)
if not Path(ssh_private_key_path).exists():
    raise FileNotFoundError(f"SSH private key not found: {ssh_private_key_path}")

run_dir = Path(prep_summary_path).parent.resolve()
run_dir.mkdir(parents=True, exist_ok=True)

run_id = str(globals().get("RUN_ID", prep.get("run_id", run_dir.name))).strip() or run_dir.name
locust_workdir = str(globals().get("LOCUST_WORKDIR", prep.get("locust_workdir", "/home/opc/locustwork")))

ui_enable = bool(globals().get("UI_ENABLE", True))
headless_enable = bool(globals().get("HEADLESS_ENABLE", False))
if (not ui_enable) and (not headless_enable):
    raise ValueError("Both UI_ENABLE and HEADLESS_ENABLE are false. Enable one mode.")
mode = "headless" if headless_enable else "ui"

ui_web_port = int(globals().get("UI_WEB_PORT", 8089))
master_bind_port = int(globals().get("LOCUST_MASTER_PORT", 5557))

target_concurrency = int(globals().get("TARGET_CONCURRENCY", 1000))
avg_session_sec = max(1, int(session_model.get("avg_session_sec", globals().get("AVG_SESSION_SEC", 1200))))
heartbeat_interval_sec = max(1, int(session_model.get("heartbeat_interval_sec", globals().get("HEARTBEAT_INTERVAL_SEC", 50))))

target_opens_per_sec_math = target_concurrency / avg_session_sec
spawn_rate = max(1.0, target_opens_per_sec_math)
expected_steady_rps_math = target_concurrency / heartbeat_interval_sec

ramp_up_sec = int(globals().get("RAMP_UP_SEC", 1200))
run_hold_sec = int(globals().get("RUN_HOLD_SEC", 3600))
expect_workers_strict = bool(globals().get("EXPECT_WORKERS_STRICT", False))
grace_sec = int(globals().get("GRACE_SEC", 60))

locust_default_host = str(globals().get("LOCUST_DEFAULT_HOST", "")).strip()
if not locust_default_host:
    raise ValueError("LOCUST_DEFAULT_HOST is empty. Run Cell 8 first.")

ui_target_users = int(globals().get("RUN_TARGET_USERS", 50000))
example_users = [10000, 25000, 50000, 1000000, 2000000, 3000000, 4000000, 5000000]
if ui_target_users not in example_users:
    example_users = sorted(set(example_users + [ui_target_users]))

ui_row = _calc_row(ui_target_users, avg_session_sec, heartbeat_interval_sec)
example_rows = [_calc_row(u, avg_session_sec, heartbeat_interval_sec) for u in example_users]

configured_target_opens = float(globals().get("TARGET_OPENS_PER_SEC", target_opens_per_sec_math))
math_mismatch = abs(configured_target_opens - target_opens_per_sec_math) > 1e-6

# -----------------------------------------------------------------------------
# 1) Stop stale sessions/processes
# -----------------------------------------------------------------------------
print("[1/5] Stopping stale master/worker sessions...")

stop_script = "\n".join(
    [
        "set +e",
        "tmux has-session -t locust_ui_master 2>/dev/null && tmux kill-session -t locust_ui_master || true",
        "tmux has-session -t locust_headless_master 2>/dev/null && tmux kill-session -t locust_headless_master || true",
        "tmux has-session -t locust_workers 2>/dev/null && tmux kill-session -t locust_workers || true",
        "pkill -f 'locust.*--master' 2>/dev/null || true",
        "pkill -f 'locust.*--worker' 2>/dev/null || true",
        "true",
    ]
)

with ThreadPoolExecutor(max_workers=min(8, len(generator_ips))) as ex:
    futs = [ex.submit(_ssh_script, h, ssh_private_key_path, stop_script, 20, True) for h in generator_ips]
    for f in as_completed(futs):
        _ = f.result()

# -----------------------------------------------------------------------------
# 2) Build remote run scripts (master + per-host workers)
# -----------------------------------------------------------------------------
print("[2/5] Uploading launch scripts to generators...")

master_private_ip = _detect_master_private_ip(master_ip, ssh_private_key_path) or master_ip

if mode == "ui":
    master_session = "locust_ui_master"
    master_run = f"""#!/usr/bin/env bash
set -euo pipefail
WORKDIR={shlex.quote(locust_workdir)}
cd "$WORKDIR"
source "$WORKDIR/session_profile.env" || true
mkdir -p "$WORKDIR/logs" "$WORKDIR/results"
exec "$LOCUST_PYTHON" -m locust -f "$WORKDIR/locustfile.py" \\
  --master --master-bind-host 0.0.0.0 --master-bind-port {master_bind_port} \\
  --web-host 0.0.0.0 --web-port {ui_web_port} \\
  --host "$LOCUST_DEFAULT_HOST" \\
  > "$WORKDIR/logs/ui_master.log" 2>&1
"""
else:
    master_session = "locust_headless_master"
    run_time_sec = max(120, ramp_up_sec + run_hold_sec + grace_sec)
    master_run = f"""#!/usr/bin/env bash
set -euo pipefail
WORKDIR={shlex.quote(locust_workdir)}
cd "$WORKDIR"
source "$WORKDIR/session_profile.env" || true
mkdir -p "$WORKDIR/logs" "$WORKDIR/results"
exec "$LOCUST_PYTHON" -m locust -f "$WORKDIR/locustfile.py" \\
  --master --master-bind-host 0.0.0.0 --master-bind-port {master_bind_port} \\
  --headless --users {target_concurrency} --spawn-rate {spawn_rate:.6f} \\
  --run-time {run_time_sec}s --stop-timeout {max(30, avg_session_sec)} \\
  --host "$LOCUST_DEFAULT_HOST" \\
  > "$WORKDIR/logs/headless_master.log" 2>&1
"""

master_script_path = f"{locust_workdir}/run_master.sh"
_remote_write_file(master_ip, ssh_private_key_path, master_script_path, master_run, mode="0755")


def _upload_worker_script(host: str, count: int) -> str:
    worker_run = f"""#!/usr/bin/env bash
set -euo pipefail
WORKDIR={shlex.quote(locust_workdir)}
cd "$WORKDIR"
source "$WORKDIR/session_profile.env" || true
mkdir -p "$WORKDIR/logs"
COUNT={count}
MASTER_HOST={shlex.quote(master_private_ip)}
MASTER_PORT={master_bind_port}

for i in $(seq 1 "$COUNT"); do
  nohup "$LOCUST_PYTHON" -m locust -f "$WORKDIR/locustfile.py" \\
    --worker --master-host "$MASTER_HOST" --master-port "$MASTER_PORT" \\
    > "$WORKDIR/logs/locust-worker-$i.log" 2>&1 &
  sleep 0.15
done

echo "WORKERS_STARTED $COUNT"
"""
    p = f"{locust_workdir}/start_workers.sh"
    _remote_write_file(host, ssh_private_key_path, p, worker_run, mode="0755")
    return p


worker_script_paths = {}
for h in generator_ips:
    worker_script_paths[h] = _upload_worker_script(h, workers_plan[h])

# -----------------------------------------------------------------------------
# 3) Launch master and workers
# -----------------------------------------------------------------------------
print("[3/5] Launching master and workers...")

launch_master_script = "\n".join(
    [
        "set -euo pipefail",
        f"WORKDIR={shlex.quote(locust_workdir)}",
        f"MASTER_SCRIPT={shlex.quote(master_script_path)}",
        f"SESSION={shlex.quote(master_session)}",
        "if command -v tmux >/dev/null 2>&1; then",
        "  tmux has-session -t \"$SESSION\" 2>/dev/null && tmux kill-session -t \"$SESSION\" || true",
        "  tmux new-session -d -s \"$SESSION\" \"$MASTER_SCRIPT\"",
        '  echo "__MASTER_LAUNCH__ tmux"',
        "else",
        "  nohup \"$MASTER_SCRIPT\" > \"$WORKDIR/logs/master_launcher.out\" 2>&1 &",
        '  echo "__MASTER_LAUNCH__ nohup"',
        "fi",
        "sleep 2",
        "pgrep -fc 'locust.*--master' || true",
    ]
)
master_launch = _ssh_script(master_ip, ssh_private_key_path, launch_master_script, timeout_sec=30, check=True)
master_launch_mode = "unknown"
for ln in (master_launch.stdout or "").splitlines():
    if ln.startswith("__MASTER_LAUNCH__ "):
        master_launch_mode = ln.split(" ", 1)[1].strip()


def _launch_workers(host: str) -> dict:
    script_path = worker_script_paths[host]
    launch_script = "\n".join(
        [
            "set -euo pipefail",
            f"WORKDIR={shlex.quote(locust_workdir)}",
            f"WORKER_SCRIPT={shlex.quote(script_path)}",
            "if command -v tmux >/dev/null 2>&1; then",
            "  tmux has-session -t locust_workers 2>/dev/null && tmux kill-session -t locust_workers || true",
            "  tmux new-session -d -s locust_workers \"$WORKER_SCRIPT\"",
            '  echo "__WORKER_LAUNCH__ tmux"',
            "else",
            "  nohup \"$WORKER_SCRIPT\" > \"$WORKDIR/logs/workers_launcher.out\" 2>&1 &",
            '  echo "__WORKER_LAUNCH__ nohup"',
            "fi",
            "sleep 1",
            "pgrep -fc 'locust.*--worker' || true",
        ]
    )
    proc = _ssh_script(host, ssh_private_key_path, launch_script, timeout_sec=30, check=True)
    mode_line = "unknown"
    for ln in (proc.stdout or "").splitlines():
        if ln.startswith("__WORKER_LAUNCH__ "):
            mode_line = ln.split(" ", 1)[1].strip()
    return {"host": host, "launch_mode": mode_line, "stdout": proc.stdout or "", "stderr": proc.stderr or ""}


worker_launch_results = []
with ThreadPoolExecutor(max_workers=min(8, len(generator_ips))) as ex:
    futs = [ex.submit(_launch_workers, h) for h in generator_ips]
    for f in as_completed(futs):
        worker_launch_results.append(f.result())
worker_launch_results = sorted(worker_launch_results, key=lambda x: x["host"])

# -----------------------------------------------------------------------------
# 4) Readiness checks
# -----------------------------------------------------------------------------
print("[4/5] Waiting for readiness (master listeners + workers)...")

ready_timeout_sec = int(globals().get("READY_TIMEOUT_SEC", 180))
deadline = time.time() + ready_timeout_sec

last_master_status = {}
last_worker_counts = {}

while time.time() < deadline:
    last_master_status = _master_status(master_ip, ssh_private_key_path, ui_mode=(mode == "ui"), ui_port=ui_web_port, master_port=master_bind_port)
    last_worker_counts = {h: _count_workers(h, ssh_private_key_path) for h in generator_ips}
    total_workers = sum(last_worker_counts.values())

    master_ok = (last_master_status.get("master_proc", 0) >= 1 and last_master_status.get("master_listen", 0) >= 1)
    ui_ok = (last_master_status.get("ui_listen", 0) >= 1) if mode == "ui" else True
    workers_ok = total_workers >= expected_workers

    if master_ok and ui_ok and (workers_ok or (not expect_workers_strict)):
        break
    time.sleep(3)

total_workers = sum(last_worker_counts.values())
master_ok = (last_master_status.get("master_proc", 0) >= 1 and last_master_status.get("master_listen", 0) >= 1)
ui_ok = (last_master_status.get("ui_listen", 0) >= 1) if mode == "ui" else True
workers_ok = total_workers >= expected_workers

issues = []
if not master_ok:
    issues.append(f"Master not ready: {last_master_status}")
if mode == "ui" and not ui_ok:
    issues.append(f"UI listener not ready on port {ui_web_port}: {last_master_status}")
if expect_workers_strict and not workers_ok:
    issues.append(f"Workers below expected: observed={total_workers}, expected={expected_workers}")
if math_mismatch:
    issues.append(
        f"TARGET_OPENS_PER_SEC mismatch: configured={configured_target_opens}, "
        f"math={round(target_opens_per_sec_math, 6)} (using math value for consistency)."
    )

# -----------------------------------------------------------------------------
# 5) Persist summary + exports + run guidance
# -----------------------------------------------------------------------------
print("[5/5] Persisting launch summary...")

if mode == "ui":
    tunnel_cmd = (
        f"ssh -o StrictHostKeyChecking=no -i {shlex.quote(ssh_private_key_path)} "
        f"-L {ui_web_port}:127.0.0.1:{ui_web_port} opc@{master_ip}"
    )
else:
    tunnel_cmd = ""

# Keep summary compact
launch_summary = {
    "generated_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "run_id": run_id,
    "mode": mode,
    "master_ip": master_ip,
    "master_private_ip_for_workers": master_private_ip,
    "generator_ips": generator_ips,
    "expected_workers": expected_workers,
    "observed_total_workers": total_workers,
    "master_status": last_master_status,
    "ui_web_port": ui_web_port if mode == "ui" else None,
    "ui_tunnel_command": tunnel_cmd,
    "session_model": {
        "avg_session_sec": avg_session_sec,
        "heartbeat_interval_sec": heartbeat_interval_sec,
    },
    "run_target": {
        "users": ui_target_users,
        "ramp_users_per_sec": ui_row["ramp_users_per_sec"],
        "expected_steady_heartbeat_rps": ui_row["expected_steady_heartbeat_rps"],
    },
    "issues": issues,
    "artifacts": {
        "run_dir": str(run_dir),
        "master_log_ui": f"{locust_workdir}/logs/ui_master.log",
        "master_log_headless": f"{locust_workdir}/logs/headless_master.log",
        "workers_log_dir": f"{locust_workdir}/logs",
    },
}

summary_path = run_dir / "cell12_launch_summary.json"
summary_path.write_text(json.dumps(launch_summary, indent=2), encoding="utf-8")

globals()["CELL12_LAUNCH_SUMMARY_PATH"] = str(summary_path)
globals()["LOCUST_MASTER_IP"] = master_ip
globals()["LOCUST_GENERATOR_IPS"] = generator_ips
globals()["LOCUST_WORKERS_OBSERVED"] = total_workers
globals()["LOCUST_MODE_ACTIVE"] = mode

os.environ["CELL12_LAUNCH_SUMMARY_PATH"] = str(summary_path)
os.environ["LOCUST_MASTER_IP"] = master_ip
os.environ["LOCUST_GENERATOR_IPS_JSON"] = json.dumps(generator_ips)
os.environ["LOCUST_WORKERS_OBSERVED"] = str(total_workers)
os.environ["LOCUST_MODE_ACTIVE"] = mode

if issues and expect_workers_strict:
    print(json.dumps(launch_summary, indent=2))
    raise RuntimeError("Cell 12 launch completed with readiness issues. See summary for details.")

print("Cell 12 complete: Locust master/workers launched and readiness verified.")
print(json.dumps(launch_summary, indent=2))

if mode == "ui":
    print("\nUI access:")
    print(f"1) Run tunnel:\n   {tunnel_cmd}")
    print(f"2) Open UI: http://127.0.0.1:{ui_web_port}")
    print("3) Enter EXACT values:")
    print(f"   Host: {locust_default_host}")
    print(f"   Number of users: {ui_target_users}")
    print(f"   Ramp up (users/s): {round(ui_row['ramp_users_per_sec'], 3)}")

    print("\nDerivation:")
    print(f"   Ramp = {ui_row['ramp_derivation']}")
    print(f"   Expected steady heartbeat RPS = {ui_row['rps_derivation']}")

    print("\nExamples (derived):")
    for row in example_rows:
        print(
            f"   users={row['users']}: "
            f"ramp={row['ramp_derivation']}; "
            f"expected_steady_rps={row['rps_derivation']}"
        )
else:
    print("\nHeadless mode started with exact math-driven spawn-rate.")
    print(f"  Spawn/Ramp used = TARGET_CONCURRENCY / AVG_SESSION_SEC = {target_concurrency} / {avg_session_sec} = {round(spawn_rate, 6)} users/s")
    print(f"  Expected steady heartbeat RPS = TARGET_CONCURRENCY / HEARTBEAT_INTERVAL_SEC = {target_concurrency} / {heartbeat_interval_sec} = {round(expected_steady_rps_math, 6)}")

print("\nNEXT: Run monitoring/results cell during test, then stop/cleanup cell when done.")


In [ ]:
# Cell 13 — Objective: Controlled stop and cleanup (capture final snapshots, stop Locust cleanly, persist artifacts)
# - Multi-generator aware
# - UI/headless aware
# - tmux/nohup compatible
# - Idempotent (safe to re-run)

import os
import json
import shlex
import time
import subprocess
from pathlib import Path
from datetime import datetime, UTC
from concurrent.futures import ThreadPoolExecutor, as_completed


def _run_local(cmd: list[str], input_text: str | None = None, check: bool = True) -> subprocess.CompletedProcess:
    proc = subprocess.run(cmd, input=input_text, text=True, capture_output=True)
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed ({proc.returncode}): {' '.join(cmd)}\n"
            f"stdout:\n{(proc.stdout or '').strip()}\n"
            f"stderr:\n{(proc.stderr or '').strip()}"
        )
    return proc


def _ssh_script(host: str, key_path: str, script: str, timeout_sec: int = 30, check: bool = True) -> subprocess.CompletedProcess:
    cmd = [
        "ssh",
        "-o", "StrictHostKeyChecking=no",
        "-o", "BatchMode=yes",
        "-o", f"ConnectTimeout={timeout_sec}",
        "-i", key_path,
        f"opc@{host}",
        "bash -s",
    ]
    return _run_local(cmd, input_text=script, check=check)


def _safe_json_loads(txt: str):
    try:
        return json.loads(txt)
    except Exception:
        return None


def _remote_http_call(host: str, key_path: str, url: str, method: str = "GET", timeout_sec: int = 10) -> dict:
    script = "\n".join(
        [
            "set +e",
            f"OUT=$(curl -sS --max-time {int(timeout_sec)} -X {shlex.quote(method)} -w '\\n__HTTP_CODE__%{{http_code}}' {shlex.quote(url)} 2>&1)",
            "RC=$?",
            "echo \"$OUT\"",
            "echo \"__CURL_RC__${RC}\"",
            "exit 0",
        ]
    )
    proc = _ssh_script(host, key_path, script, timeout_sec=max(15, timeout_sec + 5), check=False)
    out = proc.stdout or ""
    body_lines = []
    http_code = ""
    curl_rc = ""
    for ln in out.splitlines():
        ln = ln.rstrip("\n")
        if ln.startswith("__HTTP_CODE__"):
            http_code = ln.replace("__HTTP_CODE__", "", 1).strip()
        elif ln.startswith("__CURL_RC__"):
            curl_rc = ln.replace("__CURL_RC__", "", 1).strip()
        else:
            body_lines.append(ln)
    return {
        "http_code": http_code or "",
        "curl_rc": curl_rc or "",
        "body": "\n".join(body_lines).strip(),
        "stderr": (proc.stderr or "").strip(),
    }


def _remote_proc_snapshot(host: str, key_path: str) -> dict:
    script = r"""
set +e
MASTER_PROC=$(pgrep -fc 'locust.*--master' || true)
WORKER_PROC=$(pgrep -fc 'locust.*--worker' || true)

TMUX_UI=0
TMUX_HEADLESS=0
TMUX_WORKERS=0
if command -v tmux >/dev/null 2>&1; then
  tmux has-session -t locust_ui_master 2>/dev/null && TMUX_UI=1 || true
  tmux has-session -t locust_headless_master 2>/dev/null && TMUX_HEADLESS=1 || true
  tmux has-session -t locust_workers 2>/dev/null && TMUX_WORKERS=1 || true
fi

echo "__MASTER_PROC__ ${MASTER_PROC}"
echo "__WORKER_PROC__ ${WORKER_PROC}"
echo "__TMUX_UI__ ${TMUX_UI}"
echo "__TMUX_HEADLESS__ ${TMUX_HEADLESS}"
echo "__TMUX_WORKERS__ ${TMUX_WORKERS}"
exit 0
"""
    proc = _ssh_script(host, key_path, script, timeout_sec=15, check=False)
    snap = {
        "host": host,
        "master_proc": 0,
        "worker_proc": 0,
        "tmux_ui_master": False,
        "tmux_headless_master": False,
        "tmux_workers": False,
    }
    for ln in (proc.stdout or "").splitlines():
        ln = ln.strip()
        if ln.startswith("__MASTER_PROC__ "):
            snap["master_proc"] = int(ln.split(" ", 1)[1].strip())
        elif ln.startswith("__WORKER_PROC__ "):
            snap["worker_proc"] = int(ln.split(" ", 1)[1].strip())
        elif ln.startswith("__TMUX_UI__ "):
            snap["tmux_ui_master"] = ln.split(" ", 1)[1].strip() == "1"
        elif ln.startswith("__TMUX_HEADLESS__ "):
            snap["tmux_headless_master"] = ln.split(" ", 1)[1].strip() == "1"
        elif ln.startswith("__TMUX_WORKERS__ "):
            snap["tmux_workers"] = ln.split(" ", 1)[1].strip() == "1"
    return snap


def _remote_stop_host(host: str, key_path: str, is_master: bool) -> dict:
    lines = [
        "set +e",
        "if command -v tmux >/dev/null 2>&1; then",
        "  tmux has-session -t locust_workers 2>/dev/null && tmux kill-session -t locust_workers || true",
    ]
    if is_master:
        lines.extend(
            [
                "  tmux has-session -t locust_ui_master 2>/dev/null && tmux kill-session -t locust_ui_master || true",
                "  tmux has-session -t locust_headless_master 2>/dev/null && tmux kill-session -t locust_headless_master || true",
            ]
        )
    lines.extend(
        [
            "fi",
            "pkill -f 'python3 -m locust .*--worker' 2>/dev/null || true",
            "pkill -f 'locust.*--worker' 2>/dev/null || true",
        ]
    )
    if is_master:
        lines.extend(
            [
                "pkill -f 'python3 -m locust .*--master' 2>/dev/null || true",
                "pkill -f 'locust.*--master' 2>/dev/null || true",
            ]
        )
    lines.extend(
        [
            "sleep 1",
            "MASTER_PROC=$(pgrep -fc 'locust.*--master' || true)",
            "WORKER_PROC=$(pgrep -fc 'locust.*--worker' || true)",
            'echo "__MASTER_PROC__ ${MASTER_PROC}"',
            'echo "__WORKER_PROC__ ${WORKER_PROC}"',
            "exit 0",
        ]
    )
    proc = _ssh_script(host, key_path, "\n".join(lines), timeout_sec=25, check=False)
    out_master = 0
    out_worker = 0
    for ln in (proc.stdout or "").splitlines():
        ln = ln.strip()
        if ln.startswith("__MASTER_PROC__ "):
            out_master = int(ln.split(" ", 1)[1].strip())
        elif ln.startswith("__WORKER_PROC__ "):
            out_worker = int(ln.split(" ", 1)[1].strip())
    return {
        "host": host,
        "is_master": is_master,
        "master_proc_after_stop_cmd": out_master,
        "worker_proc_after_stop_cmd": out_worker,
        "stderr": (proc.stderr or "").strip(),
    }


def _remote_logs_tail(host: str, key_path: str, workdir: str, tail_lines: int = 80) -> str:
    script = "\n".join(
        [
            "set +e",
            f"WORKDIR={shlex.quote(workdir)}",
            "LOGDIR=\"$WORKDIR/logs\"",
            f"N={int(tail_lines)}",
            'echo "=== HOST ==="',
            "hostname || true",
            'echo "=== LOGDIR ==="',
            'echo "$LOGDIR"',
            'echo "=== MASTER_LOGS ==="',
            'for f in "$LOGDIR/ui_master.log" "$LOGDIR/headless_master.log"; do',
            '  if [ -f "$f" ]; then',
            '    echo "--- $f (tail $N) ---"',
            '    tail -n "$N" "$f" || true',
            "  fi",
            "done",
            'echo "=== WORKER_LOGS ==="',
            "i=0",
            'for f in "$LOGDIR"/locust-worker-*.log; do',
            '  [ -e "$f" ] || break',
            "  i=$((i+1))",
            '  if [ "$i" -le 6 ]; then',
            '    echo "--- $f (tail $N) ---"',
            '    tail -n "$N" "$f" || true',
            "  fi",
            "done",
            'echo "__WORKER_LOG_FILES__ ${i}"',
            "exit 0",
        ]
    )
    proc = _ssh_script(host, key_path, script, timeout_sec=40, check=False)
    text = (proc.stdout or "").strip()
    if proc.stderr:
        text += "\n\n=== STDERR ===\n" + proc.stderr.strip()
    return text


# -----------------------------------------------------------------------------
# Inputs / preflight
# -----------------------------------------------------------------------------
launch_summary_path = Path(
    str(globals().get("CELL12_LAUNCH_SUMMARY_PATH", os.environ.get("CELL12_LAUNCH_SUMMARY_PATH", ""))).strip()
).expanduser()

if not launch_summary_path.exists():
    raise FileNotFoundError(
        f"Cell 12 launch summary not found: {launch_summary_path}. Run Cell 12 first."
    )

launch = json.loads(launch_summary_path.read_text(encoding="utf-8"))

prep_summary_path = Path(
    str(globals().get("CELL11_PREP_SUMMARY_PATH", os.environ.get("CELL11_PREP_SUMMARY_PATH", ""))).strip()
).expanduser()
prep = {}
if prep_summary_path.exists():
    prep = json.loads(prep_summary_path.read_text(encoding="utf-8"))

run_dir = launch_summary_path.parent.resolve()
run_dir.mkdir(parents=True, exist_ok=True)

ssh_private_key_path = str(
    Path(globals().get("SSH_PRIVATE_KEY_PATH", os.environ.get("SSH_PRIVATE_KEY_PATH", "~/.ssh/SandboxKey")))
    .expanduser()
    .resolve()
)
if not Path(ssh_private_key_path).exists():
    raise FileNotFoundError(f"SSH private key not found: {ssh_private_key_path}")

run_id = str(launch.get("run_id", globals().get("RUN_ID", run_dir.name))).strip() or run_dir.name
mode = str(launch.get("mode", globals().get("LOCUST_MODE_ACTIVE", "ui"))).strip().lower()
master_ip = str(launch.get("master_ip", globals().get("LOCUST_MASTER_IP", ""))).strip()
generator_ips = [str(x).strip() for x in launch.get("generator_ips", globals().get("LOCUST_GENERATOR_IPS", [])) if str(x).strip()]

if not master_ip:
    raise RuntimeError("master_ip missing from Cell 12 summary.")

hosts = []
for h in [master_ip] + generator_ips:
    if h and h not in hosts:
        hosts.append(h)

if not hosts:
    raise RuntimeError("No generator hosts found in launch summary.")

locust_workdir = str(
    globals().get("LOCUST_WORKDIR", launch.get("artifacts", {}).get("locust_workdir", prep.get("locust_workdir", "/home/opc/locustwork")))
).strip() or "/home/opc/locustwork"

ui_web_port = int(launch.get("ui_web_port") or globals().get("UI_WEB_PORT", 8089))
drain_sec = int(globals().get("CELL13_DRAIN_SEC", 5))
tail_lines = int(globals().get("CELL13_LOG_TAIL_LINES", 80))
parallelism = min(8, max(1, len(hosts)))

# -----------------------------------------------------------------------------
# 1) Capture pre-stop snapshots
# -----------------------------------------------------------------------------
print("[1/5] Capturing pre-stop snapshots (processes + UI stats)...")

before_snapshots = {}
with ThreadPoolExecutor(max_workers=parallelism) as ex:
    futs = {ex.submit(_remote_proc_snapshot, h, ssh_private_key_path): h for h in hosts}
    for f in as_completed(futs):
        h = futs[f]
        before_snapshots[h] = f.result()
before_snapshots = {h: before_snapshots[h] for h in hosts}

ui_requests_obj = None
ui_exceptions_obj = None
ui_stop_probe = {"attempted": mode == "ui", "http_code": "", "curl_rc": "", "note": ""}

if mode == "ui":
    r1 = _remote_http_call(master_ip, ssh_private_key_path, f"http://127.0.0.1:{ui_web_port}/stats/requests", method="GET", timeout_sec=10)
    r2 = _remote_http_call(master_ip, ssh_private_key_path, f"http://127.0.0.1:{ui_web_port}/exceptions", method="GET", timeout_sec=10)
    ui_requests_obj = _safe_json_loads(r1["body"])
    ui_exceptions_obj = _safe_json_loads(r2["body"])
    ui_stop_probe["note"] = "UI stats fetched before stop (best-effort)."

ui_snapshot_before = {
    "generated_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "mode": mode,
    "ui_web_port": ui_web_port if mode == "ui" else None,
    "stats_requests": ui_requests_obj,
    "exceptions": ui_exceptions_obj,
}
ui_snapshot_before_path = run_dir / "cell13_ui_snapshot_before_stop.json"
ui_snapshot_before_path.write_text(json.dumps(ui_snapshot_before, indent=2), encoding="utf-8")

# -----------------------------------------------------------------------------
# 2) Graceful UI stop (best effort) + drain
# -----------------------------------------------------------------------------
print("[2/5] Issuing graceful stop (UI API when applicable)...")

ui_stop_result = {"attempted": mode == "ui", "method": "", "http_code": "", "curl_rc": "", "body": "", "stderr": ""}
if mode == "ui":
    # Try GET /stop first (common Locust UI API pattern), then POST fallback.
    get_stop = _remote_http_call(master_ip, ssh_private_key_path, f"http://127.0.0.1:{ui_web_port}/stop", method="GET", timeout_sec=10)
    ui_stop_result = {"attempted": True, "method": "GET", **get_stop}
    if (get_stop.get("http_code") not in ("200", "204")) and (get_stop.get("curl_rc") not in ("0", 0)):
        post_stop = _remote_http_call(master_ip, ssh_private_key_path, f"http://127.0.0.1:{ui_web_port}/stop", method="POST", timeout_sec=10)
        ui_stop_result = {"attempted": True, "method": "POST", **post_stop}

if drain_sec > 0:
    time.sleep(drain_sec)

# -----------------------------------------------------------------------------
# 3) Stop tmux sessions and remaining locust processes
# -----------------------------------------------------------------------------
print("[3/5] Stopping master/worker sessions and processes on all generators...")

stop_results = []
with ThreadPoolExecutor(max_workers=parallelism) as ex:
    futs = {
        ex.submit(_remote_stop_host, h, ssh_private_key_path, h == master_ip): h
        for h in hosts
    }
    for f in as_completed(futs):
        stop_results.append(f.result())
stop_results = sorted(stop_results, key=lambda x: x["host"])

# One quick second sweep only if anything remains
need_second_sweep = any((r["master_proc_after_stop_cmd"] > 0 or r["worker_proc_after_stop_cmd"] > 0) for r in stop_results)
second_sweep_results = []
if need_second_sweep:
    time.sleep(2)
    with ThreadPoolExecutor(max_workers=parallelism) as ex:
        futs = {
            ex.submit(_remote_stop_host, h, ssh_private_key_path, h == master_ip): h
            for h in hosts
        }
        for f in as_completed(futs):
            second_sweep_results.append(f.result())
    second_sweep_results = sorted(second_sweep_results, key=lambda x: x["host"])

# -----------------------------------------------------------------------------
# 4) Post-stop verification + log tail capture
# -----------------------------------------------------------------------------
print("[4/5] Verifying stop state and capturing log tails...")

after_snapshots = {}
with ThreadPoolExecutor(max_workers=parallelism) as ex:
    futs = {ex.submit(_remote_proc_snapshot, h, ssh_private_key_path): h for h in hosts}
    for f in as_completed(futs):
        h = futs[f]
        after_snapshots[h] = f.result()
after_snapshots = {h: after_snapshots[h] for h in hosts}

logs_manifest = {}
for h in hosts:
    tail_text = _remote_logs_tail(h, ssh_private_key_path, locust_workdir, tail_lines=tail_lines)
    local_tail_path = run_dir / f"cell13_logs_tail_{h.replace('.', '_').replace(':', '_')}.log"
    local_tail_path.write_text(tail_text + "\n", encoding="utf-8")
    logs_manifest[h] = {
        "local_tail_path": str(local_tail_path),
        "remote_logs_dir": f"{locust_workdir}/logs",
    }

# -----------------------------------------------------------------------------
# 5) Persist summaries
# -----------------------------------------------------------------------------
print("[5/5] Persisting stop/cleanup artifacts...")

master_status_before_after = {
    "master_ip": master_ip,
    "before": before_snapshots.get(master_ip, {}),
    "after": after_snapshots.get(master_ip, {}),
    "ui_stop_result": ui_stop_result,
}
workers_status_before_after = {
    "hosts": [h for h in hosts if h != master_ip],
    "before": {h: before_snapshots[h] for h in hosts if h != master_ip},
    "after": {h: after_snapshots[h] for h in hosts if h != master_ip},
}

master_status_path = run_dir / "cell13_master_status_before_after.json"
workers_status_path = run_dir / "cell13_workers_status_before_after.json"
logs_manifest_path = run_dir / "cell13_logs_manifest.json"

master_status_path.write_text(json.dumps(master_status_before_after, indent=2), encoding="utf-8")
workers_status_path.write_text(json.dumps(workers_status_before_after, indent=2), encoding="utf-8")
logs_manifest_path.write_text(json.dumps(logs_manifest, indent=2), encoding="utf-8")

issues = []
for h in hosts:
    snap = after_snapshots.get(h, {})
    if snap.get("master_proc", 0) > 0 and h == master_ip:
        issues.append(f"Master process still running on {h}: {snap.get('master_proc')}")
    if snap.get("worker_proc", 0) > 0:
        issues.append(f"Worker processes still running on {h}: {snap.get('worker_proc')}")
    if snap.get("tmux_ui_master", False) and h == master_ip:
        issues.append(f"tmux session locust_ui_master still present on {h}")
    if snap.get("tmux_headless_master", False) and h == master_ip:
        issues.append(f"tmux session locust_headless_master still present on {h}")
    if snap.get("tmux_workers", False):
        issues.append(f"tmux session locust_workers still present on {h}")

stop_summary = {
    "generated_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "run_id": run_id,
    "mode": mode,
    "master_ip": master_ip,
    "generator_ips": hosts,
    "ui_web_port": ui_web_port if mode == "ui" else None,
    "drain_sec": drain_sec,
    "before_counts": {
        h: {
            "master_proc": before_snapshots[h]["master_proc"],
            "worker_proc": before_snapshots[h]["worker_proc"],
        }
        for h in hosts
    },
    "after_counts": {
        h: {
            "master_proc": after_snapshots[h]["master_proc"],
            "worker_proc": after_snapshots[h]["worker_proc"],
        }
        for h in hosts
    },
    "stop_results_first_sweep": stop_results,
    "stop_results_second_sweep": second_sweep_results,
    "ui_snapshot_before_stop": str(ui_snapshot_before_path),
    "issues": issues,
    "artifacts": {
        "run_dir": str(run_dir),
        "cell13_stop_summary": str(run_dir / "cell13_stop_summary.json"),
        "cell13_master_status_before_after": str(master_status_path),
        "cell13_workers_status_before_after": str(workers_status_path),
        "cell13_logs_manifest": str(logs_manifest_path),
    },
}

stop_summary_path = run_dir / "cell13_stop_summary.json"
stop_summary_path.write_text(json.dumps(stop_summary, indent=2), encoding="utf-8")

globals()["CELL13_STOP_SUMMARY_PATH"] = str(stop_summary_path)
os.environ["CELL13_STOP_SUMMARY_PATH"] = str(stop_summary_path)

print("Cell 13 complete: Locust stop/cleanup executed and artifacts persisted.")
print(json.dumps({
    "cell13_stop_summary": str(stop_summary_path),
    "issues": issues,
    "master_ip": master_ip,
    "generator_count": len(hosts),
    "after_total_master_proc": sum(after_snapshots[h]["master_proc"] for h in hosts),
    "after_total_worker_proc": sum(after_snapshots[h]["worker_proc"] for h in hosts),
}, indent=2))
print("NEXT: Run your guarded teardown cell when ready to destroy infrastructure.")


In [ ]:
# Cell X — Objective: Teardown (guarded)

import os
import json
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, UTC

TEARDOWN_CONFIRM = True  # Set False to block destroy


def _run(cmd: list[str], cwd: Path, env: dict, check: bool = True) -> subprocess.CompletedProcess:
    print("$ " + " ".join(cmd))
    proc = subprocess.run(cmd, cwd=str(cwd), env=env, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {' '.join(cmd)}")
    return proc


def _resolve_workdir() -> Path:
    # Prefer paths set by earlier cells when available.
    main_from_globals = str(globals().get("TERRAFORM_MAIN_PATH", "")).strip()
    if main_from_globals:
        p = Path(main_from_globals).expanduser().resolve()
        if p.exists():
            return p.parent.resolve()

    # Fallbacks.
    candidates = [Path.cwd(), Path.cwd() / "FLBConcurrencyTest"]
    for d in candidates:
        if (d / "main.tf").exists() and (d / "terraform.tfvars").exists():
            return d.resolve()

    raise FileNotFoundError("Could not find Terraform workdir with main.tf and terraform.tfvars.")


if not TEARDOWN_CONFIRM:
    print("Teardown guard is False. Set TEARDOWN_CONFIRM=True to destroy resources.")
    print("Cell X complete: No teardown executed.")
else:
    terraform_bin = shutil.which("terraform")
    if not terraform_bin:
        raise EnvironmentError("Terraform CLI not found in PATH.")

    workdir = _resolve_workdir()
    tfvars_path = workdir / "terraform.tfvars"
    if not tfvars_path.exists():
        raise FileNotFoundError(f"terraform.tfvars not found: {tfvars_path}")

    env = os.environ.copy()
    env["TF_IN_AUTOMATION"] = "1"
    env["TF_DATA_DIR"] = "/private/tmp/flb_tf_data_destroy"

    print(f"Terraform binary: {terraform_bin}")
    print(f"Terraform working dir: {workdir}")
    print(f"Using tfvars: {tfvars_path}")
    print(f"Using TF_DATA_DIR: {env['TF_DATA_DIR']}")

    # Re-init with isolated TF_DATA_DIR for reliable provider/plugin state.
    _run([terraform_bin, "init", "-input=false", "-reconfigure"], cwd=workdir, env=env)

    # Destroy all resources from this stack.
    _run(
        [terraform_bin, "destroy", "-input=false", "-auto-approve", f"-var-file={tfvars_path}"],
        cwd=workdir,
        env=env,
    )

    print("Cell X complete: All Terraform resources destroyed.")
    print(json.dumps({
        "destroyed": True,
        "workdir": str(workdir),
        "tfvars": str(tfvars_path),
        "completed_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }, indent=2))
